# MARST — Streaming Spatiotemporal Imputation Benchmark (multi-GPU variant)

> **Multi-GPU note.** This copy auto-detects `torch.cuda.device_count()`. On Kaggle's 2× T4 setup it dispatches independent MARST training jobs concurrently across GPUs via a `ThreadPoolExecutor(max_workers=N_GPUS)`:
> - **MARST main** (3 train seeds), **anchor ablation** (7 variants), and **soft-LOCF decay sweep** (3 variants) now run two-at-a-time across the available GPUs instead of sequentially. `_train_st` takes a `gpu_id`, mirrors the data tensors to that device, and returns the trained net on the default device so every downstream eval / leak / sparsity-sensitivity cell is unchanged.
> - **Ablation epoch budget reduced.** A new `ABLATION_EPOCHS = 400` global controls the anchor ablation, decay sweep, and curriculum-fixed runs (down from 800). Main MARST and the baselines still use `TRAIN_EPOCHS = 800`. The sparsity-sensitivity sweep is evaluation-only on the already-trained MARST, so it has no training epochs to reduce.
> - **Curriculum endpoint scales with `epochs`.** The 60%→80% sparsity ramp was hard-coded to 600 of 800 epochs (75%); it now reaches its peak at `0.75 * epochs` so shorter runs still finish the ramp.
> - **Combined effect.** On 2× T4 the ablation/decay phases roughly halve in wall time from concurrency, and the per-variant epoch cut halves them again — overall ~3–4× speedup on the ablation-heavy portion vs the single-GPU 800-epoch baseline.

### Datasets: PEMS-BAY, METR-LA (speed) + PEMS04, PEMS08 (flow). 80% sparsity, 5 eval-mask seeds × 3 train seeds.
Speed datasets use the zero-as-missing convention; flow datasets are fully observed with synthetic random masking.

**MARST** (Multi-Anchor Residual Spatiotemporal Transformer) is our model: a spatiotemporal
transformer over a causal-temporal / mask-aware-spatial trunk whose residual corrects a
**learned softmax mixture** of three causal anchors (LOCF / HA / KNN).

### Protocol: streaming (causal) imputation

Unlike standard offline imputation methods that exploit bidirectional context
(CSDI, GRIN, PriSTI, ImputeFormer, …), MARST follows the **streaming protocol**: a
prediction at time *t* uses only observations at times ≤ *t*. This matches the
deployment setting where future observations are not yet available and is strictly
harder than the offline protocol used by most published imputation methods.

The streaming framing follows **BayOTIDE (Fang et al., ICML 2024)**, which established
online imputation as a distinct task. To our knowledge MARST is the first **neural,
spatiotemporal-aware, multi-anchor** streaming imputation method — extending BayOTIDE's
Bayesian-GP precedent to deep models that exploit sensor-graph structure.

### Benchmark scope

Compared against 18 causal baselines spanning classical (HA, LOCF, Global Mean, Ridge,
KNN), per-node neural (MLP, LSTM, 2L-LSTM, GRU, TCN), imputation-native
(SAITS, BRITS-forward), and recent causal spatiotemporal architectures
(DCRNN, GWN, STID, DLinear, PatchTST, iTransformer). The notebook runs a MARST
**anchor-mixture ablation**, a **sparsity-sensitivity** sweep, **missing-pattern
robustness** (point / block / sensor), an extended **evaluation-metrics** table
(incl. **JamMAE**, our congestion-regime error metric), a **multi-rate main table**,
**cross-dataset** aggregation, **paired Wilcoxon significance tests with Holm-Bonferroni**,
Q2/Q6/Q12 hyperparameter sensitivity, and a full **publication figure set**.

### Anti-leak invariants (verified per dataset)

Every model receives only the same 20% observed sensor readings as input; no model sees the
held-out 80% or any future time step. The driver runs an empirical leak audit at the end of
each per-dataset block: it perturbs the ground truth at every held-out position and asserts
`max|p_clean − p_corrupted| < 1e-4` on the held-out predictions for MARST and every baseline
(graph-API + non-graph). A future-perturbation test additionally verifies MARST's past
predictions don't move when later timesteps are corrupted. The hard `assert _ok` stops the run on any leak.

### Design notes

- **Node embeddings on MLP and SAITS.** At 80% sparsity, unobserved nodes all receive
  input `[0, 0, sin, cos]`. Without a per-node identity, the model cannot distinguish sensors
  and collapses to a global prediction. A learned node embedding gives each sensor a unique
  fingerprint even when its value is masked out.
- **HA-fill for graph models at eval.** Graph diffusion spreads masked zeros into neighbouring
  nodes, corrupting the spatial signal. Filling unobserved positions with the HA prior (from
  training-only stats) gives a neutral, informative starting point; the mask feature still
  tells the model which nodes are real. The same convention is applied uniformly to every
  graph baseline and matches MARST's HA anchor.
- **KNN imputer uses masked-distance neighbours** (only the observed coordinates contribute
  to the Euclidean distance, then a column-renormalisation) so a held-out value never
  participates in its own neighbour search.
- **HA-initialized internal LOCF** (Fix A). MARST's internal `a_LOCF` anchor starts at the
  HA prior at t=0 (matching standalone LOCF's startup behaviour) rather than zero in z-score.
- **Jam-weighted training loss** (Fix C). The training loss multiplies congested-regime
  positions (`true < p20` for speed, `true > p67` for flow) by `1 + JAM_WEIGHT`, optimizing
  MARST for the JamMAE slice that LOCF historically excels at.


## Related Work — Streaming vs Offline Imputation

Imputation methods in the time-series literature divide into two protocols.
**Offline imputation** (the majority) allows bidirectional context: to fill the missing
value at time *t*, the model freely uses observations at *both* earlier and later
timesteps. This matches the post-hoc data-cleaning use case where the full series
is already collected. **Streaming imputation** restricts each prediction to observations
at times ≤ *t* only, matching the live-deployment use case.

**Streaming-protocol precedents:**
- **BayOTIDE** (Fang et al., ICML 2024) — Bayesian online multivariate imputation with
  Gaussian-process functional decomposition. Their Table 1 explicitly classifies every
  other imputation method (BRITS, NAOMI, SAITS, TIDER, GP-VAE, CSDI, CSBI, …) as
  *offline*. To our knowledge, BayOTIDE is the only prior published streaming-imputation
  method. It treats channels independently and uses no graph structure. MARST extends
  the streaming protocol to deep, graph-aware, multi-anchor neural models.
- **ON-Traffic** (Rap & Das, 2025) — online operator learning from Lagrangian probe
  vehicles. Different sensor model (moving vehicles, not loop detectors) and different
  architecture family (DeepONet). Adjacent, not directly comparable.

**Offline imputation methods, excluded from direct comparison by protocol:**
BRITS (NeurIPS 2018), NAOMI (NeurIPS 2019), GP-VAE (AISTATS 2020), mTAN (ICLR 2021),
CSDI (NeurIPS 2021), GRIN (ICLR 2022), SAITS-full (ESWA 2023), TimesNet (ICLR 2023),
TIDER (ICLR 2023), PriSTI (ICDE 2023), CSBI (NeurIPS 2023), ImputeFormer (KDD 2024),
Casper (CIKM 2024), GSLI (AAAI 2025), PSW-I (ICLR 2025), KAI (2026). These methods
are well-known SOTA but use bidirectional or batch-iterative context that violates
the streaming protocol. We acknowledge their strong reported performance in the
offline setting but do not compare against them under the streaming protocol — doing
so would compare methods that see the future against MARST, which by design does not.

**Note on "causal" terminology.** The word *causal* appears in three distinct senses
across this literature: (1) temporal causality (predictions at time *t* use only
observations at times ≤ *t* — MARST's sense); (2) causal-inference SCM/backdoor-path
reasoning (Casper, MIRACLE); (3) missing-mechanism causality (MCAR/MAR/MNAR — Causal
View of TSI, MIRACLE). Senses (2) and (3) are about *what causes data and missingness*;
sense (1) is about *what timesteps are available at inference*. We use sense (1).


In [ ]:
import torch
import gc
gc.collect()
torch.cuda.empty_cache()
print("GPU memory ready.")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import os
import glob
import pickle
import urllib.request
import warnings
import json
from concurrent.futures import ThreadPoolExecutor

warnings.filterwarnings("ignore")

GLOBAL_SEED = 42
torch.manual_seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_GPUS  = torch.cuda.device_count() if torch.cuda.is_available() else 1
GPU_IDS = list(range(N_GPUS))
print(f'Multi-GPU: detected {N_GPUS} CUDA device(s)' + (f' -> concurrent per-GPU jobs across {GPU_IDS}' if N_GPUS > 1 else ' (single-device run)'))

# ============================================================================
#  Datasets: 'PEMS-BAY', 'METR-LA', 'PEMS04', 'PEMS08'. The driver loop near the
#  end calls run_dataset(name) for each; every run writes results_<DATASET>.json.
#  No manual DATASET switch — run top-to-bottom and all four are processed.
# ============================================================================

# --- Shared experiment config (identical across datasets for comparability) ---
SPARSITY      = 0.80      # fraction of sensor readings hidden (blind) at train/eval
BATCH_TIME    = 48
HIDDEN_DIM    = 96
N_LAYERS      = 5
N_HEADS       = 4
DROPOUT       = 0.1
TRAIN_EPOCHS    = 800
ABLATION_EPOCHS = 400   # epoch budget for anchor / decay / curriculum ablations (vs TRAIN_EPOCHS for main runs)
N_BASELINE_SEEDS = 3       # multi-seed training for every nn baseline (statistical-power parity with MARST)
JAM_WEIGHT       = 1.0     # jam-regime loss weight: positions where true < p20 get (1 + JAM_WEIGHT)x loss
STEPS_PER_DAY = 288       # all four datasets are 5-min sampled -> 288 steps/day
EVAL_SEEDS    = [42, 43, 44, 45, 46]
HUBER_BETA    = 1.0

WINDOW     = 5000         # use first WINDOW timesteps (keeps runs comparable & fast)
TRAIN_END  = 4000
EVAL_START = 4500
EVAL_LEN   = 450
# gap [TRAIN_END:EVAL_START] separates train and eval -> no temporal overlap.

# --- Per-dataset metadata -----------------------------------------------------
#  kind 'speed' : raw value 0 == missing reading (PEMS-BAY / METR-LA convention)
#  kind 'flow'  : .npz [T, N, C]; every entry is real (valid all-ones). Missingness
#                 is purely synthetic via the random mask. Impute channel 0 (flow).
DATASETS = {
    'PEMS-BAY': dict(kind='speed', steps_per_day=288, unit='mph',
                     data_files=['pems-bay.h5', 'PEMS-BAY.h5', 'pems-bay.csv', 'PEMS-BAY.csv'],
                     adj_files=['adj_mx_bay.pkl', 'adj_mx_pems_bay.pkl'],
                     data_url='https://zenodo.org/records/5146275/files/PEMS-BAY.csv?download=1',
                     adj_url='https://zenodo.org/records/5146275/files/adj_mx_bay.pkl?download=1'),
    'METR-LA':  dict(kind='speed', steps_per_day=288, unit='mph',
                     data_files=['metr-la.h5', 'METR-LA.h5', 'metr-la.csv', 'METR-LA.csv'],
                     adj_files=['adj_mx.pkl', 'adj_mx_la.pkl', 'adj_mx_metr_la.pkl'],
                     data_url='https://zenodo.org/records/5146275/files/METR-LA.csv?download=1',
                     adj_url='https://raw.githubusercontent.com/liyaguang/DCRNN/master/data/sensor_graph/adj_mx.pkl'),
    'PEMS04':   dict(kind='flow', steps_per_day=288, channel=0, unit='veh/5min',
                     data_files=['pems04.npz', 'PEMS04.npz', 'pems04_data.npz'],
                     adj_files=['distance_04.csv', 'PEMS04_distance.csv', 'PEMS04.csv', 'pems04.csv', 'distance.csv'],
                     data_url='https://raw.githubusercontent.com/Davidham3/ASTGCN-2019-mxnet/master/data/PEMS04/pems04.npz',
                     adj_url='https://raw.githubusercontent.com/Davidham3/ASTGCN-2019-mxnet/master/data/PEMS04/distance.csv'),
    'PEMS08':   dict(kind='flow', steps_per_day=288, channel=0, unit='veh/5min',
                     data_files=['pems08.npz', 'PEMS08.npz', 'pems08_data.npz'],
                     adj_files=['distance_08.csv', 'PEMS08_distance.csv', 'PEMS08.csv', 'pems08.csv', 'distance.csv'],
                     data_url='https://raw.githubusercontent.com/Davidham3/ASTGCN-2019-mxnet/master/data/PEMS08/pems08.npz',
                     adj_url='https://raw.githubusercontent.com/Davidham3/ASTGCN-2019-mxnet/master/data/PEMS08/distance.csv'),
}


# Data search roots. On Colab/local, set DATA_DIR to the folder holding your
# data files (e.g. '/content' or a mounted-Drive path). Colab's /content and
# Kaggle inputs are searched automatically; sub-folders are searched recursively.
DATA_DIR = None

def _roots():
    cands = [DATA_DIR, '.', os.getcwd(), '/content', '/content/drive/MyDrive',
             '/kaggle/input', '/kaggle/working']
    out = []
    for r in cands:
        if r and os.path.isdir(r) and r not in out:
            out.append(r)
    return out

def find_file(candidates, search_roots=None):
    if search_roots is None:
        search_roots = _roots()
    cand_lower = [c.lower() for c in candidates]
    for root in search_roots:
        if not os.path.isdir(root):
            continue
        for path in glob.glob(os.path.join(root, '**', '*'), recursive=True):
            if os.path.isfile(path) and os.path.basename(path).lower() in cand_lower:
                return path
    return None


def download_if_missing(url, dest):
    if url is None:
        return None
    if not os.path.exists(dest):
        print(f"Downloading {dest} from {url} ...")
        urllib.request.urlretrieve(url, dest)
    return dest


def load_raw_array():
    """Return value_raw [T, N] float32: speed (km/h) or flow (veh/5min)."""
    if CFG['kind'] == 'speed':
        h5 = find_file([f for f in CFG['data_files'] if f.lower().endswith('.h5')])
        if h5 is not None:
            print(f"  H5: {h5}")
            df = pd.read_hdf(h5)
            return np.nan_to_num(df.values.astype(np.float32), nan=0.0)
        csv = find_file([f for f in CFG['data_files'] if f.lower().endswith('.csv')])
        if csv is None:
            csv = download_if_missing(CFG['data_url'], f'{DATASET}.csv')
        if csv is None:
            raise FileNotFoundError(
                f"{DATASET}: no .h5/.csv found and no usable download URL. "
                f"Add the dataset to your Kaggle inputs (expected one of {CFG['data_files']}).")
        print(f"  CSV (timeseries): {csv}")
        df = pd.read_csv(csv, index_col=0)
        return np.nan_to_num(df.values.astype(np.float32), nan=0.0)
    else:  # flow .npz, shape [T, N, C]
        npz = find_file([f for f in CFG['data_files'] if f.lower().endswith('.npz')])
        if npz is None and CFG.get('data_url'):
            npz = download_if_missing(CFG['data_url'], CFG['data_files'][0])
        if npz is None:
            _present = [os.path.basename(p) for r in _roots()
                        if os.path.isdir(r)
                        for p in glob.glob(os.path.join(r, '**', '*.npz'), recursive=True)]
            raise FileNotFoundError(
                f"{DATASET}: no .npz matching {CFG['data_files']} found. "
                f".npz files present: {_present or 'none'}. "
                f"Add the {DATASET} dataset (.npz + edge-list .csv) to your inputs.")
        print(f"  NPZ: {npz}")
        _npz = np.load(npz)
        _key = 'data' if 'data' in _npz.files else _npz.files[0]
        arr = _npz[_key].astype(np.float32)              # [T, N, C]
        ch = CFG.get('channel', 0)
        print(f"  npz shape={arr.shape}, using channel {ch} (flow)")
        return arr[:, :, ch]                              # [T, N]


def load_adjacency(num_nodes):
    """Return binary adjacency [N, N], diagonal zeroed (no self-loop).

    Anti-leak: zeroing the diagonal guarantees the 1-hop / 2-hop neighbour-mean
    features can never read a blind node's own (masked) value back to itself.
    """
    if CFG['kind'] == 'speed':
        pkl = find_file(CFG['adj_files'])
        if pkl is None:
            pkl = download_if_missing(CFG['adj_url'], CFG['adj_files'][0])
        if pkl is None:
            raise FileNotFoundError(f"{DATASET}: adjacency pickle not found ({CFG['adj_files']}).")
        print(f"  Adj PKL: {pkl}")
        with open(pkl, 'rb') as f:
            obj = pickle.load(f, encoding='latin1')
        adj_mx = obj[2] if isinstance(obj, (list, tuple)) and len(obj) >= 3 else obj
        adj_mx = np.asarray(adj_mx, dtype=np.float32)
        if adj_mx.shape[0] != num_nodes:
            raise ValueError(
                f"Adjacency shape {adj_mx.shape} != data ({num_nodes} nodes). "
                f"Wrong pickle picked up -- delete cached adj_mx*.pkl and re-run.")
        adj = (adj_mx > 0.1).astype(np.float32)
    else:  # flow: build from distance.csv edge list (columns: from, to, cost)
        csv = find_file(CFG['adj_files'])
        if csv is None and CFG.get('adj_url'):
            csv = download_if_missing(CFG['adj_url'], CFG['adj_files'][0])
        if csv is None:
            _csvs = [os.path.basename(p) for r in _roots()
                     if os.path.isdir(r)
                     for p in glob.glob(os.path.join(r, '**', '*.csv'), recursive=True)]
            raise FileNotFoundError(
                f"{DATASET}: edge-list not found ({CFG['adj_files']}). "
                f".csv files present: {_csvs or 'none'}.")
        print(f"  Adj CSV (edge list): {csv}")
        ed = pd.read_csv(csv)
        cols = [str(c).lower() for c in ed.columns]
        fi = cols.index('from') if 'from' in cols else 0
        ti = cols.index('to')   if 'to'   in cols else 1
        f_arr = ed.iloc[:, fi].astype(float).astype(int).values
        t_arr = ed.iloc[:, ti].astype(float).astype(int).values
        adj = np.zeros((num_nodes, num_nodes), dtype=np.float32)
        ok = (f_arr >= 0) & (f_arr < num_nodes) & (t_arr >= 0) & (t_arr < num_nodes)
        for a, b in zip(f_arr[ok], t_arr[ok]):
            adj[a, b] = 1.0
            adj[b, a] = 1.0   # symmetric undirected graph
        if adj.sum() == 0:
            raise ValueError(
                f"{DATASET}: edge list '{csv}' produced 0 valid edges for {num_nodes} nodes. "
                f"Wrong distance.csv (node-id range mismatch)? Use the {DATASET}-specific edge list.")
        _oob = int((~ok).sum())
        if _oob > 0.5 * max(len(f_arr), 1):
            print(f"  WARNING: {_oob}/{len(f_arr)} edges fall outside node range [0,{num_nodes}); "
                  f"is '{csv}' the correct edge list for {DATASET}?")
    np.fill_diagonal(adj, 0)
    return adj


print(f"Device: {device} | Datasets: {list(DATASETS)} | Target Sparsity: {SPARSITY*100:.0f}%")





In [ ]:
class STBlock(nn.Module):
    def __init__(self, hidden, n_heads, ff_mult=2, dropout=0.0):
        super().__init__()
        self.temp = nn.TransformerEncoderLayer(
            d_model=hidden, nhead=n_heads,
            dim_feedforward=hidden * ff_mult, dropout=dropout,
            activation='gelu', batch_first=True, norm_first=True)
        self.spat = nn.TransformerEncoderLayer(
            d_model=hidden, nhead=n_heads,
            dim_feedforward=hidden * ff_mult, dropout=dropout,
            activation='gelu', batch_first=True, norm_first=True)

    def forward(self, h, spatial_pad_mask=None):
        B, N, T, H = h.shape
        cm = torch.triu(torch.full((T, T), float('-inf'), device=h.device), diagonal=1)
        h = self.temp(h.reshape(B * N, T, H), src_mask=cm).reshape(B, N, T, H)
        h = h.permute(0, 2, 1, 3).contiguous().reshape(B * T, N, H)
        if spatial_pad_mask is not None:
            h = self.spat(h, src_key_padding_mask=spatial_pad_mask)
        else:
            h = self.spat(h)
        h = h.reshape(B, T, N, H).permute(0, 2, 1, 3).contiguous()
        return h


class MaskedSTTransformerMARST(nn.Module):
    """MARST — Multi-Anchor Residual Spatiotemporal Transformer.

    Generalises a single-anchor residual gate to a *learned mixture*
    of three causal anchors, computed at every (sensor, time) cell:
      a_LOCF : last observed value at this sensor        (recency)
      a_HA   : per-sensor historical average for ToD     (seasonality)
      a_KNN  : mean of currently-observed graph neighbours (spatial context)
    A small head emits a softmax pi in the 2-simplex over the three anchors;
    the blended anchor is  a = pi_LOCF*a_LOCF + pi_HA*a_HA + pi_KNN*a_KNN.
    The spatiotemporal trunk (alternating temporal/spatial STBlock,
    causal temporal mask + mask-aware spatial attention) predicts a residual r,
    and the meta-gate alpha blends it onto the anchor:
        y_hat = a + alpha * r.

    All anchors are z-scored. Anti-leak invariants are preserved: the adjacency
    diagonal is zero, so a_KNN never reads a node's own (masked) value, and
    every signal is causal (no future leakage).

    Input features (9): x, m, t_sin, t_cos, a_LOCF, a_HA, a_KNN, staleness, n_mean_2hop
    """
    def __init__(self, num_nodes, adj, node_means_t, node_stds_t,
                 hidden=128, n_heads=4, n_layers=6, max_T=288, dropout=0.1,
                 soft_locf_decay=0.95):
        super().__init__()
        self.num_nodes = num_nodes
        self.hidden = hidden
        self.soft_locf_decay = soft_locf_decay
        self.register_buffer('adj_static', adj)
        self.register_buffer('node_means', node_means_t)
        self.register_buffer('node_stds',  node_stds_t)

        # Precompute row-normalised 2-hop adjacency (no self-loops)
        with torch.no_grad():
            adj_sq = torch.matmul(adj, adj)
            adj_sq = adj_sq * (1 - torch.eye(num_nodes, device=adj.device))
            self.register_buffer('adj_2hop',
                                 adj_sq / (adj_sq.sum(1, keepdim=True) + 1e-6))

        self.in_proj  = nn.Linear(9, hidden)
        self.node_emb = nn.Parameter(torch.randn(num_nodes, hidden) * 0.02)
        self.pos_emb  = nn.Parameter(torch.randn(max_T,     hidden) * 0.02)
        self.blocks   = nn.ModuleList([STBlock(hidden, n_heads, dropout=dropout)
                                       for _ in range(n_layers)])
        self.final_norm = nn.LayerNorm(hidden)

        # Multi-anchor mixing head: trunk state -> 3 logits -> softmax(pi)
        self.anchor_gate = nn.Sequential(
            nn.Linear(hidden, 32), nn.GELU(),
            nn.Linear(32, 3))
        # Meta-gate and residual head
        self.meta_gate  = nn.Sequential(
            nn.Linear(hidden + 1, 32), nn.GELU(),
            nn.Linear(32, 1), nn.Sigmoid())
        self.residual_head = nn.Sequential(
            nn.Linear(hidden, hidden // 2), nn.GELU(),
            nn.Linear(hidden // 2, 1))
        self.last_alpha = None
        self.last_pi    = None   # [.,3] anchor mix weights (diagnostic)

    def _compute_causal_signals(self, x, m, ha_prior):
        """Causal LOCF, staleness (steps-since-obs/48), soft_locf (EMA toward HA).
        Kept stable so the shared plotting helpers work unchanged."""
        B, N, T = x.shape
        locf      = torch.zeros_like(x)
        staleness = torch.zeros_like(x)
        soft_locf = torch.zeros_like(x)
        cur_locf  = ha_prior[:, :, 0].clone()    # Fix A: HA-init (was zeros — matches standalone LOCF startup)
        cur_stale = torch.zeros(B, N, device=x.device)
        cur_soft  = torch.zeros(B, N, device=x.device)
        decay = self.soft_locf_decay
        for t in range(T):
            obs_t  = x[:, :, t]
            mask_t = m[:, :, t]
            ha_t   = ha_prior[:, :, t]
            obs    = mask_t > 0.5
            cur_locf  = torch.where(obs, obs_t, cur_locf)
            cur_stale = torch.where(obs, torch.zeros_like(cur_stale), cur_stale + 1.0)
            cur_soft  = torch.where(obs, obs_t,
                                    decay * cur_soft + (1.0 - decay) * ha_t)
            locf     [:, :, t] = cur_locf
            staleness[:, :, t] = cur_stale / 48.0
            soft_locf[:, :, t] = cur_soft
        return locf, staleness, soft_locf

    def _knn_anchor(self, x, m, ha_prior):
        """a_KNN: masked mean of currently-observed 1-hop neighbours, z-scored by
        the target node. Where no neighbour is observed, fall back to the HA prior.
        Anti-leak: adj diagonal is zero, so node i never reads its own value."""
        mean_v = self.node_means.view(1, -1, 1)
        std_v  = self.node_stds .view(1, -1, 1)
        x_kmh  = x * std_v + mean_v                          # de-z-score per node
        num    = torch.matmul(self.adj_static, x_kmh * m)    # sum observed nbr kmh
        den    = torch.matmul(self.adj_static, m)            # count observed nbrs
        knn_z  = (num / (den + 1e-6) - mean_v) / std_v       # z-score by target node
        return torch.where(den > 0, knn_z, ha_prior)         # fallback: seasonality

    def forward(self, x, m, t_sin, t_cos, ha_prior):
        B, N, T = x.shape
        mean_v = self.node_means.view(1, -1, 1)
        std_v  = self.node_stds .view(1, -1, 1)

        locf, staleness, soft_locf = self._compute_causal_signals(x, m, ha_prior)
        locf_kmh = locf * std_v + mean_v

        # Three causal anchors (all z-scored)
        a_locf = locf                                    # recency
        a_ha   = ha_prior                                # seasonality
        a_knn  = self._knn_anchor(x, m, ha_prior)        # spatial (observed nbrs)

        # 2-hop neighbour mean of LOCF (z-scored) -- extra spatial context feature
        n_mean_2hop = (torch.matmul(self.adj_2hop, locf_kmh) - mean_v) / std_v

        feat = torch.stack([x, m, t_sin, t_cos, a_locf, a_ha, a_knn,
                            staleness, n_mean_2hop], dim=-1)
        h = self.in_proj(feat)
        h = h + self.node_emb.view(1, N, 1, self.hidden)
        h = h + self.pos_emb[:T].view(1, 1, T, self.hidden)

        spatial_pad_mask = (m == 0).permute(0, 2, 1).contiguous().view(B * T, N)
        for blk in self.blocks:
            h = blk(h, spatial_pad_mask=spatial_pad_mask)
        h = self.final_norm(h)

        # Multi-anchor mixture: pi in the 2-simplex over [LOCF, HA, KNN]
        pi      = torch.softmax(self.anchor_gate(h), dim=-1)     # [B,N,T,3]
        anchors = torch.stack([a_locf, a_ha, a_knn], dim=-1)     # [B,N,T,3]
        blended = (pi * anchors).sum(dim=-1)                     # [B,N,T]
        self.last_pi = pi.detach().reshape(-1, 3)

        # Residual correction gated by alpha (meta-gate)
        residual  = self.residual_head(h).squeeze(-1)
        adj_m_obs = torch.matmul(self.adj_static, m)
        n_obs     = adj_m_obs / (self.adj_static.sum(dim=-1, keepdim=True) + 1e-8)
        alpha     = self.meta_gate(torch.cat([h, n_obs.unsqueeze(-1)], dim=-1)).squeeze(-1)
        self.last_alpha = alpha.detach()

        return blended + alpha * residual


class MaskedSTTransformerMARSTAblate(nn.Module):
    """MARST with per-anchor on/off switches for the anchor-mixture ablation.

    Anchors: a_LOCF (recency), a_HA (seasonality), a_KNN (spatial). A disabled
    anchor is removed from BOTH the input feature stack and the softmax mixture.
    With all three on this is identical to MaskedSTTransformerMARST. At least one
    anchor must stay on. last_pi is always reported as a padded [.,3] vector
    (zeros in disabled slots) so shared train/eval helpers work unchanged.

    Fixed input features (6): x, m, t_sin, t_cos, staleness, n_mean_2hop
    plus the active anchors (1-3). Anti-leak invariants identical to MARST.
    """
    def __init__(self, num_nodes, adj, node_means_t, node_stds_t,
                 hidden=128, n_heads=4, n_layers=6, max_T=288, dropout=0.1,
                 soft_locf_decay=0.95, use_locf=True, use_ha=True, use_knn=True):
        super().__init__()
        assert use_locf or use_ha or use_knn, "MARST ablation needs >=1 anchor"
        self.num_nodes = num_nodes
        self.hidden = hidden
        self.soft_locf_decay = soft_locf_decay
        self.use_locf, self.use_ha, self.use_knn = use_locf, use_ha, use_knn
        self.register_buffer('adj_static', adj)
        self.register_buffer('node_means', node_means_t)
        self.register_buffer('node_stds',  node_stds_t)
        with torch.no_grad():
            adj_sq = torch.matmul(adj, adj)
            adj_sq = adj_sq * (1 - torch.eye(num_nodes, device=adj.device))
            self.register_buffer('adj_2hop', adj_sq / (adj_sq.sum(1, keepdim=True) + 1e-6))

        self.n_anchor = int(use_locf) + int(use_ha) + int(use_knn)
        self.in_proj  = nn.Linear(6 + self.n_anchor, hidden)
        self.node_emb = nn.Parameter(torch.randn(num_nodes, hidden) * 0.02)
        self.pos_emb  = nn.Parameter(torch.randn(max_T,     hidden) * 0.02)
        self.blocks   = nn.ModuleList([STBlock(hidden, n_heads, dropout=dropout)
                                       for _ in range(n_layers)])
        self.final_norm = nn.LayerNorm(hidden)
        self.anchor_gate = nn.Sequential(
            nn.Linear(hidden, 32), nn.GELU(), nn.Linear(32, self.n_anchor))
        self.meta_gate  = nn.Sequential(
            nn.Linear(hidden + 1, 32), nn.GELU(), nn.Linear(32, 1), nn.Sigmoid())
        self.residual_head = nn.Sequential(
            nn.Linear(hidden, hidden // 2), nn.GELU(), nn.Linear(hidden // 2, 1))
        self.last_alpha = None
        self.last_pi    = None

    def _compute_causal_signals(self, x, m, ha_prior):
        B, N, T = x.shape
        locf = torch.zeros_like(x); staleness = torch.zeros_like(x); soft_locf = torch.zeros_like(x)
        cl = ha_prior[:, :, 0].clone()    # Fix A: HA-init (was zeros — matches main MARST + standalone LOCF)
        cs = torch.zeros(B, N, device=x.device)
        cf = torch.zeros(B, N, device=x.device)
        d = self.soft_locf_decay
        for t in range(T):
            o = m[:, :, t] > 0.5; xt = x[:, :, t]; ht = ha_prior[:, :, t]
            cl = torch.where(o, xt, cl)
            cs = torch.where(o, torch.zeros_like(cs), cs + 1.0)
            cf = torch.where(o, xt, d * cf + (1.0 - d) * ht)
            locf[:, :, t] = cl; staleness[:, :, t] = cs / 48.0; soft_locf[:, :, t] = cf
        return locf, staleness, soft_locf

    def _knn_anchor(self, x, m, ha_prior):
        mean_v = self.node_means.view(1, -1, 1); std_v = self.node_stds.view(1, -1, 1)
        x_kmh  = x * std_v + mean_v
        num    = torch.matmul(self.adj_static, x_kmh * m)
        den    = torch.matmul(self.adj_static, m)
        knn_z  = (num / (den + 1e-6) - mean_v) / std_v
        return torch.where(den > 0, knn_z, ha_prior)

    def forward(self, x, m, t_sin, t_cos, ha_prior):
        B, N, T = x.shape
        mean_v = self.node_means.view(1, -1, 1); std_v = self.node_stds.view(1, -1, 1)
        locf, staleness, soft_locf = self._compute_causal_signals(x, m, ha_prior)
        locf_kmh = locf * std_v + mean_v
        a_locf = locf
        a_ha   = ha_prior
        a_knn  = self._knn_anchor(x, m, ha_prior)
        n_mean_2hop = (torch.matmul(self.adj_2hop, locf_kmh) - mean_v) / std_v

        active, slots = [], []
        if self.use_locf: active.append(a_locf); slots.append(0)
        if self.use_ha:   active.append(a_ha);   slots.append(1)
        if self.use_knn:  active.append(a_knn);  slots.append(2)

        feat = torch.stack([x, m, t_sin, t_cos] + active + [staleness, n_mean_2hop], dim=-1)
        h = self.in_proj(feat)
        h = h + self.node_emb.view(1, N, 1, self.hidden)
        h = h + self.pos_emb[:T].view(1, 1, T, self.hidden)
        spatial_pad_mask = (m == 0).permute(0, 2, 1).contiguous().view(B * T, N)
        for blk in self.blocks:
            h = blk(h, spatial_pad_mask=spatial_pad_mask)
        h = self.final_norm(h)

        pi      = torch.softmax(self.anchor_gate(h), dim=-1)   # [B,N,T,n_anchor]
        anchors = torch.stack(active, dim=-1)                  # [B,N,T,n_anchor]
        blended = (pi * anchors).sum(dim=-1)
        full_pi = torch.zeros(*pi.shape[:-1], 3, device=h.device)  # padded diag [.,3]
        for j, s in enumerate(slots):
            full_pi[..., s] = pi[..., j]
        self.last_pi = full_pi.detach().reshape(-1, 3)

        residual  = self.residual_head(h).squeeze(-1)
        adj_m_obs = torch.matmul(self.adj_static, m)
        n_obs     = adj_m_obs / (self.adj_static.sum(dim=-1, keepdim=True) + 1e-8)
        alpha     = self.meta_gate(torch.cat([h, n_obs.unsqueeze(-1)], dim=-1)).squeeze(-1)
        self.last_alpha = alpha.detach()
        return blended + alpha * residual





## Models (Fair, Causal Evaluation)

All models receive **only the 20% observed sensor readings** (`x * m_eff`) as input.
No model has access to the held-out 80% or to future time steps. Causality is enforced
architecturally (upper-triangular attention masks; left-padded dilated convolutions;
forward-only recurrence) and verified empirically by the leak audit.

**Baselines**

| # | Model | Family | Graph? | Causal? |
|---|-------|--------|--------|---------|
| 1 | Historical Average (HA) | Statistical | No | ✓ |
| 2 | LOCF | Statistical | No | ✓ |
| 3 | Global Mean | Statistical | No | ✓ |
| 4 | Ridge Regression | Linear | No | ✓ |
| 5 | KNN Imputer (masked-distance, train-set neighbours) | Non-param | No | ✓ |
| 6 | MLP (per-node + node-emb) | Neural | No | ✓ |
| 7 | LSTM (per-node) | RNN | No | ✓ |
| 8 | 2L-LSTM (per-node, causal) | RNN | No | ✓ |
| 9 | GRU (per-node) | RNN | No | ✓ |
| 10 | TCN (causal dilated) | Conv | No | ✓ |
| 11 | SAITS (causal Attn + node-emb) | Transformer | No | ✓ |
| 12 | **BRITS (forward, causal)** | RNN+imputation | No | ✓ |
| 13 | DCRNN (DiffGCN + GRU) | GNN+RNN | Yes | ✓ |
| 14 | GWN (Adaptive GCN + Gated TCN) | GNN+TCN | Yes | ✓ |

**Recent causal spatiotemporal baselines** (faithful implementations adapted to the streaming protocol)

| # | Model | Venue | Mechanism | Causal? |
|---|-------|-------|-----------|---------|
| 15 | **STID** | CIKM 2022 | node + time-of-day identity embeddings + causal TCN | ✓ |
| 16 | **DLinear** | AAAI 2023 | trend/seasonal decomposition + causal FIR filters | ✓ |
| 17 | **PatchTST** | ICLR 2023 | patching + causal-masked Transformer | ✓ |
| 18 | **iTransformer** | ICLR 2024 | cross-sensor attention + causal temporal encoder | ✓ |

**Our model**

| # | Model | Anchor | Graph? | Causal? |
|---|-------|--------|--------|---------|
| 19 | **MARST** (ours) | multi-anchor LOCF/HA/KNN softmax mix + meta-gate | Yes | ✓ |


## Recent Causal Baselines — STID · DLinear · PatchTST · iTransformer
Faithful implementations of four recent models, each adapted to the shared **causal** protocol (a prediction at time *t* uses only observed values at times ≤ *t*; verified by a future-perturbation test). They reuse `train_g`/`eval_g`, so masks, HA-fill, de-normalisation and eval seeds are identical to every other model.

## MARST — Multi-Anchor Residual Spatiotemporal Transformer

MARST is our proposed model. Over a spatiotemporal trunk (alternating temporal/spatial `STBlock` with a causal temporal mask and mask-aware spatial attention), a residual head and meta-gate `α`, MARST replaces a single fixed imputation anchor with a **learned mixture of three causal anchors** at every (sensor, time) cell:

- **a_LOCF** — last observed value at the sensor (recency)
- **a_HA** — per-sensor historical average for the time-of-day (seasonality)
- **a_KNN** — mean of *currently-observed* graph neighbours (spatial context)

A small head emits a softmax `π ∈ Δ²`; the blended anchor is `a = π_LOCF·a_LOCF + π_HA·a_HA + π_KNN·a_KNN`, and the prediction is `ŷ = a + α·r`. Anti-leak invariants hold: the anchors are causal and the adjacency diagonal is zero, so a blind node never reads its own masked value. Hyperparameters: hidden 128, 6 layers, 800 epochs, 60%→80% masking curriculum.

MARST is trained over **3 initialization seeds** (`TRAIN_SEEDS`); the reported MAE is **mean ± std across training seeds** (each averaged over the 5 shared eval masks). The representative first-seed run is reused for the qualitative figures and the extended-metrics table.

## Leak Audit
Empirical anti-leak check run after all models are trained. It corrupts the held-out ground truth by a large random amount and confirms **no** model's predictions change at the held-out positions (so held-out values / offline sensors never enter any model, MARST included), then perturbs the future and confirms MARST's past predictions are unchanged (temporal causality).

## Sparsity Sensitivity

Evaluate the **trained** MARST (against LOCF and Historical-Average references) as the fraction of blind sensors grows. Training sparsity was fixed at 80%; this probes robustness/extrapolation. Same anti-leak masking: only `(masked) & (valid)` positions are scored, blind sensors are zeroed.

## MARST Anchor-Mixture Ablation

Leave-one-out (and keep-one) over MARST's three causal anchors -- **LOCF** (recency),
**HA** (seasonality), **KNN** (spatial). Each variant trains a MARST whose anchor set
is restricted; the trunk, residual head, meta-gate, training budget, masking
curriculum, and seed are identical to the full model. `dMAE > 0` means removing /
restricting that anchor *hurt* (it was contributing). One model per variant is
trained at seed 0 and evaluated over the shared `EVAL_SEEDS` masks. Anti-leak invariants are preserved in every variant.

## Missing-Pattern Robustness

Re-evaluates the already-trained models (no retraining) at the configured sparsity (80%) under
three controlled missing patterns. We avoid strict MCAR/MAR/MNAR labels because none of our
synthetic patterns satisfy the textbook definitions exactly; we instead describe the *structure*
each pattern imposes.

| Pattern | Construction | Structure |
|---------|--------------|-----------|
| `point` | i.i.d. Bernoulli per (sensor, timestep) cell | unstructured (closest to MCAR) |
| `block` | contiguous time-blocks held out per sensor; block onsets uniform | temporally structured |
| `sensor` | whole sensors offline for the full eval window | spatially structured |

**Caveat on `sensor`.** At 80% sparsity ~80% of sensors are offline simultaneously. The remaining
active sensors may not form a connected subgraph for some seeds, which disadvantages graph
models relative to per-node baselines. We do not exclude these seeds — the result is reported
as-is.

Same `EVAL_SEEDS`, same eval window, same de-normalisation. Reports per-pattern MAE per model.


## Per-Dataset Pipeline — `run_dataset(name)`
All the methodology above (data loading, every baseline, the recent causal models, MARST, evaluation, leak audit, per-dataset save) is wrapped in **`run_dataset(name)`** so the whole study can run for all four datasets in one execution. Model classes and shared config are defined once (cells above); `run_dataset` sets the per-dataset state as module globals so the existing helpers work unchanged.

In [ ]:
def run_dataset(name):
    """Full per-dataset pipeline: load data, train all baselines + the
    recent causal models + MARST, evaluate, run the leak audit, and write
    results_<DATASET>.json. Globals persist so the figure cells below show
    the last dataset run; the cross-dataset cells read the saved JSONs."""
    global ANCHOR_ABLATION_EPOCHS, A_t, BATCH_SIZE_MARST, BLOCK_LEN, TwoLayerLSTM, CFG, CLAMP_HI, CLAMP_LO, CausalConv, D, DATASET, DATASET_NAME, DCRNNLite, DLinear, DiffGCN, EXT_METRICS, EXT_PRED, F, GE, GH, GLR, GRUNet, GWNLite, HIDDEN_DIM_MARST, HIT_EPS_MAX, HIT_TOL, ITransformer, K_NEIGH, LSTM, MARST_ABLATIONS, MARST_ANCHOR_ABLATION, MARST_ANCHOR_ABLATION_SAVE, MISSING_PATTERNS, MISSING_PATTERN_SAVE, MLP, NBT, NE, NH, NLR, NUM_NODES, N_LAYERS_MARST, OURS, PATTERN_MODELS, PATTERN_RESULTS, PatchTST, REGIME_EDGES, RESULTS, Ridge, SAITSLite, SPARSITY_LEVELS, SPARSITY_SAVE, SPARSITY_SWEEP, STEPS_PER_DAY, STID, TCNet, TRAIN_EPOCHS_MARST, TRAIN_SEEDS, VALUE_NAME, VALUE_UNIT, W, Xtr, _EL, _ES, _PRED, _W, _arr, _arr2list, _best_name, _causal_avg, _colors, _cos, _ctor, _d, _dc, _e, _eval_at_sparsity, _eval_st_mae, _flags, _full, _graph, _ha, _ha_fn, _hat, _held, _idx, _k, _kind, _lead_num, _m, _maes, _me, _means, _mnt, _n_params_marst, _name, _names, _net, _net_ab, _obj, _ok, _order, _pat, _pk, _pred_classical, _pred_graph, _pred_graph_leak, _pred_st, _pred_st_leak, _run_model_ext_full, _sc, _seed, _sin, _sm, _st, _stds, _stt, _tc, _ti, _title, _tk_full, _train_st, _ts, _v, _vt, _vv, _wb, _x, _xc, _xf, _xn, _xt, adj, adj_norm, arr, astgcn, ax, clf, cnts, cols_ext, ctor_marst, d, dcrnn, diff, dlin, enough, eval_fn, eval_g, eval_nw, extended_metrics, f, fig, gwn, ha_mae, ha_prior, itrf, j, keys, knn_masked, label, locf, m, mae_marst_seeds, make_eval_mask_np, make_pattern_mask_np, marst_mask_mat, mk, mw, n, net, net_marst, nn, node_means, node_means_t, node_stds, node_stds_t, np, out_path, p, plt, ptst, re, ridge, ridge_models, row, s, save_obj, sel, si, slot_idx, sp, speed_gpu, speed_norm, speed_raw, stid, sub_data, sub_valid, sums, tag, title, tod_mean, torch, tr_norm, tr_ref, tr_valid, train_g, train_nw, tseed, tv_ref, valid_gpu, valid_raw, vals, value_raw, w, warnings, ytr
    DATASET = name
    assert DATASET in DATASETS, f'unknown dataset {name!r}'
    CFG = DATASETS[DATASET]
    DATASET_NAME = DATASET
    STEPS_PER_DAY = CFG['steps_per_day']
    VALUE_UNIT = CFG['unit']
    VALUE_NAME = 'Speed' if CFG['kind'] == 'speed' else 'Flow'
    print('\n' + '#' * 74)
    print(f"#  RUN DATASET: {DATASET}  ({CFG['kind']}, {int(SPARSITY*100)}% sparsity)")
    print('#' * 74)
    print(f"Loading data for {DATASET_NAME} ...")
    value_raw = load_raw_array()[:WINDOW]
    NUM_NODES = value_raw.shape[1]
    print(f"  Detected: T={value_raw.shape[0]}, N={NUM_NODES}, kind={CFG['kind']}")

    # Validity mask. Speed datasets: 0 == missing. Flow datasets: fully observed.
    if CFG['kind'] == 'speed':
        valid_raw = (value_raw > 0).astype(np.float32)
    else:
        valid_raw = np.ones_like(value_raw, dtype=np.float32)
    print(f"  Valid fraction: {valid_raw.mean():.3f}")

    # Legacy variable name kept so all downstream cells work unchanged.
    speed_raw = value_raw

    # Clamp range for de-normalised predictions/targets (speed=mph, flow=veh/5min).
    if CFG['kind'] == 'speed':
        CLAMP_LO, CLAMP_HI = 0.0, 120.0
    else:
        CLAMP_LO, CLAMP_HI = 0.0, float(value_raw.max()) * 1.5
    print(f"  Clamp range: [{CLAMP_LO:.1f}, {CLAMP_HI:.1f}]")

    # Error-regime buckets and Hit tolerances (data-driven per dataset).
    # Speed: jam = slowest 20% (true < p20).  Flow: jam = busiest 33% (true > p67).
    # Hit tolerances are 5% / 10% of the median value, floored sensibly per unit.
    _vt = speed_raw[:TRAIN_END][valid_raw[:TRAIN_END] > 0]
    if CFG['kind'] == 'speed':
        REGIME_EDGES = tuple(float(np.round(p)) for p in np.percentile(_vt, [20, 50]))
    else:
        REGIME_EDGES = tuple(float(np.round(p)) for p in np.percentile(_vt, [33, 67]))
    _sc = float(np.median(_vt))
    if CFG['kind'] == 'speed':
        HIT_TOL = (max(1.0, float(np.round(0.05 * _sc))),
                   max(2.0, float(np.round(0.10 * _sc))))
    else:
        HIT_TOL = (max(1.0, float(np.round(0.05 * _sc))),
                   max(2.0, float(np.round(0.10 * _sc))))
    HIT_EPS_MAX = HIT_TOL[1] * 2.0        # x-range for the hit-rate curve
    print(f"  Regime edges: {REGIME_EDGES} {VALUE_UNIT} | Hit tol: {HIT_TOL} {VALUE_UNIT}")

    # Per-node mean/std over VALID TRAIN entries only (no eval-window leakage).
    node_means = np.zeros(NUM_NODES, dtype=np.float32)
    node_stds = np.ones(NUM_NODES, dtype=np.float32)
    for n in range(NUM_NODES):
        vals = speed_raw[:TRAIN_END, n][valid_raw[:TRAIN_END, n] > 0]
        if len(vals) > 0:
            node_means[n] = vals.mean()
            node_stds[n] = max(float(vals.std()), 1e-3) + 1e-8

    speed_norm = (speed_raw - node_means) / node_stds

    # HA prior per (node, time-of-day) over VALID TRAIN entries only.
    slot_idx = np.arange(len(speed_norm)) % STEPS_PER_DAY
    tod_mean = np.zeros((NUM_NODES, STEPS_PER_DAY), dtype=np.float32)
    for s in range(STEPS_PER_DAY):
        sel = slot_idx[:TRAIN_END] == s
        sub_data = speed_norm[:TRAIN_END][sel]
        sub_valid = valid_raw[:TRAIN_END][sel]
        sums = (sub_data * sub_valid).sum(axis=0)
        cnts = sub_valid.sum(axis=0) + 1e-8
        tod_mean[:, s] = sums / cnts

    ha_prior = torch.tensor(tod_mean[:, slot_idx].T, dtype=torch.float32).to(device)
    speed_gpu = torch.tensor(speed_norm, dtype=torch.float32).to(device)
    valid_gpu = torch.tensor(valid_raw, dtype=torch.float32).to(device)
    node_means_t = torch.tensor(node_means, dtype=torch.float32).to(device)
    node_stds_t = torch.tensor(node_stds, dtype=torch.float32).to(device)

    adj = load_adjacency(NUM_NODES)
    D = np.diag(1.0 / np.sqrt(adj.sum(axis=1) + 1e-8))
    adj_norm = D @ adj @ D
    A_t = torch.tensor(adj_norm, dtype=torch.float32).to(device)

    print(f"Data and adjacency ready. NUM_NODES={NUM_NODES}, avg degree={adj.sum(1).mean():.2f}, edges={int(adj.sum())}")


    import numpy as np, torch

    RESULTS = {}
    EXT_PRED = {}   # {label: (pred_flat, true_flat)} held-out points over EVAL_SEEDS
    MODEL_STATS = {}     # {label: {'params': int, 'kind': str}} for efficiency table
    TRAINED_MODELS = {}  # {label: (model_or_callable, kind)} for multi-rate sweep

    # ── Shared mask generator (ALL models must use this) ──────────────────────
    # Uses numpy default_rng so masks are identical regardless of torch state.
    def make_eval_mask_np(seed, EL, N, sparsity=None):
        """Return bool mask [EL, N] — True = observed (20%), False = held-out (80%)."""
        rng = np.random.default_rng(seed)
        sparsity = SPARSITY if sparsity is None else sparsity
        mask = (rng.random((EL, N)) > sparsity).astype(np.float32)  # [T, N]
        # Guarantee >=1 observed sensor per timestep: an all-blind step makes the
        # spatial attention key-padding mask all-True -> softmax(-inf) -> NaN that
        # then spreads across time. At normal sparsity (large N) this never fires,
        # so existing results and per-seed reproducibility are unchanged.
        empty = mask.sum(axis=1) == 0
        if empty.any():
            rows = np.where(empty)[0]
            mask[rows, rng.integers(0, N, size=rows.size)] = 1.0
        return mask

    # ── Shared eval helper (numpy, for non-graph models) ──────────────────────
    def eval_fn(pred_fn, label):
        TRAINED_MODELS[label] = (pred_fn, 'classical')
        ES, EL = EVAL_START, EVAL_LEN
        x_ev = speed_norm[ES:ES+EL]   # [T,N]
        v_ev = valid_raw [ES:ES+EL]
        maes = []; _pp = []; _tt = []
        for seed in EVAL_SEEDS:
            m_ev = make_eval_mask_np(seed, EL, NUM_NODES)  # [T,N]
            sm   = (m_ev == 0) & (v_ev > 0)
            if not sm.any(): continue
            idx  = np.arange(ES, ES+EL)
            p    = np.clip(pred_fn(x_ev.T, v_ev.T, m_ev.T, idx), CLAMP_LO, CLAMP_HI)
            t    = np.clip(x_ev.T * node_stds[:,None] + node_means[:,None], CLAMP_LO, CLAMP_HI)
            maes.append(np.abs(p[sm.T] - t[sm.T]).mean())
            _pp.append(p[sm.T]); _tt.append(t[sm.T])
        arr = np.array(maes)
        print(f"{label:<52}  MAE: {arr.mean():.4f} +/- {arr.std():.4f}")
        RESULTS[label] = arr
        EXT_PRED[label] = (np.concatenate(_pp), np.concatenate(_tt))

    # m1 sanity check: confirm EVAL_SEEDS produce sufficiently distinct masks
    # (pairwise Jaccard overlap < 0.5 — otherwise the "5 seeds" is effectively fewer).
    _mask_seeds = [make_eval_mask_np(s, EVAL_LEN, NUM_NODES) for s in EVAL_SEEDS]
    _max_jac = 0.0
    for i in range(len(_mask_seeds)):
        for j in range(i+1, len(_mask_seeds)):
            mi, mj = _mask_seeds[i] == 0, _mask_seeds[j] == 0   # held-out cells
            inter = (mi & mj).sum(); union = (mi | mj).sum() + 1e-9
            _max_jac = max(_max_jac, inter / union)
    # Expected Jaccard for two i.i.d. Bernoulli(SPARSITY) masks is SPARSITY/(2-SPARSITY).
    # At 80% sparsity that's 0.667 by construction; we only flag if observed is 10% above that
    # (i.e., effectively degenerate seeds producing near-identical masks).
    _expected_jac = SPARSITY / (2 - SPARSITY)
    print(f"  EVAL_SEEDS mask-overlap check: max pairwise Jaccard = {_max_jac:.3f}  "
          f"(expected for i.i.d. {int(SPARSITY*100)}% masks: {_expected_jac:.3f}, "
          f"tolerance: +0.10)")
    assert _max_jac < _expected_jac + 0.10, (
        f"EVAL_SEEDS masks overlap suspiciously much: max Jaccard {_max_jac:.3f} "
        f"vs expected {_expected_jac:.3f} for i.i.d. seeds at sparsity={SPARSITY:.2f}")

    # 1. Historical Average
    eval_fn(lambda xn,vn,mn,idx: tod_mean[:,idx%STEPS_PER_DAY]*node_stds[:,None]+node_means[:,None],
            "1. Historical Average (HA)")

    # 2. LOCF
    def locf(xn, vn, mn, idx):
        N, T = xn.shape
        out  = tod_mean[:,idx%STEPS_PER_DAY]*node_stds[:,None]+node_means[:,None]
        last = out[:,0].copy()
        for t in range(T):
            obs  = (mn[:,t]>0)&(vn[:,t]>0)
            last = np.where(obs, xn[:,t]*node_stds+node_means, last)
            out[:,t] = last
        return out
    eval_fn(locf, "2. LOCF (Last-Obs Carried Forward)")

    # 3. Global mean
    eval_fn(lambda xn,vn,mn,idx: np.broadcast_to(node_means[:,None],xn.shape).copy(),
            "3. Global Mean (per-node train mean)")


    from sklearn.linear_model import Ridge
    import warnings; warnings.filterwarnings("ignore")

    # 4. Ridge Regression (unchanged)
    print("Fitting Ridge regressors...")
    ridge_models = []
    for n in range(NUM_NODES):
        si  = slot_idx[:TRAIN_END]
        Xtr = np.column_stack([np.sin(2*np.pi*si/STEPS_PER_DAY),
                               np.cos(2*np.pi*si/STEPS_PER_DAY),
                               tod_mean[n, si]])
        ytr = speed_norm[:TRAIN_END, n]
        mk  = valid_raw[:TRAIN_END, n] > 0
        clf = Ridge(alpha=1.0); clf.fit(Xtr[mk], ytr[mk])
        ridge_models.append(clf)

    def ridge(xn, vn, mn, idx):
        N, T = xn.shape; out = np.zeros((N,T),dtype=np.float32)
        for n in range(N):
            Xn = np.column_stack([np.sin(2*np.pi*(idx%STEPS_PER_DAY)/STEPS_PER_DAY),
                                  np.cos(2*np.pi*(idx%STEPS_PER_DAY)/STEPS_PER_DAY),
                                  tod_mean[n, idx%STEPS_PER_DAY]])
            out[n] = ridge_models[n].predict(Xn)*node_stds[n]+node_means[n]
        return out
    eval_fn(ridge, "4. Node-wise Ridge Regression")

    # 5. KNN Imputer — causal spatial imputation with masked Euclidean distance.
    #
    # Why the previous fix failed: at 80% sparsity, a 325-dim query has ~260 zeros
    # (unobserved sensors set to 0). Standard Euclidean distance treats those zeros
    # as signal, so neighbours are found based on the zero pattern, not speed values.
    #
    # Fix: compute distance only over the ~65 observed dimensions, normalised by
    # the number of shared observed sensors. This is the standard 'missing-value
    # aware' nearest-neighbour approach used in the imputation literature.
    # Causal: each timestep is treated independently (spatial, not temporal KNN).
    print("Fitting KNN (masked-distance spatial imputer)...")
    tr_norm = speed_norm[:TRAIN_END].copy()           # [T_tr, N] z-scored
    tr_valid = (valid_raw[:TRAIN_END] > 0).astype(np.float32)  # [T_tr, N]
    # Keep only training rows with >=10% valid sensors
    enough    = tr_valid.mean(axis=1) >= 0.10
    tr_ref    = tr_norm[enough]          # [M, N]
    tv_ref    = tr_valid[enough]         # [M, N]  1=valid, 0=missing
    K_NEIGH   = 5

    def knn_masked(xn, vn, mn, idx):
        """Causal spatial KNN with masked Euclidean distance.
        xn,vn,mn: [N,T] numpy arrays (z-scored / valid / observed-mask).
        Returns [N,T] in the dataset's native unit (speed=mph, flow=veh/5min).
        Each timestep is solved independently — no future used.
        """
        N, T = xn.shape
        out = (tod_mean[:, idx % STEPS_PER_DAY] * node_stds[:, None]
               + node_means[:, None]).copy()   # HA fallback [N,T]
        for t in range(T):
            obs = (mn[:, t] > 0) & (vn[:, t] > 0)   # [N] bool — truly observed
            if obs.sum() < 2:
                continue   # too few observed sensors — keep HA
            q_vals = xn[:, t]          # [N] z-scored query row

            # Masked squared distance: average over shared valid dimensions only
            # shared[m,n] = 1 if both query and ref row m have sensor n valid
            shared = tv_ref * obs.astype(np.float32)  # [M, N]
            n_shared = shared.sum(axis=1) + 1e-8      # [M]
            diff = (tr_ref - q_vals) * shared         # [M, N] — zero out unshared
            dist = (diff ** 2).sum(axis=1) / n_shared  # [M] mean-sq dist over shared

            # k nearest neighbours
            knn_idx = np.argpartition(dist, K_NEIGH)[:K_NEIGH]  # [K]
            neigh_v  = tr_ref[knn_idx]    # [K, N] z-scored
            neigh_ok = tv_ref[knn_idx]    # [K, N] validity

            # Fill: observed sensors keep their value; missing → weighted avg of neighbours
            neigh_sum = (neigh_v * neigh_ok).sum(0)     # [N]
            neigh_cnt = neigh_ok.sum(0) + 1e-8          # [N]
            imputed   = neigh_sum / neigh_cnt            # [N] z-scored
            row_norm  = np.where(obs, q_vals, imputed)   # [N] z-scored
            out[:, t] = np.clip(row_norm * node_stds + node_means, CLAMP_LO, CLAMP_HI)
        return out

    eval_fn(knn_masked, "5. KNN Imputer (k=5, masked-dist)")

    # ============================================================================
    #  Q12 — Ridge α + KNN k sensitivity (no training; refit + re-eval).
    # ============================================================================
    def _eval_classical_collect(pred_fn):
        """Per-eval-seed MAE array for a classical pred_fn(xn, vn, mn, idx). No dict writes."""
        ES, EL = EVAL_START, EVAL_LEN
        x_ev = speed_norm[ES:ES+EL]; v_ev = valid_raw[ES:ES+EL]
        maes = []
        for seed in EVAL_SEEDS:
            m_ev = make_eval_mask_np(seed, EL, NUM_NODES)
            sm = (m_ev == 0) & (v_ev > 0)
            if not sm.any():
                continue
            idx = np.arange(ES, ES+EL)
            p = np.clip(pred_fn(x_ev.T, v_ev.T, m_ev.T, idx), CLAMP_LO, CLAMP_HI)
            t = np.clip(x_ev.T * node_stds[:,None] + node_means[:,None], CLAMP_LO, CLAMP_HI)
            maes.append(float(np.abs(p[sm.T] - t[sm.T]).mean()))
        return np.array(maes)

    # Ridge α sweep
    RIDGE_ALPHA_SWEEP = {1.0: RESULTS["4. Node-wise Ridge Regression"].copy()}
    _ridge_models_save = list(ridge_models)
    for _alpha in [0.1, 10.0]:
        _new = []
        for n in range(NUM_NODES):
            si  = slot_idx[:TRAIN_END]
            Xtr = np.column_stack([np.sin(2*np.pi*si/STEPS_PER_DAY),
                                   np.cos(2*np.pi*si/STEPS_PER_DAY),
                                   tod_mean[n, si]])
            ytr = speed_norm[:TRAIN_END, n]
            mk  = valid_raw[:TRAIN_END, n] > 0
            _clf = Ridge(alpha=_alpha); _clf.fit(Xtr[mk], ytr[mk])
            _new.append(_clf)
        ridge_models[:] = _new
        RIDGE_ALPHA_SWEEP[_alpha] = _eval_classical_collect(ridge)
    ridge_models[:] = _ridge_models_save
    print(f"Q12 Ridge α sweep: " + "  ".join(
        f"α={a:>4}->MAE={RIDGE_ALPHA_SWEEP[a].mean():.4f}" for a in sorted(RIDGE_ALPHA_SWEEP)))

    # KNN k sweep — K_NEIGH is read by knn_masked at call time (closure over outer scope)
    KNN_K_SWEEP = {5: RESULTS["5. KNN Imputer (k=5, masked-dist)"].copy()}
    _K_save = K_NEIGH
    for _k in [3, 10]:
        K_NEIGH = _k
        KNN_K_SWEEP[_k] = _eval_classical_collect(knn_masked)
    K_NEIGH = _K_save
    print(f"Q12 KNN k sweep:    " + "  ".join(
        f"k={k:>2}->MAE={KNN_K_SWEEP[k].mean():.4f}" for k in sorted(KNN_K_SWEEP)))


    import torch.nn as nn
    import torch.nn.functional as F

    NE, NH, NLR, NBT = 800, 64, 1e-3, 48

    def train_nw(cls, label, train_seed=0, **kw):
        torch.manual_seed(train_seed)
        np.random.seed(train_seed)
        net = cls(4, NH, **kw).to(device)
        opt = torch.optim.Adam(net.parameters(), lr=NLR)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=NE)
        xg, vg = speed_gpu[:TRAIN_END], valid_gpu[:TRAIN_END]
        T_tr, N = xg.shape
        for ep in range(1, NE+1):
            net.train()
            t0 = np.random.randint(0, T_tr-NBT)
            xb, vb = xg[t0:t0+NBT], vg[t0:t0+NBT]
            idx = torch.arange(t0, t0+NBT, device=device)
            s   = torch.sin(2*np.pi*(idx%STEPS_PER_DAY)/STEPS_PER_DAY)
            c   = torch.cos(2*np.pi*(idx%STEPS_PER_DAY)/STEPS_PER_DAY)
            mb  = (torch.rand(NBT, N, device=device) > SPARSITY).float()
            me  = mb * vb; xi = xb * me
            feat = torch.stack([xi.T, me.T,
                s.unsqueeze(0).expand(N, -1),
                c.unsqueeze(0).expand(N, -1)], dim=-1)   # [N,T,4]
            pred = net(feat).squeeze(-1)                  # [N,T]
            lm = (mb.T == 0) & (vb.T > 0)
            if not lm.any(): continue
            loss = F.smooth_l1_loss(pred[lm], xb.T[lm])  # z-scored targets
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), 0.5)
            opt.step(); sch.step()
        print(f"  {label} trained."); return net

    def _eval_nw_collect(net):
        """Pure per-mask eval. Returns (maes [n_eval_masks], pp [list], tt [list])."""
        net.eval()
        ES, EL = EVAL_START, EVAL_LEN
        xev = speed_gpu[ES:ES+EL]; vev = valid_gpu[ES:ES+EL]
        idx = torch.arange(ES, ES+EL, device=device)
        s = torch.sin(2*np.pi*(idx%STEPS_PER_DAY)/STEPS_PER_DAY)
        c = torch.cos(2*np.pi*(idx%STEPS_PER_DAY)/STEPS_PER_DAY)
        st  = torch.tensor(node_stds,  device=device)
        mn_ = torch.tensor(node_means, device=device)
        maes = []; _pp = []; _tt = []
        with torch.no_grad():
            for seed in EVAL_SEEDS:
                m_ev = torch.tensor(make_eval_mask_np(seed, EL, NUM_NODES), device=device)
                me   = m_ev * vev
                feat = torch.stack([
                    (xev * me).T, me.T,
                    s.unsqueeze(0).expand(NUM_NODES, -1),
                    c.unsqueeze(0).expand(NUM_NODES, -1)], dim=-1)
                pred = net(feat).squeeze(-1)
                pk   = (pred * st[:, None] + mn_[:, None]).clamp(CLAMP_LO, CLAMP_HI)
                tk   = (xev.T * st[:, None] + mn_[:, None]).clamp(CLAMP_LO, CLAMP_HI)
                sm   = (m_ev.T == 0) & (vev.T > 0)
                maes.append(torch.abs(pk[sm] - tk[sm]).mean().item())
                _pp.append(pk[sm].cpu().numpy()); _tt.append(tk[sm].cpu().numpy())
        return np.array(maes), _pp, _tt

    def eval_nw(net, label):
        """Single-seed: kept for back-compat."""
        MODEL_STATS[label]    = {'params': sum(p.numel() for p in net.parameters() if p.requires_grad), 'kind': 'nn'}
        TRAINED_MODELS[label] = (net, 'nw')
        arr, pp, tt = _eval_nw_collect(net)
        print(f"{label:<52}  MAE: {arr.mean():.4f} +/- {arr.std():.4f}")
        RESULTS[label]  = arr
        EXT_PRED[label] = (np.concatenate(pp), np.concatenate(tt))

    def eval_nw_multi(cls, label, n_seeds=N_BASELINE_SEEDS, **kw):
        """Train cls() with n_seeds different seeds, stack MAE matrix [n_seeds, n_eval_masks]."""
        matrix = []
        rep_pp, rep_tt, rep_net = None, None, None
        short = label.split('.', 1)[0].strip()
        for ts in range(n_seeds):
            net = train_nw(cls, short, train_seed=ts, **kw)
            arr, pp, tt = _eval_nw_collect(net)
            matrix.append(arr)
            if ts == 0:
                rep_pp, rep_tt, rep_net = pp, tt, net
            else:
                del net
                if torch.cuda.is_available(): torch.cuda.empty_cache()
        matrix = np.array(matrix)
        seed_means = matrix.mean(axis=1)
        MODEL_STATS[label]    = {'params': sum(p.numel() for p in rep_net.parameters() if p.requires_grad), 'kind': 'nn'}
        TRAINED_MODELS[label] = (rep_net, 'nw')
        RESULTS[label]        = matrix
        EXT_PRED[label]       = (np.concatenate(rep_pp), np.concatenate(rep_tt))
        print(f"{label:<52}  MAE: {matrix.mean():.4f} +/- {seed_means.std():.4f}  "
              f"(n={matrix.size}, between-seed std: {seed_means.std():.4f})")
        return rep_net

    # ── MLP with node embedding ───────────────────────────────────────────────
    # Note: at 80% sparsity, an unobserved node's input is [0,0,sin,cos]
    # — indistinguishable from any other unobserved node at the same timestep.
    # Without a node identity, the MLP predicts the same value for all missing
    # nodes, behaving like a global mean. Adding a learned node embedding gives
    # each sensor a unique fingerprint even when its value is masked out.
    class MLP(nn.Module):
        def __init__(self, F, H, n_nodes=325, **k):
            super().__init__()
            self.node_emb = nn.Embedding(n_nodes, H)
            self.proj = nn.Linear(F, H)
            self.net = nn.Sequential(
                nn.LayerNorm(H * 2),
                nn.Linear(H * 2, H * 2), nn.GELU(),
                nn.LayerNorm(H * 2),
                nn.Linear(H * 2, H),     nn.GELU(),
                nn.Linear(H, 1))
            self._n = n_nodes

        def forward(self, x):   # x: [N, T, F]
            N, T, _ = x.shape
            nids = torch.arange(N, device=x.device)
            ne   = self.node_emb(nids).unsqueeze(1).expand(N, T, -1)  # [N,T,H]
            hf   = self.proj(x)                                        # [N,T,H]
            return self.net(torch.cat([hf, ne], dim=-1))               # [N,T,1]

    class LSTM(nn.Module):
        def __init__(self,F,H,**k):
            super().__init__(); self.r=nn.LSTM(F,H,batch_first=True); self.h=nn.Linear(H,1)
        def forward(self,x): o,_=self.r(x); return self.h(o)

    # 2-layer forward LSTM (bidirectional would leak the future).
    class TwoLayerLSTM(nn.Module):
        def __init__(self,F,H,**k):
            super().__init__()
            self.r=nn.LSTM(F, H, num_layers=2, batch_first=True, dropout=0.1)
            self.h=nn.Linear(H,1)
        def forward(self,x): o,_=self.r(x); return self.h(o)

    class GRUNet(nn.Module):
        def __init__(self,F,H,**k):
            super().__init__(); self.r=nn.GRU(F,H,batch_first=True); self.h=nn.Linear(H,1)
        def forward(self,x): o,_=self.r(x); return self.h(o)

    eval_nw_multi(MLP, "6.  MLP (per-node + node-emb)", n_nodes=NUM_NODES)
    eval_nw_multi(LSTM, "7.  LSTM (per-node)")
    eval_nw_multi(TwoLayerLSTM, "8.  2L-LSTM (per-node, causal)")
    eval_nw_multi(GRUNet, "9.  GRU (per-node)")


    # 10. TCN — causal dilated convolutions: no fix needed.
    class CausalConv(nn.Module):
        def __init__(self,c,k,d):
            super().__init__(); self.p=(k-1)*d; self.c=nn.Conv1d(c,c,k,dilation=d)
        def forward(self,x): return self.c(F.pad(x,(self.p,0)))

    class TCNet(nn.Module):
        def __init__(self,F,H,**k):
            super().__init__()
            self.proj=nn.Linear(F,H)
            self.convs=nn.ModuleList([CausalConv(H,3,2**i) for i in range(4)])
            self.norms=nn.ModuleList([nn.LayerNorm(H) for _ in range(4)])
            self.head=nn.Linear(H,1)
        def forward(self,x):
            h=self.proj(x).permute(0,2,1)
            for cv,nm in zip(self.convs,self.norms):
                r=h; h=F.gelu(nm(cv(h).permute(0,2,1))).permute(0,2,1)+r
            return self.head(h.permute(0,2,1))
    eval_nw_multi(TCNet, "10. TCN (causal dilated, per-node)")

    # 11. SAITS — causal temporal mask + per-node embedding.
    # Same node-embedding rationale as MLP: at 80% sparsity, unobserved nodes all have
    # input [0,0,sin,cos]. Without node identity the transformer cannot distinguish
    # sensors and collapses to a global prediction. Node embedding added.
    class SAITSLite(nn.Module):
        def __init__(self, F, H, n_nodes=325, **k):
            super().__init__()
            self.node_emb = nn.Embedding(n_nodes, H)
            self.proj = nn.Linear(F, H)
            kw = dict(nhead=4, dim_feedforward=H*2, dropout=0.1,
                      activation='gelu', batch_first=True, norm_first=True)
            self.a1 = nn.TransformerEncoderLayer(H, **kw)
            self.a2 = nn.TransformerEncoderLayer(H, **kw)
            self.h1 = nn.Linear(H, 1); self.h2 = nn.Linear(H, 1)
            self.alpha = nn.Parameter(torch.tensor(0.5))
            self._n = n_nodes

        @staticmethod
        def _cmask(T, device):
            return torch.triu(torch.full((T, T), float('-inf'), device=device), diagonal=1)

        def forward(self, x):   # x: [N, T, F]
            N, T, _ = x.shape
            cm  = self._cmask(T, x.device)
            ne  = self.node_emb(torch.arange(N, device=x.device))  # [N, H]
            h   = self.proj(x) + ne.unsqueeze(1)                   # [N, T, H]
            h1  = self.a1(h,  src_mask=cm)
            h2  = self.a2(h1, src_mask=cm)
            a   = torch.sigmoid(self.alpha)
            return a * self.h1(h1) + (1-a) * self.h2(h2)

    eval_nw_multi(SAITSLite, "11. SAITS (causal Attn + node-emb)", n_nodes=NUM_NODES)

    # 12. BRITS (forward, causal) — Cao et al. NeurIPS 2018, adapted to streaming.
    # Forward GRU with imputed-value feedback; the canonical streaming RNN baseline
    # referenced in BayOTIDE (ICML 2024). Full BRITS is bidirectional; this variant
    # is the forward-only causal analog.
    class BRITSForward(nn.Module):
        def __init__(self, F, H, **k):
            super().__init__(); self.H = H
            self.gf   = nn.GRU(F*2, H, batch_first=True)
            self.impf = nn.Linear(H, F)
            self.head = nn.Linear(H, 1)
        def _run(self, x, m, gru, imp):
            N, T, Ff = x.shape
            h = torch.zeros(1, N, self.H, device=x.device); outs = []
            for t in range(T):
                xh  = imp(h.squeeze(0))
                xc  = m[:, t, :] * x[:, t, :] + (1 - m[:, t, :]) * xh
                o, h = gru(torch.cat([xc, m[:, t, :]], -1).unsqueeze(1), h)
                outs.append(o)
            return torch.cat(outs, 1)
        def forward(self, x):
            m  = x[:, :, 1:2].expand_as(x)
            hf = self._run(x, m, self.gf, self.impf)
            return self.head(hf)
    eval_nw_multi(BRITSForward, "12. BRITS (forward GRU only, causal)")


    # Graph baselines: feat [B,N,T,F] + adj A -> [B,N,T]
    GE, GH, GLR = 800, 64, 1e-3

    def train_g(net, label, train_seed=0):
        torch.manual_seed(train_seed)
        np.random.seed(train_seed)
        # Graph models: unobserved nodes previously had x=0
        # fed into graph diffusion, spreading zeros into neighbouring nodes'
        # representations. Fix: fill unobserved node values with HA prior
        # (z-scored). The mask feature still tells the model which nodes are real.
        opt=torch.optim.Adam(net.parameters(),lr=GLR)
        sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=GE)
        xg,vg=speed_gpu[:TRAIN_END],valid_gpu[:TRAIN_END]; T_tr,N=xg.shape
        for ep in range(1,GE+1):
            net.train()
            t0=np.random.randint(0,T_tr-48)
            xb=xg[t0:t0+48].T.unsqueeze(0); vb=vg[t0:t0+48].T.unsqueeze(0)
            ha=ha_prior[t0:t0+48].T.unsqueeze(0)   # [1,N,48] z-scored HA prior
            idx=torch.arange(t0,t0+48,device=device)
            s=torch.sin(2*np.pi*(idx%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,1,-1).expand(1,N,-1)
            c=torch.cos(2*np.pi*(idx%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,1,-1).expand(1,N,-1)
            mb=(torch.rand(1,N,48,device=device)>SPARSITY).float(); me=mb*vb
            # ▶ FIX: unobserved positions filled with HA (not 0) before graph diffusion
            xi = xb*me + ha*(1-me)   # observed: real value; unobserved: HA prior
            feat=torch.stack([xi[0],me[0],s[0],c[0]],dim=-1).unsqueeze(0)
            pred=net(feat,A_t)
            lm=(mb==0)&(vb>0)
            if not lm.any(): continue
            loss=F.smooth_l1_loss(pred[lm],xb[lm])
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(),0.5)
            opt.step(); sch.step()
        print(f"  {label} trained."); return net

    def _eval_g_collect(net):
        """Pure per-mask eval for graph models. Returns (maes, pp, tt)."""
        net.eval()
        ES, EL = EVAL_START, EVAL_LEN
        xev=speed_gpu[ES:ES+EL].T.unsqueeze(0); vev=valid_gpu[ES:ES+EL].T.unsqueeze(0)
        hae=ha_prior[ES:ES+EL].T.unsqueeze(0)
        idx=torch.arange(ES,ES+EL,device=device)
        s=torch.sin(2*np.pi*(idx%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,1,-1).expand(1,NUM_NODES,-1)
        c=torch.cos(2*np.pi*(idx%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,1,-1).expand(1,NUM_NODES,-1)
        st=torch.tensor(node_stds,device=device).view(1,-1,1)
        mn_=torch.tensor(node_means,device=device).view(1,-1,1)
        maes=[]; _pp=[]; _tt=[]
        with torch.no_grad():
            for seed in EVAL_SEEDS:
                m_np = make_eval_mask_np(seed, EL, NUM_NODES)
                mev=torch.tensor(m_np, device=device).T.unsqueeze(0)
                me=mev*vev
                xi = xev*me + hae*(1-me)
                feat=torch.stack([xi[0],me[0],s[0],c[0]],dim=-1).unsqueeze(0)
                pred=net(feat,A_t)
                pk=(pred*st+mn_).clamp(CLAMP_LO, CLAMP_HI); tk=(xev*st+mn_).clamp(CLAMP_LO, CLAMP_HI)
                sm=(mev==0)&(vev>0)
                maes.append(torch.abs(pk[sm]-tk[sm]).mean().item())
                _pp.append(pk[sm].cpu().numpy()); _tt.append(tk[sm].cpu().numpy())
        return np.array(maes), _pp, _tt

    def eval_g(net, label):
        """Single-seed back-compat."""
        MODEL_STATS[label]    = {'params': sum(p.numel() for p in net.parameters() if p.requires_grad), 'kind': 'nn'}
        TRAINED_MODELS[label] = (net, 'graph')
        arr, pp, tt = _eval_g_collect(net)
        print(f"{label:<52}  MAE: {arr.mean():.4f} +/- {arr.std():.4f}")
        RESULTS[label]  = arr
        EXT_PRED[label] = (np.concatenate(pp), np.concatenate(tt))

    def eval_g_multi(make_model, label, n_seeds=N_BASELINE_SEEDS):
        """make_model: callable returning a fresh untrained model on device."""
        matrix = []
        rep_pp, rep_tt, rep_net = None, None, None
        short = label.split('.', 1)[0].strip()
        for ts in range(n_seeds):
            torch.manual_seed(ts); np.random.seed(ts)
            net = make_model()
            train_g(net, short, train_seed=ts)
            arr, pp, tt = _eval_g_collect(net)
            matrix.append(arr)
            if ts == 0:
                rep_pp, rep_tt, rep_net = pp, tt, net
            else:
                del net
                if torch.cuda.is_available(): torch.cuda.empty_cache()
        matrix = np.array(matrix)
        seed_means = matrix.mean(axis=1)
        MODEL_STATS[label]    = {'params': sum(p.numel() for p in rep_net.parameters() if p.requires_grad), 'kind': 'nn'}
        TRAINED_MODELS[label] = (rep_net, 'graph')
        RESULTS[label]        = matrix
        EXT_PRED[label]       = (np.concatenate(rep_pp), np.concatenate(rep_tt))
        print(f"{label:<52}  MAE: {matrix.mean():.4f} +/- {seed_means.std():.4f}  "
              f"(n={matrix.size}, between-seed std: {seed_means.std():.4f})")
        return rep_net

    class DiffGCN(nn.Module):
        def __init__(self,i,o,K=2):
            super().__init__(); self.K=K; self.l=nn.Linear((K+1)*i,o)
        def forward(self,x,A):
            out=[x]; Ax=x
            for _ in range(self.K): Ax=torch.matmul(A.unsqueeze(0),Ax); out.append(Ax)
            return self.l(torch.cat(out,-1))

    # 13. DCRNN — forward GRU + graph diffusion: strictly causal.
    class DCRNNLite(nn.Module):
        def __init__(self,F=4,H=64,**k):
            super().__init__(); self.H=H
            self.gr=DiffGCN(F+H,H); self.gu=DiffGCN(F+H,H); self.gc=DiffGCN(F+H,H)
            self.head=nn.Linear(H,1)
        def _step(self,x,h,A):
            xu=torch.cat([x,h],-1)
            r=torch.sigmoid(self.gr(xu,A)); u=torch.sigmoid(self.gu(xu,A))
            c=torch.tanh(self.gc(torch.cat([x,r*h],-1),A))
            return u*h+(1-u)*c
        def forward(self,feat,A):
            B,N,T,_=feat.shape; h=torch.zeros(B,N,self.H,device=feat.device); outs=[]
            for t in range(T): h=self._step(feat[:,:,t,:],h,A); outs.append(self.head(h))
            return torch.stack(outs,2).squeeze(-1)
    dcrnn = eval_g_multi(lambda: DCRNNLite(H=GH).to(device), "13. DCRNN (DiffGCN + GRU)")

    # 14. GWN — causal dilated convolutions: strictly left-to-right.
    class GWNLite(nn.Module):
        def __init__(self,F=4,H=64,num_nodes=325,**k):
            super().__init__()
            self.E1=nn.Parameter(torch.randn(num_nodes,10))
            self.E2=nn.Parameter(torch.randn(10,num_nodes))
            self.proj=nn.Linear(F,H)
            self.convs=nn.ModuleList([nn.Conv1d(H,H*2,2,dilation=2**i) for i in range(4)])
            self.gcn=DiffGCN(H,H,K=1); self.head=nn.Linear(H,1)
        def forward(self,feat,A_static):
            B,N,T,_=feat.shape
            Aa=torch.softmax(torch.relu(self.E1@self.E2),-1)
            Am=0.5*(A_static+Aa)
            h=self.proj(feat).permute(0,1,3,2).reshape(B*N,-1,T)
            skip=[]
            for cv in self.convs:
                d=cv.dilation[0]; p=(cv.kernel_size[0]-1)*d
                g=cv(F.pad(h,(p,0))); g1,g2=g.chunk(2,1)
                s=torch.tanh(g1)*torch.sigmoid(g2)
                h=h+s[:,:,:T]; skip.append(h)
            h=sum(skip).reshape(B,N,-1,T).permute(0,3,1,2).reshape(B*T,N,-1)
            h=self.gcn(h,Am).reshape(B,T,N,-1).permute(0,2,1,3)
            return self.head(h).squeeze(-1)
    gwn = eval_g_multi(lambda: GWNLite(H=GH,num_nodes=NUM_NODES).to(device), "14. GWN (Adaptive GCN + Gated TCN)")


    # ============================================================================
    #  Recent causal baselines: STID, DLinear, PatchTST, iTransformer.
    #  All four are FAITHFUL implementations and CAUSAL: a prediction at time t uses
    #  only observed values at times <= t, exactly like every other model here
    #  (verified by a future-perturbation test). They reuse the graph-model
    #  interface forward(feat, A) with feat=[B,N,T,4]=[xi,mask,sin,cos] -> [B,N,T],
    #  so train_g / eval_g give them the identical masks, HA-fill, de-norm and
    #  shared eval seeds as the other baselines.  The three originals are natively
    #  full-window (bidirectional); the documented causal adaptations are noted per
    #  class so they never read the future (no information advantage over MARST).
    # ============================================================================

    def _causal_avg(x, k):
        """[B,N,T] left-padded (causal) moving average, window k."""
        B, N, T = x.shape
        xp = F.pad(x, (k - 1, 0), mode='replicate')
        w = torch.ones(1, 1, k, device=x.device) / k
        return F.conv1d(xp.reshape(B * N, 1, T + k - 1), w).reshape(B, N, T)


    class STID(nn.Module):
        """STID — Spatial-Temporal Identity (Shao et al., CIKM 2022).
        Core idea kept verbatim: learnable node-identity + time-of-day-identity
        embeddings added to a simple per-step value/mask projection. The original's
        fixed-window input MLP is replaced by left-padded (causal) dilated convs so
        it never reads future steps; the time-of-day identity is a small MLP of the
        (sin,cos) clock features."""
        def __init__(self, num_nodes, d=64, layers=3, **kw):
            super().__init__()
            self.d = d
            self.in_proj = nn.Linear(2, d)                      # [xi, mask] per step
            self.tod = nn.Sequential(nn.Linear(2, d), nn.ReLU(), nn.Linear(d, d))
            self.node_emb = nn.Parameter(torch.randn(num_nodes, d) * 0.02)
            self.tconv = nn.ModuleList([nn.Conv1d(d, d, 3, dilation=2 ** i) for i in range(layers)])
            self.head = nn.Sequential(nn.ReLU(), nn.Linear(d, 1))

        def forward(self, feat, A=None):
            B, N, T, _ = feat.shape
            h = self.in_proj(feat[..., :2]) + self.node_emb[None, :, None, :] + self.tod(feat[..., 2:4])
            hc = h.permute(0, 1, 3, 2).reshape(B * N, self.d, T)
            for cv in self.tconv:
                d_ = cv.dilation[0]; p = (cv.kernel_size[0] - 1) * d_
                hc = hc + F.relu(cv(F.pad(hc, (p, 0))))          # left-pad -> causal
            h = hc.reshape(B, N, self.d, T).permute(0, 1, 3, 2)
            return self.head(h).squeeze(-1)


    class DLinear(nn.Module):
        """DLinear — decomposition + linear (Zeng et al., AAAI 2023).
        Series split into trend (causal moving average) + seasonal residual; each is
        passed through a learned CAUSAL FIR filter. This is the length-agnostic,
        causal form of DLinear's input->output Linear (the original fixed-window
        Linear would mix future steps and only fit one window length)."""
        def __init__(self, kernel=25, lookback=24, **kw):
            super().__init__()
            self.kernel = kernel; self.lb = lookback
            self.trend = nn.Conv1d(1, 1, lookback)
            self.seas = nn.Conv1d(1, 1, lookback)

        def forward(self, feat, A=None):
            B, N, T, _ = feat.shape
            s = feat[..., 0]
            trend = _causal_avg(s, self.kernel)
            seas = s - trend

            def fir(x, conv):
                xp = F.pad(x.reshape(B * N, 1, T), (self.lb - 1, 0))   # left-pad -> causal
                return conv(xp).reshape(B, N, T)
            return fir(trend, self.trend) + fir(seas, self.seas)


    class PatchTST(nn.Module):
        """PatchTST — patch + channel-independent Transformer (Nie et al., ICLR 2023).
        Per-sensor series is split into patches; a Transformer with a causal patch
        mask encodes them, and each patch is reconstructed from STRICTLY-earlier
        patches only (a learned start token seeds patch 0), so there is no
        intra-window future leak. Length-agnostic via padding to whole patches."""
        def __init__(self, patch=4, d=64, heads=4, layers=2, **kw):
            super().__init__()
            self.p = patch; self.d = d
            self.embed = nn.Linear(patch * 2, d)                 # patch of [xi, mask]
            self.pos = nn.Parameter(torch.randn(1, 256, d) * 0.02)
            self.start = nn.Parameter(torch.randn(1, 1, d) * 0.02)
            enc = nn.TransformerEncoderLayer(d, heads, d * 2, dropout=0.1,
                                             activation='gelu', batch_first=True, norm_first=True)
            self.tr = nn.TransformerEncoder(enc, layers)
            self.head = nn.Linear(d, patch)

        def forward(self, feat, A=None):
            B, N, T, _ = feat.shape; p = self.p
            pad = (p - T % p) % p
            xv = feat[..., 0]; mk = feat[..., 1]
            if pad:
                xv = F.pad(xv, (0, pad)); mk = F.pad(mk, (0, pad))
            Tp = T + pad; P = Tp // p
            tok = torch.stack([xv, mk], -1).reshape(B, N, P, p * 2)
            z = (self.embed(tok) + self.pos[:, :P].unsqueeze(1)).reshape(B * N, P, self.d)
            cm = torch.triu(torch.full((P, P), float('-inf'), device=feat.device), diagonal=1)  # causal
            out = self.tr(z, mask=cm)                            # out[i] saw patches <= i
            start = self.start.expand(B * N, 1, self.d)
            shifted = torch.cat([start, out[:, :-1]], dim=1)     # pos j sees <= j-1 -> strictly causal
            return self.head(shifted).reshape(B, N, Tp)[..., :T]


    class ITransformer(nn.Module):
        """iTransformer — inverted Transformer (Liu et al., ICLR 2024).
        Attention is taken ACROSS sensors (variates), not across time. The original
        embeds each sensor's whole series into one token (non-causal); here a causal
        temporal encoder (left-padded convs) yields a per-(sensor,time) state and the
        cross-sensor attention is applied independently at each timestep, so no future
        information is ever used."""
        def __init__(self, num_nodes, d=64, heads=4, layers=2, **kw):
            super().__init__()
            self.d = d
            self.in_proj = nn.Linear(2, d)
            self.tod = nn.Sequential(nn.Linear(2, d), nn.ReLU(), nn.Linear(d, d))
            self.tconv = nn.ModuleList([nn.Conv1d(d, d, 3, dilation=2 ** i) for i in range(2)])
            enc = nn.TransformerEncoderLayer(d, heads, d * 2, dropout=0.1,
                                             activation='gelu', batch_first=True, norm_first=True)
            self.attn = nn.TransformerEncoder(enc, layers)       # across sensors
            self.head = nn.Linear(d, 1)

        def forward(self, feat, A=None):
            B, N, T, _ = feat.shape
            h = self.in_proj(feat[..., :2]) + self.tod(feat[..., 2:4])
            hc = h.permute(0, 1, 3, 2).reshape(B * N, self.d, T)
            for cv in self.tconv:
                d_ = cv.dilation[0]; p = (cv.kernel_size[0] - 1) * d_
                hc = hc + F.relu(cv(F.pad(hc, (p, 0))))          # causal temporal encoder
            h = hc.reshape(B, N, self.d, T).permute(0, 3, 1, 2).reshape(B * T, N, self.d)  # [B,T,N,d]
            h = self.attn(h)                                     # cross-sensor, per timestep
            h = h.reshape(B, T, N, self.d).permute(0, 2, 1, 3)   # -> [B,N,T,d]
            return self.head(h).squeeze(-1)


    # Train + evaluate under the shared protocol (same train_g / eval_g as the graph baselines).
    stid = eval_g_multi(lambda: STID(NUM_NODES).to(device), "15. STID (node+ToD identity, causal)")
    dlin = eval_g_multi(lambda: DLinear().to(device), "16. DLinear (decomp + causal linear)")
    ptst = eval_g_multi(lambda: PatchTST().to(device), "17. PatchTST (causal patches)")
    itrf = eval_g_multi(lambda: ITransformer(NUM_NODES).to(device), "18. iTransformer (cross-sensor, causal)")


    # Shared multi-seed train/eval helpers (used by MARST and the anchor-mixture ablation).
    # Training (initialization) seeds -> error bars reflect training variance, not
    # just eval-mask variance. Eval masks stay shared across all models/seeds.
    TRAIN_SEEDS = [0, 1, 2]


    def _train_st(ctor, epochs, batch_size, train_seed, label, use_curriculum=True, gpu_id=0):
        """Train one spatiotemporal model on a single specified GPU.
        use_curriculum: ramp sparsity 60%->80% over the first 75% of epochs
        (was hard-coded 600 of 800). gpu_id selects the CUDA device; data
        tensors are mirrored onto that device. Net is returned on the
        default `device` so downstream eval (which uses module-scope
        speed_gpu/valid_gpu/ha_prior) keeps working unchanged."""
        dev = torch.device(f'cuda:{gpu_id}') if torch.cuda.is_available() else torch.device('cpu')
        if torch.cuda.is_available():
            torch.cuda.set_device(dev)
        # Per-GPU data mirrors (.to is a no-op when already on dev, so the default GPU
        # job never copies; off-device jobs copy once at start). All small (~25MB).
        sg, vg, hp = speed_gpu.to(dev), valid_gpu.to(dev), ha_prior.to(dev)
        nm_t, ns_t = node_means_t.to(dev), node_stds_t.to(dev)
        # Per-thread RNGs (avoids global np.random / torch.manual_seed races across threads).
        rng_np = np.random.default_rng(train_seed)
        g_cuda = torch.Generator(device=dev)
        g_cuda.manual_seed(int(train_seed) * 31 + int(gpu_id))
        torch.manual_seed(train_seed)   # for ctor init (minor race when concurrent, acceptable)
        net = ctor().to(dev)
        opt = torch.optim.Adam(net.parameters(), lr=1e-3)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
        # Scale curriculum endpoint with total epochs (was 600/800 = 75%).
        curr_end = max(1, int(epochs * 0.75))
        for ep in range(1, epochs + 1):
            net.train()
            if use_curriculum:
                sparsity_ep = min(SPARSITY, 0.60 + (SPARSITY - 0.60) * min(1.0, (ep - 1) / curr_end))
            else:
                sparsity_ep = SPARSITY
            t0_list = rng_np.integers(0, TRAIN_END - BATCH_TIME, batch_size)
            xs, has, ss, cs, ms, vs = [], [], [], [], [], []
            for t0 in t0_list:
                xs .append(sg[t0:t0+BATCH_TIME].T)
                has.append(hp[t0:t0+BATCH_TIME].T)
                vs .append(vg[t0:t0+BATCH_TIME].T)
                ti = torch.arange(int(t0), int(t0) + BATCH_TIME, device=dev)
                ss.append(torch.sin(2*np.pi*(ti%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,-1).expand(NUM_NODES,-1))
                cs.append(torch.cos(2*np.pi*(ti%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,-1).expand(NUM_NODES,-1))
                ms.append((torch.rand(NUM_NODES, BATCH_TIME, device=dev, generator=g_cuda) > sparsity_ep).float())
            xb, hb, sb, cb, mb, vb = map(torch.stack, (xs, has, ss, cs, ms, vs))
            m_eff = mb * vb
            p = net(xb * m_eff, m_eff, sb, cb, hb)
            lm = (mb == 0) & (vb > 0)
            if not lm.any():
                continue
            _per_elem = F.smooth_l1_loss(p[lm], xb[lm], beta=HUBER_BETA, reduction='none')
            _xb_raw   = xb * ns_t.view(1, -1, 1) + nm_t.view(1, -1, 1)
            if CFG['kind'] == 'speed':
                _jam = _xb_raw < REGIME_EDGES[0]
            else:
                _jam = _xb_raw > REGIME_EDGES[1]
            _w = 1.0 + JAM_WEIGHT * _jam[lm].float()
            loss = (_per_elem * _w).sum() / _w.sum().clamp(min=1e-6)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), 0.5)
            opt.step(); sch.step()
            if ep % 500 == 0 or ep == epochs:
                msg = f"  [{label} seed {train_seed} gpu {gpu_id}] ep {ep:4d}/{epochs} | loss {loss.item():.4f}"
                if getattr(net, 'last_pi', None) is not None:
                    pi = net.last_pi.mean(dim=0)
                    msg += f" | pi[LOCF={pi[0]:.2f} HA={pi[1]:.2f} KNN={pi[2]:.2f}]"
                print(msg, flush=True)
        # IMPORTANT: pin move-back to cuda:0 explicitly. Using the module-scope
        # `device = torch.device("cuda")` resolves to the THREAD-LOCAL current
        # device, which is cuda:gpu_id in this worker -- that would leave seed-1
        # nets on cuda:1 and break every downstream eval that slices from
        # speed_gpu / valid_gpu / ha_prior (all on cuda:0).
        if torch.cuda.is_available():
            net = net.to(torch.device("cuda:0"))
            torch.cuda.set_device(torch.device("cuda:0"))
        return net

    def _train_parallel(jobs):
        """Dispatch a list of training jobs across N_GPUS workers.
        Each job is a dict with keys: ctor, epochs, batch_size, seed, label,
        and optionally use_curr (default True). Returns trained nets in the
        same order, all moved back to the default `device`."""
        if N_GPUS <= 1:
            return [_train_st(j['ctor'], j['epochs'], j['batch_size'], j['seed'],
                              j['label'], use_curriculum=j.get('use_curr', True),
                              gpu_id=0) for j in jobs]
        with ThreadPoolExecutor(max_workers=N_GPUS) as ex:
            futs = []
            for i, j in enumerate(jobs):
                gpu = i % N_GPUS
                futs.append(ex.submit(
                    _train_st, j['ctor'], j['epochs'], j['batch_size'], j['seed'],
                    j['label'], use_curriculum=j.get('use_curr', True), gpu_id=gpu))
            return [f.result() for f in futs]


    def _eval_st_mae(net):
        """Per-eval-mask MAE (native unit) over the shared EVAL_SEEDS masks for one trained model."""
        net.eval()
        ES, EL = EVAL_START, EVAL_LEN
        x_ev  = speed_gpu[ES:ES+EL].T.unsqueeze(0)
        ha_ev = ha_prior [ES:ES+EL].T.unsqueeze(0)
        v_ev  = valid_gpu[ES:ES+EL].T.unsqueeze(0)
        ti = torch.arange(ES, ES+EL, device=device)
        ts = torch.sin(2*np.pi*(ti%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,1,-1).expand(1,NUM_NODES,-1)
        tc = torch.cos(2*np.pi*(ti%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,1,-1).expand(1,NUM_NODES,-1)
        st = torch.tensor(node_stds,  device=device).view(1,-1,1)
        mn = torch.tensor(node_means, device=device).view(1,-1,1)
        out = []
        with torch.no_grad():
            for seed in EVAL_SEEDS:
                m_ev = torch.tensor(make_eval_mask_np(seed, EL, NUM_NODES), device=device).T.unsqueeze(0)
                m_eff = m_ev * v_ev
                preds = []
                for c0 in range(0, EL, BATCH_TIME):
                    c1 = min(c0 + BATCH_TIME, EL)
                    preds.append(net(x_ev[:,:,c0:c1]*m_eff[:,:,c0:c1], m_eff[:,:,c0:c1],
                                     ts[:,:,c0:c1], tc[:,:,c0:c1], ha_ev[:,:,c0:c1]))
                p  = torch.cat(preds, dim=2)
                pk = (p    * st + mn).clamp(CLAMP_LO, CLAMP_HI)
                tk = (x_ev * st + mn).clamp(CLAMP_LO, CLAMP_HI)
                sm = (m_ev == 0) & (v_ev > 0)
                out.append(torch.abs(pk[sm] - tk[sm]).mean().item())
        return np.array(out)





    # MARST hyperparameters.
    # Reuses _train_st / _eval_st_mae and TRAIN_SEEDS defined in the helpers cell above.
    HIDDEN_DIM_MARST   = 128
    N_LAYERS_MARST     = 6
    TRAIN_EPOCHS_MARST = 800
    BATCH_SIZE_MARST   = 4

    ctor_marst = lambda: MaskedSTTransformerMARST(
        NUM_NODES, A_t, node_means_t, node_stds_t,
        hidden=HIDDEN_DIM_MARST, n_heads=N_HEADS, n_layers=N_LAYERS_MARST, dropout=DROPOUT).to(device)

    _n_params_marst = sum(p.numel() for p in ctor_marst().parameters() if p.requires_grad)
    MODEL_STATS["19. MARST (ours, multi-anchor)"] = {"params": int(_n_params_marst), "kind": "nn"}
    print(f"Training MARST x{len(TRAIN_SEEDS)} seeds [{DATASET_NAME}] | params: {_n_params_marst/1e6:.2f}M | "
          f"hidden={HIDDEN_DIM_MARST} layers={N_LAYERS_MARST} epochs={TRAIN_EPOCHS_MARST} | "
          f"anchors=3 (LOCF/HA/KNN) softmax-mixed")

    marst_mask_mat = []       # [n_train_seeds, n_eval_masks] MAE
    _marst_jobs = [dict(ctor=ctor_marst, epochs=TRAIN_EPOCHS_MARST, batch_size=BATCH_SIZE_MARST,
                        seed=tseed, label=f"MARST(seed={tseed})") for tseed in TRAIN_SEEDS]
    print(f"Training MARST {len(_marst_jobs)} seed(s) across {N_GPUS} GPU(s) at {TRAIN_EPOCHS_MARST} epochs each...", flush=True)
    _marst_nets = _train_parallel(_marst_jobs)
    for si, (tseed, net) in enumerate(zip(TRAIN_SEEDS, _marst_nets)):
        marst_mask_mat.append(_eval_st_mae(net))
        if si == 0:
            net_marst = net       # representative run for extended metrics / figures
            TRAINED_MODELS["19. MARST (ours, multi-anchor)"] = (net_marst, 'st')
        else:
            del net
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    marst_mask_mat = np.array(marst_mask_mat)
    print("MARST per-seed MAE (mean over eval masks):", np.round(marst_mask_mat.mean(1), 4))


    # MARST result
    mae_marst_seeds = marst_mask_mat.mean(axis=1)      # [n_train_seeds]
    print(f"MARST MAE: {mae_marst_seeds.mean():.4f} +/- {mae_marst_seeds.std():.4f} {VALUE_UNIT}  "
          f"(over {len(TRAIN_SEEDS)} training seeds x {len(EVAL_SEEDS)} eval masks)")
    print("  per training seed:", {s: round(float(v), 4) for s, v in zip(TRAIN_SEEDS, mae_marst_seeds)})



    # Final comparison table — store full [n_train_seeds, n_eval_masks] matrix for parity with baselines
    RESULTS["19. MARST (ours, multi-anchor)"]   = marst_mask_mat

    ha_mae = RESULTS["1. Historical Average (HA)"].mean()
    OURS = {"19. MARST (ours, multi-anchor)"}
    print()
    print("="*72)
    print(f"{f'{DATASET_NAME} IMPUTATION BENCHMARK  ({int(SPARSITY*100)}% Sparsity, {len(EVAL_SEEDS)} seeds)':^72}")
    print("="*72)
    print(f"{'Model':<52} {'MAE':>8}  {'Std':>6}  {'vs HA':>7}")
    print("-"*72)
    for name, arr in sorted(RESULTS.items(), key=lambda kv: kv[1].mean()):
        tag = " < OURS" if name in OURS else ""
        diff = arr.mean() - ha_mae
        print(f"{name:<52} {arr.mean():>8.4f}  {arr.std():>6.4f}  {diff:>+7.4f}{tag}")
    print("-"*72)
    d = ha_mae - RESULTS["19. MARST (ours, multi-anchor)"].mean()
    print(f"Ours (MARST) vs HA: {d:+.4f} {VALUE_UNIT}  ({100*d/ha_mae:+.1f}%)")
    print(f"std: between-seed for nn baselines + MARST (n={N_BASELINE_SEEDS}); eval-mask for classical (n={len(EVAL_SEEDS)})")
    print("="*72)


    # ── Extended Evaluation Metrics — ALL models ────────────────────────────────
    # RMSE / MAPE / R² / Pearson / MBE / MedAE / Hit@tol for every model. Baselines'
    # predictions were stashed into EXT_PRED during their eval (eval_fn / eval_nw /
    # eval_g); here we add the representative MARST run, then score everyone
    # on the same held-out positions (all EVAL_SEEDS).
    import numpy as np, torch, re

    def _run_model_ext_full(net):
        """Flattened (pred_kmh, true_kmh) over all EVAL_SEEDS held-out positions."""
        net.eval()
        ES, EL = EVAL_START, EVAL_LEN
        x_ev  = speed_gpu[ES:ES+EL].T.unsqueeze(0)
        ha_ev = ha_prior [ES:ES+EL].T.unsqueeze(0)
        v_ev  = valid_gpu[ES:ES+EL].T.unsqueeze(0)
        ti = torch.arange(ES, ES+EL, device=device)
        ts = torch.sin(2*np.pi*(ti%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,1,-1).expand(1,NUM_NODES,-1)
        tc = torch.cos(2*np.pi*(ti%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,1,-1).expand(1,NUM_NODES,-1)
        st = torch.tensor(node_stds,  device=device).view(1,-1,1)
        mn = torch.tensor(node_means, device=device).view(1,-1,1)
        pp, tt = [], []
        with torch.no_grad():
            for seed in EVAL_SEEDS:
                m_ev = torch.tensor(make_eval_mask_np(seed, EL, NUM_NODES), device=device).T.unsqueeze(0)
                me   = m_ev * v_ev
                preds = []
                for c0 in range(0, EL, BATCH_TIME):
                    c1 = min(c0 + BATCH_TIME, EL)
                    preds.append(net(x_ev[:,:,c0:c1]*me[:,:,c0:c1], me[:,:,c0:c1],
                                     ts[:,:,c0:c1], tc[:,:,c0:c1], ha_ev[:,:,c0:c1]))
                p  = torch.cat(preds, dim=2)
                pk = (p    * st + mn).clamp(CLAMP_LO, CLAMP_HI)
                tk = (x_ev * st + mn).clamp(CLAMP_LO, CLAMP_HI)
                sm = (m_ev == 0) & (v_ev > 0)
                pp.append(pk[sm].cpu().numpy()); tt.append(tk[sm].cpu().numpy())
        return np.concatenate(pp), np.concatenate(tt)

    # Add the representative MARST run (baselines already in EXT_PRED)
    EXT_PRED["19. MARST (ours, multi-anchor)"]   = _run_model_ext_full(net_marst)

    def extended_metrics(pred, true):
        err   = pred - true
        mae   = np.abs(err).mean()
        rmse  = np.sqrt((err**2).mean())
        mape  = (np.abs(err) / np.maximum(true, 1.0)).mean() * 100
        ss_r  = (err**2).sum();  ss_t = ((true - true.mean())**2).sum()
        r2    = 1 - ss_r / ss_t
        mbe   = err.mean()
        medae = np.median(np.abs(err))
        pear  = float(np.corrcoef(pred, true)[0, 1])
        hit5  = (np.abs(err) < HIT_TOL[0]).mean() * 100
        hit10 = (np.abs(err) < HIT_TOL[1]).mean() * 100
        # Jam / congestion-regime MAE: error on the most-congested held-out points.
        #   speed datasets -> jam = slowest 20%   (true < p20  regime edge)
        #   flow  datasets -> jam = busiest 33%   (true > p67  regime edge)
        if CFG['kind'] == 'speed':
            jam = true < REGIME_EDGES[0]
        else:
            jam = true > REGIME_EDGES[1]
        jam_mae = float(np.abs(err[jam]).mean()) if bool(jam.any()) else float('nan')
        return dict(MAE=mae, RMSE=rmse, MedAE=medae, MAPE=mape,
                    R2=r2, Pearson=pear, MBE=mbe, Hit5=hit5, Hit10=hit10,
                    JamMAE=jam_mae)

    def _lead_num(k):
        m = re.match(r'\s*(\d+)', k); return int(m.group(1)) if m else 999

    # Score every model, ordered by its benchmark number
    EXT_METRICS = {k: extended_metrics(*EXT_PRED[k]) for k in sorted(EXT_PRED, key=_lead_num)}
    _best_name      = min(EXT_METRICS, key=lambda k: EXT_METRICS[k]["MAE"])
    _best_jam_name  = min((k for k in EXT_METRICS
                           if EXT_METRICS[k]["JamMAE"] == EXT_METRICS[k]["JamMAE"]),  # filter NaN
                          key=lambda k: EXT_METRICS[k]["JamMAE"], default=None)

    cols_ext = ["MAE", "JamMAE", "RMSE", "MedAE", "MAPE%", "R2", "Pearson", "MBE",
                f"Hit@{HIT_TOL[0]:g}", f"Hit@{HIT_TOL[1]:g}"]
    mw = 34; w = 11
    W = mw + w * len(cols_ext)
    print("=" * W)
    print(f"{'EXTENDED EVALUATION METRICS  --  ' + DATASET_NAME + '  (80% sparsity, ' + str(len(EVAL_SEEDS)) + ' eval masks)':^{W}}")
    print("=" * W)
    print(f"{'Model':<{mw}}" + "".join(f"{c:>{w}}" for c in cols_ext))
    print("-" * W)
    for name, m in EXT_METRICS.items():
        label = name.split(". ", 1)[-1][:mw-2]
        row = (f"{label:<{mw}}"
               f"{m['MAE']:>{w}.4f}{m['JamMAE']:>{w}.4f}{m['RMSE']:>{w}.4f}{m['MedAE']:>{w}.4f}"
               f"{m['MAPE']:>{w}.2f}{m['R2']:>{w}.4f}{m['Pearson']:>{w}.4f}"
               f"{m['MBE']:>{w}.4f}{m['Hit5']:>{w}.2f}{m['Hit10']:>{w}.2f}")
        print(row + ("  <-- BEST MAE" if name == _best_name else ""))
    print("=" * W)
    if _best_jam_name is not None and _best_jam_name != _best_name:
        _mae_lead = EXT_METRICS[_best_name]["JamMAE"]
        _jam_lead = EXT_METRICS[_best_jam_name]["JamMAE"]
        _gap      = _mae_lead - _jam_lead
        print(f">> DISCLOSE: MAE leader is '{_best_name.split('. ',1)[-1]}' "
              f"but JamMAE leader is '{_best_jam_name.split('. ',1)[-1]}' "
              f"(JamMAE {_jam_lead:.3f} vs {_mae_lead:.3f}, gap +{_gap:.3f}). "
              f"Report this in the main text.")
    elif _best_jam_name is not None:
        print(f">> JamMAE leader matches MAE leader: '{_best_name.split('. ',1)[-1]}'.")


    # ============================================================================
    #  LEAK AUDIT — empirical proof that NO held-out value or offline sensor ever
    #  reaches any model (MARST included), and that the ST model is causal.
    #  Method: (1) p0 = predictions on the standard masked eval input.
    #          (2) Corrupt the GROUND TRUTH at every held-out (blind) position by a
    #              large random amount, rebuild each model's input, re-predict -> p1.
    #  A leak-free model never sees held-out data, so p0 == p1 at the held positions.
    #  Any nonzero difference would expose a leak. We also perturb the FUTURE and
    #  confirm MARST's past predictions are unchanged (temporal causality).
    # ============================================================================
    import numpy as np, torch

    _ES, _EL = EVAL_START, EVAL_LEN
    assert _ES >= TRAIN_END, "eval window overlaps training window — temporal leak!"
    print(f"train/eval split: TRAIN_END={TRAIN_END}, EVAL_START={_ES} (gap {_ES-TRAIN_END} steps) -> OK")

    _seed = EVAL_SEEDS[0]
    _m  = torch.tensor(make_eval_mask_np(_seed, _EL, NUM_NODES), device=device).T.unsqueeze(0)
    _v  = valid_gpu[_ES:_ES+_EL].T.unsqueeze(0)
    _me = _m * _v
    _held = (_m == 0) & (_v > 0)
    _x  = speed_gpu[_ES:_ES+_EL].T.unsqueeze(0)
    _ha = ha_prior [_ES:_ES+_EL].T.unsqueeze(0)
    _ti = torch.arange(_ES, _ES+_EL, device=device)
    _sin = torch.sin(2*np.pi*(_ti%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,1,-1).expand(1,NUM_NODES,-1)
    _cos = torch.cos(2*np.pi*(_ti%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,1,-1).expand(1,NUM_NODES,-1)

    torch.manual_seed(0)
    _xc = _x.clone()
    _xc[_held] = _xc[_held] + 50.0 * torch.randn_like(_xc[_held])   # corrupt held-out truth

    def _pred_st_leak(net, xin):
        net.eval()
        with torch.no_grad():
            outs = [net(xin[:,:,c0:min(c0+BATCH_TIME,_EL)]*_me[:,:,c0:min(c0+BATCH_TIME,_EL)],
                        _me[:,:,c0:min(c0+BATCH_TIME,_EL)], _sin[:,:,c0:min(c0+BATCH_TIME,_EL)],
                        _cos[:,:,c0:min(c0+BATCH_TIME,_EL)], _ha[:,:,c0:min(c0+BATCH_TIME,_EL)])
                    for c0 in range(0, _EL, BATCH_TIME)]
        return torch.cat(outs, 2)

    def _pred_graph_leak(net, xin):
        net.eval()
        xi = xin*_me + _ha*(1-_me)
        feat = torch.stack([xi[0], _me[0], _sin[0], _cos[0]], -1).unsqueeze(0)
        with torch.no_grad():
            return net(feat, A_t)

    def _pred_nw_leak(net, xin):
        # eval_nw input shape: [N, T, F=4] = [xi, mask, sin, cos]
        net.eval()
        feat = torch.stack([(xin[0] * _me[0]), _me[0], _sin[0], _cos[0]], dim=-1)
        with torch.no_grad():
            return net(feat).squeeze(-1).unsqueeze(0)   # [1, N, T]

    _st    = [('MARST (ours)', globals().get('net_marst'))]
    _graph = [(n, globals().get(v)) for n, v in
              [('DCRNN','dcrnn'), ('GWN','gwn'),
               ('STID','stid'), ('DLinear','dlin'), ('PatchTST','ptst'), ('iTransformer','itrf')]]
    # Non-graph baselines pulled from TRAINED_MODELS (they aren't saved as globals)
    _nw = [(lbl.split('. ', 1)[-1].split(' (', 1)[0], m)
           for lbl, (m, kind) in TRAINED_MODELS.items() if kind == 'nw']

    print("\n" + "="*66)
    print("LEAK AUDIT - corrupt held-out truth; leak-free => predictions unchanged")
    print("="*66)
    _ok = True
    for _name, _net in _st:
        if _net is None: continue
        d = (_pred_st_leak(_net, _x)[_held] - _pred_st_leak(_net, _xc)[_held]).abs().max().item()
        _ok &= d < 1e-4
        print(f"  {_name:<14} held-out-truth leak: max|dpred|={d:.2e}  -> {'NO LEAK' if d<1e-4 else 'LEAK!!'}")
    for _name, _net in _graph:
        if _net is None: continue
        d = (_pred_graph_leak(_net, _x)[_held] - _pred_graph_leak(_net, _xc)[_held]).abs().max().item()
        _ok &= d < 1e-4
        print(f"  {_name:<14} held-out-truth leak: max|dpred|={d:.2e}  -> {'NO LEAK' if d<1e-4 else 'LEAK!!'}")
    for _name, _net in _nw:
        if _net is None: continue
        d = (_pred_nw_leak(_net, _x)[_held] - _pred_nw_leak(_net, _xc)[_held]).abs().max().item()
        _ok &= d < 1e-4
        print(f"  {_name:<14} held-out-truth leak: max|dpred|={d:.2e}  -> {'NO LEAK' if d<1e-4 else 'LEAK!!'}")

    # Temporal causality: perturb inputs in the second half; MARST's first-half output must not move.
    if globals().get('net_marst') is not None:
        _k = _EL // 2
        _xf = _x.clone(); _xf[:,:,_k:] = _xf[:,:,_k:] + 50.0*torch.randn_like(_xf[:,:,_k:])
        _dc = (_pred_st_leak(net_marst, _x)[:,:,:_k] - _pred_st_leak(net_marst, _xf)[:,:,:_k]).abs().max().item()
        _ok &= _dc < 1e-4
        print(f"  {'MARST (causal)':<14} future-perturb leak: max|dpast|={_dc:.2e}  -> {'CAUSAL' if _dc<1e-4 else 'LEAK!!'}")

    print("="*66)
    print("AUDIT PASSED: no model reads held-out data; ST model is strictly causal."
          if _ok else "AUDIT FAILED - investigate the flagged model(s).")
    assert _ok, "Leak audit failed"


    # Sparsity-sensitivity sweep on the already-trained MARST
    SPARSITY_LEVELS = [0.50, 0.80, 0.95]   # shrunk from [0.50,0.60,0.70,0.80,0.90,0.95] -- endpoints + main eval point; covers low/main/stress

    def _eval_at_sparsity(model, sparsity, kind='model'):
        x_ev  = speed_gpu[EVAL_START:EVAL_START+EVAL_LEN].T.unsqueeze(0)
        ha_ev = ha_prior [EVAL_START:EVAL_START+EVAL_LEN].T.unsqueeze(0)
        v_ev  = valid_gpu[EVAL_START:EVAL_START+EVAL_LEN].T.unsqueeze(0)
        ti = torch.arange(EVAL_START, EVAL_START+EVAL_LEN, device=device)
        ts = torch.sin(2*np.pi*(ti%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,1,-1).expand(1,NUM_NODES,-1)
        tc = torch.cos(2*np.pi*(ti%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,1,-1).expand(1,NUM_NODES,-1)
        st = torch.tensor(node_stds,  device=device).view(1,-1,1)
        mn = torch.tensor(node_means, device=device).view(1,-1,1)
        maes = []
        with torch.no_grad():
            for seed in EVAL_SEEDS:
                m_ev = torch.tensor(make_eval_mask_np(seed, EVAL_LEN, NUM_NODES, sparsity=sparsity),
                                    device=device).T.unsqueeze(0)
                me = m_ev * v_ev
                if kind == 'model':
                    preds = []
                    for c0 in range(0, EVAL_LEN, BATCH_TIME):
                        c1 = min(c0 + BATCH_TIME, EVAL_LEN)
                        preds.append(model(x_ev[:,:,c0:c1]*me[:,:,c0:c1], me[:,:,c0:c1],
                                           ts[:,:,c0:c1], tc[:,:,c0:c1], ha_ev[:,:,c0:c1]))
                    p = torch.cat(preds, dim=2)
                elif kind == 'ha':
                    p = ha_ev
                elif kind == 'locf':
                    prev = ha_ev[:, :, 0:1].clone(); out = []
                    for t in range(EVAL_LEN):
                        obs = x_ev[:, :, t:t+1] * me[:, :, t:t+1]
                        prev = torch.where(me[:, :, t:t+1] > 0, obs, prev)
                        out.append(prev.clone())
                    p = torch.cat(out, dim=2)
                pk = (p * st + mn).clamp(CLAMP_LO, CLAMP_HI)
                tk = (x_ev * st + mn).clamp(CLAMP_LO, CLAMP_HI)
                sm = (m_ev == 0) & (v_ev > 0)
                e = torch.abs(pk[sm] - tk[sm])
                e = e[torch.isfinite(e)]  # drop rare all-blind-timestep NaNs (extreme sparsity)
                maes.append(e.mean().item())
        return np.array(maes)

    net_marst.eval()
    SPARSITY_SWEEP = {'MARST (ours)': {}, 'LOCF': {}, 'Hist. Average': {}}
    for sp in SPARSITY_LEVELS:
        SPARSITY_SWEEP['MARST (ours)'][sp]  = _eval_at_sparsity(net_marst, sp, 'model')
        SPARSITY_SWEEP['LOCF'][sp]          = _eval_at_sparsity(None,   sp, 'locf')
        SPARSITY_SWEEP['Hist. Average'][sp] = _eval_at_sparsity(None,   sp, 'ha')
        print(f"sparsity {int(sp*100):2d}%  MARST={SPARSITY_SWEEP['MARST (ours)'][sp].mean():.4f}  "
              f"LOCF={SPARSITY_SWEEP['LOCF'][sp].mean():.4f}  "
              f"HA={SPARSITY_SWEEP['Hist. Average'][sp].mean():.4f}")

    title = f"SPARSITY SENSITIVITY  ({DATASET_NAME})"
    keys = list(SPARSITY_SWEEP)
    W = 11 + 16 * len(keys)
    print("\n" + "=" * W)
    print(f"{title:^{W}}")
    print("=" * W)
    print(f"{'Sparsity':>10}" + "".join(f"{k:>16}" for k in keys))
    print("-" * W)
    for sp in SPARSITY_LEVELS:
        print(f"{int(sp*100):>9}%" + "".join(f"{SPARSITY_SWEEP[k][sp].mean():>16.4f}" for k in keys))
    print("=" * W)

    SPARSITY_SAVE = {k: {str(sp): [float(x) for x in v] for sp, v in d.items()}
                     for k, d in SPARSITY_SWEEP.items()}


    # ============================================================================
    #  MARST anchor-mixture ablation. Trains one model per variant (seed 0,
    #  ANCHOR_ABLATION_EPOCHS) and evaluates over the shared EVAL_SEEDS masks.
    #  Reuses _train_st / _eval_st_mae (same forward signature as MARST).
    # ============================================================================
    import matplotlib.pyplot as plt

    ANCHOR_ABLATION_EPOCHS = ABLATION_EPOCHS if 'ABLATION_EPOCHS' in globals() else 800

    MARST_ABLATIONS = {
        'Full MARST (LOCF+HA+KNN)': dict(),
        '- LOCF anchor':            dict(use_locf=False),
        '- HA anchor':              dict(use_ha=False),
        '- KNN anchor':             dict(use_knn=False),
        'LOCF only':                dict(use_ha=False,  use_knn=False),
        'HA only':                  dict(use_locf=False, use_knn=False),
        'KNN only':                 dict(use_locf=False, use_ha=False),
    }

    MARST_ANCHOR_ABLATION = {}
    _ab_items = list(MARST_ABLATIONS.items())
    _ab_jobs = []
    for _name, _flags in _ab_items:
        _ctor = (lambda f=_flags: MaskedSTTransformerMARSTAblate(
            NUM_NODES, A_t, node_means_t, node_stds_t,
            hidden=HIDDEN_DIM_MARST, n_heads=N_HEADS, n_layers=N_LAYERS_MARST,
            dropout=DROPOUT, **f).to(device))
        _ab_jobs.append(dict(ctor=_ctor, epochs=ANCHOR_ABLATION_EPOCHS, batch_size=BATCH_SIZE_MARST,
                             seed=0, label=_name))
    print(f"Training {len(_ab_jobs)} MARST ablations across {N_GPUS} GPU(s) at {ANCHOR_ABLATION_EPOCHS} epochs each...", flush=True)
    _ab_nets = _train_parallel(_ab_jobs)
    for (_name, _flags), _net_ab in zip(_ab_items, _ab_nets):
        MARST_ANCHOR_ABLATION[_name] = _eval_st_mae(_net_ab)
        del _net_ab
        print(f"  {_name:<26} MAE: {MARST_ANCHOR_ABLATION[_name].mean():.4f} "
              f"+/- {MARST_ANCHOR_ABLATION[_name].std():.4f}")
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    _full = MARST_ANCHOR_ABLATION['Full MARST (LOCF+HA+KNN)'].mean()
    _title = f"MARST ANCHOR ABLATION  ({DATASET_NAME}, {int(SPARSITY*100)}% sparsity, {len(EVAL_SEEDS)} seeds)"
    print("\n" + "=" * 70)
    print(f"{_title:^70}")
    print("=" * 70)
    print(f"{'Variant':<28}{'MAE':>9}{'Std':>9}{'dMAE':>9}{'%':>9}")
    print("-" * 70)
    for _name, _arr in MARST_ANCHOR_ABLATION.items():
        _d = _arr.mean() - _full
        print(f"{_name:<28}{_arr.mean():>9.4f}{_arr.std():>9.4f}{_d:>+9.4f}{100*_d/_full:>+8.1f}%")
    print("=" * 70)
    print("dMAE > 0 => removing / restricting the anchor HURTS (it was contributing).")

    MARST_ANCHOR_ABLATION_SAVE = {k: [float(x) for x in v] for k, v in MARST_ANCHOR_ABLATION.items()}

    _names  = list(MARST_ANCHOR_ABLATION.keys())
    _means  = [MARST_ANCHOR_ABLATION[n].mean() for n in _names]
    _stds   = [MARST_ANCHOR_ABLATION[n].std()  for n in _names]
    _colors = ['#B71C1C' if n.startswith('Full') else '#42A5F5' for n in _names]
    fig, ax = plt.subplots(figsize=(8, 4.4))
    ax.barh(range(len(_names)), _means, xerr=_stds, color=_colors, alpha=.88, capsize=3)
    ax.set_yticks(range(len(_names))); ax.set_yticklabels(_names)
    ax.axvline(_full, color='#B71C1C', ls='--', lw=1, alpha=.6, label='Full MARST')
    ax.invert_yaxis(); ax.set_xlabel(f'MAE ({VALUE_UNIT})')
    ax.set_title('MARST anchor ablation (higher = anchor matters)', fontweight='bold'); ax.legend()
    plt.tight_layout(); plt.savefig(f'fig_marst_anchor_ablation_{DATASET_NAME}.png', dpi=140, bbox_inches='tight'); plt.show()
    print(f"Saved fig_marst_anchor_ablation_{DATASET_NAME}.png")


    # ============================================================================
    #  Q2 — Soft-LOCF EMA decay sensitivity. Trains MARST (seed 0 only) at
    #  decay ∈ {0.80, 0.90, 0.99}; reuses the main run for decay=0.95.
    # ============================================================================
    SOFT_LOCF_DECAY_SWEEP = {0.95: marst_mask_mat[0].copy()}   # seed-0 row from main MARST
    _decays = [0.80, 0.90, 0.99]
    _decay_jobs = []
    for _decay in _decays:
        _ctor_d = (lambda d=_decay: MaskedSTTransformerMARST(
            NUM_NODES, A_t, node_means_t, node_stds_t,
            hidden=HIDDEN_DIM_MARST, n_heads=N_HEADS, n_layers=N_LAYERS_MARST,
            dropout=DROPOUT, soft_locf_decay=d).to(device))
        _decay_jobs.append(dict(ctor=_ctor_d, epochs=ANCHOR_ABLATION_EPOCHS, batch_size=BATCH_SIZE_MARST,
                                seed=0, label=f"MARST(d={_decay})"))
    print(f"Q2: training {len(_decay_jobs)} decay variants across {N_GPUS} GPU(s) at {ANCHOR_ABLATION_EPOCHS} epochs each...", flush=True)
    _decay_nets = _train_parallel(_decay_jobs)
    for _decay, _net_d in zip(_decays, _decay_nets):
        SOFT_LOCF_DECAY_SWEEP[_decay] = _eval_st_mae(_net_d)
        del _net_d
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    print("Q2 Soft-LOCF decay sweep: " + "  ".join(
        f"d={d}->MAE={SOFT_LOCF_DECAY_SWEEP[d].mean():.4f}" for d in sorted(SOFT_LOCF_DECAY_SWEEP)))

    # ============================================================================
    #  Q6 — Curriculum vs fixed-sparsity training. Trains MARST (seed 0) at fixed
    #  SPARSITY from epoch 0; reuses the main run for the curriculum variant.
    # ============================================================================
    CURRICULUM_ABLATION = {"curriculum (60% -> 80%)": marst_mask_mat[0].copy()}
    print(f"Q6: training MARST with fixed sparsity (no curriculum) ...", flush=True)
    _net_fix = _train_st(ctor_marst, ANCHOR_ABLATION_EPOCHS, BATCH_SIZE_MARST, 0, "MARST-fixed", use_curriculum=False)
    CURRICULUM_ABLATION["fixed " + str(int(SPARSITY*100)) + "%"] = _eval_st_mae(_net_fix)
    del _net_fix
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    print("Q6 Curriculum vs fixed: " + "  ".join(
        f"{k}->MAE={v.mean():.4f}" for k, v in CURRICULUM_ABLATION.items()))


    # ============================================================================
    #  Missing-pattern robustness (point / block / sensor). No retraining: reuses
    #  the trained MARST and graph nets, plus classical references, at SPARSITY.
    # ============================================================================
    import numpy as np, torch
    import matplotlib.pyplot as plt

    MISSING_PATTERNS = ['point', 'block', 'sensor']
    BLOCK_LEN = 8   # consecutive blind steps per temporal gap (~40 min @ 5-min sampling)

    def make_pattern_mask_np(seed, EL, N, sparsity, pattern='point', block_len=BLOCK_LEN):
        """[EL,N] float mask: 1=observed, 0=blind, ~`sparsity` fraction blind.
        Guarantees >=1 observed sensor per timestep (avoids all-blind-step NaN)."""
        rng = np.random.default_rng(seed)
        if pattern == 'point':
            mask = (rng.random((EL, N)) > sparsity).astype(np.float32)
        elif pattern == 'sensor':
            mask = np.ones((EL, N), dtype=np.float32)
            mask[:, rng.random(N) < sparsity] = 0.0
        elif pattern == 'block':
            mask = np.ones((EL, N), dtype=np.float32)
            n_blk = max(1, int(round(sparsity * EL / block_len)))
            for n in range(N):
                for _ in range(n_blk):
                    s0 = int(rng.integers(0, max(1, EL - block_len)))
                    mask[s0:s0 + block_len, n] = 0.0
        else:
            raise ValueError(f"unknown pattern {pattern!r}")
        empty = mask.sum(axis=1) == 0
        if empty.any():
            rows = np.where(empty)[0]
            mask[rows, rng.integers(0, N, size=rows.size)] = 1.0
        return mask

    # Shared eval-window arrays
    _ES, _EL = EVAL_START, EVAL_LEN
    _xn  = speed_norm[_ES:_ES+_EL]                 # [EL,N] z-scored
    _vv  = valid_raw [_ES:_ES+_EL]                 # [EL,N] validity
    _idx = np.arange(_ES, _ES+_EL)
    _xt  = speed_gpu[_ES:_ES+_EL].T.unsqueeze(0)   # [1,N,EL]
    _hat = ha_prior [_ES:_ES+_EL].T.unsqueeze(0)
    _vt  = valid_gpu[_ES:_ES+_EL].T.unsqueeze(0)
    _ti  = torch.arange(_ES, _ES+_EL, device=device)
    _ts  = torch.sin(2*np.pi*(_ti%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,1,-1).expand(1,NUM_NODES,-1)
    _tc  = torch.cos(2*np.pi*(_ti%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,1,-1).expand(1,NUM_NODES,-1)
    _stt = torch.tensor(node_stds,  device=device).view(1,-1,1)
    _mnt = torch.tensor(node_means, device=device).view(1,-1,1)
    _tk_full = np.clip(_xn.T*node_stds[:,None]+node_means[:,None], CLAMP_LO, CLAMP_HI)  # [N,EL]

    def _pred_st(net, m_TN):
        net.eval()
        m_ev = torch.tensor(m_TN, device=device).T.unsqueeze(0); me = m_ev*_vt
        with torch.no_grad():
            preds = []
            for c0 in range(0, _EL, BATCH_TIME):
                c1 = min(c0+BATCH_TIME, _EL)
                preds.append(net(_xt[:,:,c0:c1]*me[:,:,c0:c1], me[:,:,c0:c1],
                                 _ts[:,:,c0:c1], _tc[:,:,c0:c1], _hat[:,:,c0:c1]))
            pk = (torch.cat(preds, 2)*_stt + _mnt).clamp(CLAMP_LO, CLAMP_HI)
        return pk[0].cpu().numpy()

    def _pred_graph(net, m_TN):
        net.eval()
        m_ev = torch.tensor(m_TN, device=device).T.unsqueeze(0); me = m_ev*_vt
        xi = _xt*me + _hat*(1-me)
        feat = torch.stack([xi[0], me[0], _ts[0], _tc[0]], dim=-1).unsqueeze(0)
        with torch.no_grad():
            pk = (net(feat, A_t)*_stt + _mnt).clamp(CLAMP_LO, CLAMP_HI)
        return pk[0].cpu().numpy()

    def _pred_classical(fn, m_TN):
        return np.clip(fn(_xn.T, _vv.T, m_TN.T, _idx), CLAMP_LO, CLAMP_HI)

    _ha_fn = lambda xn, vn, mn, idx: tod_mean[:, idx%STEPS_PER_DAY]*node_stds[:,None] + node_means[:,None]
    PATTERN_MODELS = [
        ('MARST (ours)', 'st',        net_marst),
        ('DCRNN',        'graph',     dcrnn),
        ('GWN',          'graph',     gwn),
        ('KNN',          'classical', knn_masked),
        ('LOCF',         'classical', locf),
        ('HA',           'classical', _ha_fn),
    ]
    _PRED = {'st': _pred_st, 'graph': _pred_graph, 'classical': _pred_classical}

    PATTERN_RESULTS = {p: {} for p in MISSING_PATTERNS}
    for _pat in MISSING_PATTERNS:
        for _name, _kind, _obj in PATTERN_MODELS:
            _maes = []
            for _seed in EVAL_SEEDS:
                _m = make_pattern_mask_np(_seed, _EL, NUM_NODES, SPARSITY, _pat)
                _pk = _PRED[_kind](_obj, _m)
                _sm = (_m.T == 0) & (_vv.T > 0)
                _e  = np.abs(_pk[_sm] - _tk_full[_sm]); _e = _e[np.isfinite(_e)]
                _maes.append(float(_e.mean()))
            PATTERN_RESULTS[_pat][_name] = np.array(_maes)
        print(f"[{_pat:>6}] " + "  ".join(
            f"{n}={PATTERN_RESULTS[_pat][n].mean():.3f}" for n, _, _ in PATTERN_MODELS))

    _order = sorted(PATTERN_RESULTS['point'], key=lambda k: PATTERN_RESULTS['point'][k].mean())
    _W = 26 + 14*len(MISSING_PATTERNS)
    print("\n" + "=" * _W)
    print(f"{'MISSING-PATTERN ROBUSTNESS  (' + DATASET_NAME + f', {int(SPARSITY*100)}% sparsity)':^{_W}}")
    print("=" * _W)
    print(f"{'Model':<26}" + "".join(f"{p+' MAE':>14}" for p in MISSING_PATTERNS))
    print("-" * _W)
    for _k in _order:
        print(f"{_k:<26}" + "".join(f"{PATTERN_RESULTS[p][_k].mean():>14.4f}" for p in MISSING_PATTERNS))
    print("=" * _W)

    MISSING_PATTERN_SAVE = {p: {k: [float(x) for x in v] for k, v in d.items()}
                            for p, d in PATTERN_RESULTS.items()}

    fig, ax = plt.subplots(figsize=(11, 4.6))
    _x = np.arange(len(_order)); _wb = 0.8/len(MISSING_PATTERNS)
    for j, p in enumerate(MISSING_PATTERNS):
        ax.bar(_x + j*_wb, [PATTERN_RESULTS[p][k].mean() for k in _order], _wb,
               yerr=[PATTERN_RESULTS[p][k].std() for k in _order], capsize=2, label=p)
    ax.set_xticks(_x + _wb*(len(MISSING_PATTERNS)-1)/2)
    ax.set_xticklabels(_order, rotation=35, ha='right')
    ax.set_ylabel(f'MAE ({VALUE_UNIT})'); ax.legend(title='missing pattern')
    ax.set_title(f'Missing-pattern robustness -- {DATASET_NAME} ({int(SPARSITY*100)}% sparsity)')
    ax.grid(axis='y', alpha=.25)
    plt.tight_layout(); plt.savefig(f'fig_missing_patterns_{DATASET_NAME}.png', dpi=140, bbox_inches='tight'); plt.show()
    print(f"Saved fig_missing_patterns_{DATASET_NAME}.png")


    # ============================================================================
    #  Multi-Rate Sparsity Sweep (no retraining)
    #  ------------------------------------------------------------------
    #  Evaluates every trained model at multiple sparsity levels using the
    #  EXISTING trained weights — produces the standard paper main table:
    #    methods (rows) x sparsity rates (columns) x {MAE, RMSE}.
    #  Uses the shared eval-window arrays from the missing-pattern block.
    # ============================================================================
    MULTI_RATE_LEVELS = SPARSITY_LEVELS  # follows the shrunk SPARSITY_LEVELS above
    MULTI_RATE = {}

    def _eval_any_at_sparsity(mdl, kind, sparsity):
        """Eval one model at one sparsity. Returns dict(mae=[...], rmse=[...])
        with one entry per EVAL_SEED."""
        ES_, EL_ = EVAL_START, EVAL_LEN
        st_ = node_stds[:, None]; mn_ = node_means[:, None]
        maes, rmses = [], []
        for seed in EVAL_SEEDS:
            m_TN = make_eval_mask_np(seed, EL_, NUM_NODES, sparsity=sparsity)  # [T,N]
            if kind == 'classical':
                p_kmh = np.clip(mdl(_xn.T, _vv.T, m_TN.T, _idx), CLAMP_LO, CLAMP_HI)
                t_kmh = _tk_full
                sm    = (m_TN.T == 0) & (_vv.T > 0)
                err   = (p_kmh[sm] - t_kmh[sm]).astype(np.float64)
                err   = err[np.isfinite(err)]
                if err.size == 0: continue
                maes.append(float(np.abs(err).mean()))
                rmses.append(float(np.sqrt((err**2).mean())))
            elif kind == 'nw':
                mdl.eval()
                m_ev = torch.tensor(m_TN, device=device)               # [T,N]
                xev  = speed_gpu[ES_:ES_+EL_]; vev = valid_gpu[ES_:ES_+EL_]
                me   = m_ev * vev
                idx  = torch.arange(ES_, ES_+EL_, device=device)
                s    = torch.sin(2*np.pi*(idx % STEPS_PER_DAY)/STEPS_PER_DAY)
                c    = torch.cos(2*np.pi*(idx % STEPS_PER_DAY)/STEPS_PER_DAY)
                feat = torch.stack([(xev*me).T, me.T,
                    s.unsqueeze(0).expand(NUM_NODES, -1),
                    c.unsqueeze(0).expand(NUM_NODES, -1)], dim=-1)
                with torch.no_grad():
                    pred = mdl(feat).squeeze(-1)                       # [N,T]
                stt = torch.tensor(node_stds, device=device)
                mnt = torch.tensor(node_means, device=device)
                pk  = (pred  * stt[:, None] + mnt[:, None]).clamp(CLAMP_LO, CLAMP_HI)
                tk  = (xev.T * stt[:, None] + mnt[:, None]).clamp(CLAMP_LO, CLAMP_HI)
                sm  = (m_ev.T == 0) & (vev.T > 0)
                err = (pk[sm] - tk[sm]).detach().cpu().numpy().astype(np.float64)
                if err.size == 0: continue
                maes.append(float(np.abs(err).mean()))
                rmses.append(float(np.sqrt((err**2).mean())))
            elif kind == 'graph':
                mdl.eval()
                m_ev = torch.tensor(m_TN, device=device).T.unsqueeze(0)  # [1,N,T]
                me   = m_ev * _vt
                xi   = _xt*me + _hat*(1-me)
                feat = torch.stack([xi[0], me[0], _ts[0], _tc[0]], dim=-1).unsqueeze(0)
                with torch.no_grad():
                    pk = (mdl(feat, A_t)*_stt + _mnt).clamp(CLAMP_LO, CLAMP_HI)
                tk_all = (_xt*_stt + _mnt).clamp(CLAMP_LO, CLAMP_HI)
                sm_t   = (m_ev == 0) & (_vt > 0)
                err = (pk[sm_t] - tk_all[sm_t]).detach().cpu().numpy().astype(np.float64)
                if err.size == 0: continue
                maes.append(float(np.abs(err).mean()))
                rmses.append(float(np.sqrt((err**2).mean())))
            elif kind == 'st':
                mdl.eval()
                m_ev = torch.tensor(m_TN, device=device).T.unsqueeze(0)
                me   = m_ev * _vt
                preds = []
                with torch.no_grad():
                    for c0 in range(0, EL_, BATCH_TIME):
                        c1 = min(c0 + BATCH_TIME, EL_)
                        preds.append(mdl(_xt[:,:,c0:c1]*me[:,:,c0:c1], me[:,:,c0:c1],
                                         _ts[:,:,c0:c1], _tc[:,:,c0:c1], _hat[:,:,c0:c1]))
                p   = torch.cat(preds, dim=2)
                pk  = (p*_stt + _mnt).clamp(CLAMP_LO, CLAMP_HI)
                tk_all = (_xt*_stt + _mnt).clamp(CLAMP_LO, CLAMP_HI)
                sm_t = (m_ev == 0) & (_vt > 0)
                err = (pk[sm_t] - tk_all[sm_t]).detach().cpu().numpy().astype(np.float64)
                if err.size == 0: continue
                maes.append(float(np.abs(err).mean()))
                rmses.append(float(np.sqrt((err**2).mean())))
            else:
                raise ValueError(f"unknown kind {kind!r}")
        return dict(mae=maes, rmse=rmses)

    print("\nMulti-rate sweep (no retraining): {} models x {} rates ...".format(
        len(TRAINED_MODELS), len(MULTI_RATE_LEVELS)))
    for _label, (_mdl, _kind) in TRAINED_MODELS.items():
        MULTI_RATE[_label] = {f"{sp:.2f}": _eval_any_at_sparsity(_mdl, _kind, sp)
                              for sp in MULTI_RATE_LEVELS}
    print(f"  Multi-rate sweep done.")

    # Save this run's results for cross-dataset aggregation
    def _arr2list(d):
        """Serialise RESULTS dict (1D for classical, 2D for nn models) to JSON-safe nested lists."""
        out = {}
        for k, v in d.items():
            a = np.asarray(v)
            if a.ndim == 1:
                out[k] = [float(x) for x in a]
            elif a.ndim == 2:
                out[k] = [[float(x) for x in row] for row in a]
            else:
                out[k] = a.tolist()
        return out

    save_obj = dict(
        dataset=DATASET_NAME,
        kind=CFG['kind'],
        unit=CFG['unit'],
        sparsity=SPARSITY,
        num_nodes=int(NUM_NODES),
        T=int(value_raw.shape[0]),
        valid_fraction=float(valid_raw.mean()),
        data_url=CFG.get('data_url', ''),
        model_stats=MODEL_STATS,
        seeds=EVAL_SEEDS,
        baseline_train_seeds=list(range(N_BASELINE_SEEDS)),
        marst_train_seeds=list(TRAIN_SEEDS),
        hit_tol=[float(HIT_TOL[0]), float(HIT_TOL[1])],
        regime_edges=[float(REGIME_EDGES[0]), float(REGIME_EDGES[1])],
        mae=_arr2list(RESULTS),
        ext_metrics=({k: {m: float(val) for m, val in d.items()} for k, d in EXT_METRICS.items()}
                     if 'EXT_METRICS' in globals() else {}),
        ablation=globals().get('ABLATION_SAVE', {}),
        sparsity_sweep=globals().get('SPARSITY_SAVE', {}),
        marst_anchor_ablation=globals().get('MARST_ANCHOR_ABLATION_SAVE', {}),
        missing_patterns=globals().get('MISSING_PATTERN_SAVE', {}),
        multi_rate=MULTI_RATE,
        ridge_alpha_sweep={float(a): [float(x) for x in v] for a, v in RIDGE_ALPHA_SWEEP.items()},
        knn_k_sweep={int(k): [float(x) for x in v] for k, v in KNN_K_SWEEP.items()},
        soft_locf_decay_sweep={float(d): [float(x) for x in v] for d, v in SOFT_LOCF_DECAY_SWEEP.items()},
        curriculum_ablation={k: [float(x) for x in v] for k, v in CURRICULUM_ABLATION.items()},
    )
    out_path = f"results_{DATASET_NAME}.json"
    with open(out_path, 'w') as f:
        json.dump(save_obj, f, indent=2)
    print(f"Saved {out_path}")

    # Also dump raw per-model held-out predictions so future metric tweaks
    # (new JamMAE threshold, new Hit tolerance, additional metric) can be
    # evaluated WITHOUT retraining. See the "Recompute from saved predictions"
    # cell at the bottom of this notebook.
    try:
        _names = list(EXT_PRED.keys())
        _true  = next(iter(EXT_PRED.values()))[1].astype('float32')
        _pred_arrs = {f"pred_{i:03d}": EXT_PRED[n][0].astype('float32')
                      for i, n in enumerate(_names)}
        np.savez_compressed(f"predictions_{DATASET_NAME}.npz",
                            __names__=np.array(_names),
                            __true__=_true,
                            **_pred_arrs)
        print(f"Saved predictions_{DATASET_NAME}.npz ({len(_names)} models, "
              f"{_true.size:,} held-out points)")
    except Exception as _e:
        print(f"Warning: could not save predictions_{DATASET_NAME}.npz: {_e}")
    print(f"  models in MAE table : {len(save_obj['mae'])}")
    print(f"  extended-metric rows: {len(save_obj['ext_metrics'])}")
    print(f"  ablation variants   : {len(save_obj['ablation'])}")
    print(f"  sparsity levels     : {len(save_obj['sparsity_sweep'].get('MARST (ours)', {}))}")
    print(f"  MARST anchor variants: {len(save_obj['marst_anchor_ablation'])}")
    print(f"  missing patterns    : {len(save_obj['missing_patterns'])}")

    print(f'[done] {DATASET}: wrote results_{DATASET}.json')
    return DATASET



















## Run All Datasets
Runs PEMS-BAY → METR-LA → PEMS04 → PEMS08 sequentially, **checkpointing** to `results_<DATASET>.json` after each and **skipping** any already done (resume after a Colab disconnect; flip `FORCE_RERUN` to recompute). Expect this to take a while on GPU. The figure cells further below render for the **last** dataset processed; the cross-dataset cells compare all of them.

In [ ]:
# Driver: run every dataset in one go. Checkpoints to results_<DATASET>.json
# after each, and SKIPS any dataset already done (set FORCE_RERUN=True to redo).
import os, traceback
DATASETS_TO_RUN = ['PEMS-BAY', 'METR-LA', 'PEMS04', 'PEMS08']
FORCE_RERUN = False
for _ds in DATASETS_TO_RUN:
    _out = f'results_{_ds}.json'
    if os.path.exists(_out) and not FORCE_RERUN:
        print(f'skip {_ds}: {_out} exists (set FORCE_RERUN=True to recompute)')
        continue
    try:
        run_dataset(_ds)
    except Exception as _e:
        print(f'!! FAILED {_ds}: {_e}'); traceback.print_exc()
print('\nAll requested datasets processed. Run the cross-dataset + figure cells below.')


## Save run + Cross-Dataset Aggregation
`run_dataset()` writes `results_<DATASET>.json` per dataset (MAE table, extended metrics, ablation, sparsity sweep). The aggregation cell below stitches every saved run into a single cross-dataset table; the comparison cell after it plots them.

In [ ]:
# Cross-dataset aggregation: loads every results_<DATASET>.json present.
import glob
files = sorted(glob.glob('results_*.json'))
runs = {}
for fp in files:
    try:
        o = json.load(open(fp))
        runs[o['dataset']] = o
    except Exception as e:
        print(f"skip {fp}: {e}")
print(f"Found {len(runs)} dataset run(s): {list(runs.keys())}\n")

def _mean(o, name):
    v = o['mae'].get(name)
    return float(np.mean(v)) if v else float('nan')

if runs:
    ds_names = list(runs.keys())
    ref = ds_names[0]
    model_names = set()
    for o in runs.values():
        model_names |= set(o['mae'].keys())
    model_names = sorted(model_names,
                         key=lambda n: _mean(runs[ref], n) if not np.isnan(_mean(runs[ref], n)) else 1e9)

    W = 46 + 13 * len(ds_names)
    title = f"CROSS-DATASET MAE  ({int(SPARSITY*100)}% sparsity)"
    print("=" * W)
    print(f"{title:^{W}}")
    print("=" * W)
    print(f"{'Model':<46}" + "".join(f"{d:>13}" for d in ds_names))
    print("-" * W)
    for name in model_names:
        row = f"{name:<46}"
        for d in ds_names:
            mv = _mean(runs[d], name)
            row += (f"{mv:>13.4f}" if not np.isnan(mv) else f"{'-':>13}")
        print(row)
    print("=" * W)

    # Extended metrics, per dataset (compact)
    for d, o in runs.items():
        em = o.get('ext_metrics', {})
        if not em:
            continue
        mkeys = ["MAE", "JamMAE", "RMSE", "MedAE", "MAPE", "R2", "Pearson", "MBE", "Hit5", "Hit10"]
        tol = o.get("hit_tol", [5, 10])
        hdr = ["MAE", "JamMAE", "RMSE", "MedAE", "MAPE", "R2", "Pearson", "MBE",
               f"Hit@{tol[0]:g}", f"Hit@{tol[1]:g}"]
        ww = 10
        WW = 20 + ww * len(mkeys)
        print("\n" + "-" * WW)
        print(f"Extended metrics -- {d}")
        print("-" * WW)
        print(f"{'Model':<20}" + "".join(f"{c:>{ww}}" for c in hdr))
        for mname, mv in em.items():
            print(f"{mname:<20}" + "".join(f"{mv.get(c, float('nan')):>{ww}.3f}" for c in mkeys))
else:
    print("No results_*.json found yet. Run the notebook once per DATASET, then re-run this cell.")


## Cross-Dataset Comparison
Compares the saved runs. Absolute MAE is not comparable across datasets (speed = km/h vs flow = veh/5min), so these plots use **unit-agnostic** measures: MARST's improvement over HA and over the best baseline (%), and R²/Pearson/MAPE/JamMAE/Hit across datasets.

In [ ]:
# ============================================================================
#  Cross-dataset comparison plots.  Reads every results_<DATASET>.json.
#  NB: absolute MAE is NOT comparable across datasets (speed=km/h vs flow=veh/5min),
#  so cross-dataset plots use UNIT-AGNOSTIC measures: improvement-over-HA (%),
#  R2, Pearson, MAPE(%), Hit-rates, and MARST-vs-best-baseline gap (%).
# ============================================================================
import glob as _glob, json as _json, numpy as np
import matplotlib.pyplot as plt

_runs = {}
for _fp in sorted(_glob.glob('results_*.json')):
    try:
        _o = _json.load(open(_fp)); _runs[_o['dataset']] = _o
    except Exception as _e:
        print(f"skip {_fp}: {_e}")

MK = '19. MARST (ours, multi-anchor)'
HK = '1. Historical Average (HA)'

def _mae(o, k):
    v = o.get('mae', {}).get(k); return float(np.mean(v)) if v else float('nan')

if len(_runs) >= 1:
    _dsl = list(_runs)
    print(f"Cross-dataset comparison over {len(_dsl)} run(s): {_dsl}")

    # ---- Fig X1 — MARST improvement over HA, and over the best baseline (%) ----
    impr_ha, impr_best, best_names = [], [], []
    for d in _dsl:
        o = _runs[d]; mm = _mae(o, MK); ha = _mae(o, HK)
        impr_ha.append(100 * (ha - mm) / ha if ha == ha else np.nan)
        cand = {k: _mae(o, k) for k in o.get('mae', {})
                if k != MK and not k.endswith('(ours, multi-anchor)') and np.isfinite(_mae(o, k))}
        bk = min(cand, key=cand.get) if cand else None
        impr_best.append(100 * (cand[bk] - mm) / cand[bk] if bk else np.nan)
        best_names.append(bk.split('. ', 1)[-1][:12] if bk else '-')
    x = np.arange(len(_dsl)); w = 0.38
    fig, ax = plt.subplots(figsize=(max(7, 2.3 * len(_dsl)), 5))
    b1 = ax.bar(x - w/2, impr_ha,   w, label='vs HA',            color='#1f77b4', edgecolor='black', lw=.7)
    b2 = ax.bar(x + w/2, impr_best, w, label='vs best baseline', color='#D62728', edgecolor='black', lw=.7)
    for bars in (b1, b2):
        for bb in bars:
            ax.text(bb.get_x()+bb.get_width()/2, bb.get_height()+0.2, f'{bb.get_height():.1f}%',
                    ha='center', fontsize=8)
    for xi, bn in zip(x, best_names):
        ax.text(xi + w/2, 0.2, bn, rotation=90, ha='center', va='bottom', fontsize=7, color='white')
    ax.set_xticks(x); ax.set_xticklabels(_dsl)
    ax.set_ylabel('MARST MAE improvement (%)'); ax.axhline(0, color='#333', lw=1)
    ax.set_title('MARST improvement across datasets  (higher = better)')
    ax.legend(); ax.grid(axis='y', alpha=.25)
    plt.tight_layout(); plt.savefig('figX1_marst_improvement_cross.png', dpi=200, bbox_inches='tight')
    plt.savefig('figX1_marst_improvement_cross.pdf', bbox_inches='tight'); plt.show()
    print('Saved figX1_marst_improvement_cross.png/.pdf')

    # ---- Fig X2 — MARST unit-agnostic metric heatmap (datasets x metrics) ----
    _mk = ['R2', 'Pearson', 'MAPE', 'JamMAE', 'Hit5', 'Hit10']
    _have = [d for d in _dsl if MK in _runs[d].get('ext_metrics', {})]
    if _have:
        # normalise each metric column to [0,1] (higher=better; MAPE/JamMAE inverted) for colour only
        raw = np.array([[ _runs[d]['ext_metrics'][MK].get(m, np.nan) for m in _mk] for d in _have], float)
        norm = np.zeros_like(raw); lower_better = {'MAPE', 'JamMAE'}
        for j, m in enumerate(_mk):
            col = raw[:, j]; rng = np.nanmax(col) - np.nanmin(col) + 1e-9
            z = (col - np.nanmin(col)) / rng
            norm[:, j] = (1 - z) if m in lower_better else z
        fig, ax = plt.subplots(figsize=(1.4 * len(_mk) + 2, 0.8 * len(_have) + 2))
        im = ax.imshow(norm, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
        ax.set_xticks(range(len(_mk))); ax.set_xticklabels(_mk)
        ax.set_yticks(range(len(_have))); ax.set_yticklabels(_have)
        for i in range(len(_have)):
            for j, m in enumerate(_mk):
                v = raw[i, j]
                ax.text(j, i, ('%.3f' % v) if m in ('R2', 'Pearson') else ('%.1f' % v),
                        ha='center', va='center', fontsize=9,
                        color='black' if .25 < norm[i, j] < .8 else 'white')
        ax.set_title('MARST metrics across datasets  (green = better)')
        plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
        plt.tight_layout(); plt.savefig('figX2_marst_metrics_cross.png', dpi=200, bbox_inches='tight')
        plt.savefig('figX2_marst_metrics_cross.pdf', bbox_inches='tight'); plt.show()
        print('Saved figX2_marst_metrics_cross.png/.pdf')
    else:
        print('Fig X2 skipped: no MARST ext_metrics in saved runs.')
else:
    print('No results_*.json found — run the driver loop first.')




## Visualisations
Publication figures **fig1–6** (all-model comparison): benchmark bar, extended-metrics heatmap, error CDF / hit-rate curves, RMSE-vs-MAE scatter, per-seed stability, and the MARST learned anchor mixture **π**.

In [ ]:
# ============================================================================
#  Publication figures (all-model comparison).  PNG @ 300 dpi.
#  Consumes: RESULTS (per-seed MAE), EXT_PRED (flat preds), EXT_METRICS,
#            marst_mask_mat, net_marst.
# ============================================================================
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import numpy as np, re

plt.rcParams.update({
    'figure.dpi'      : 120,
    'savefig.dpi'     : 300,
    'font.size'       : 12,
    'axes.titlesize'  : 13,
    'axes.labelsize'  : 12,
    'axes.titleweight': 'bold',
    'legend.fontsize' : 9,
    'axes.spines.top' : False,
    'axes.spines.right': False,
    'figure.autolayout': False,
})

# ── Model metadata ─────────────────────────────────────────────────────────
FAMILY_COLOR = {'stat':'#7F7F7F', 'linear':'#9E9E9E', 'nonparam':'#BCBD22',
                'nn':'#1F77B4', 'rnn':'#17BECF', 'conv':'#2CA02C',
                'attn':'#9467BD', 'graph':'#FF7F0E', 'recent':'#8C564B', 'ours':'#D62728'}
FAMILY_LABEL = {'stat':'Statistical', 'linear':'Linear', 'nonparam':'Non-parametric',
                'nn':'Neural', 'rnn':'RNN', 'conv':'TCN', 'attn':'Attention',
                'graph':'Graph', 'recent':'Recent SOTA', 'ours':'Ours'}
MODEL_FAMILY = {
    '1. Historical Average (HA)':'stat', '2. LOCF (Last-Obs Carried Forward)':'stat',
    '3. Global Mean (per-node train mean)':'stat', '4. Node-wise Ridge Regression':'linear',
    '5. KNN Imputer (k=5, masked-dist)':'nonparam', '6.  MLP (per-node + node-emb)':'nn',
    '7.  LSTM (per-node)':'rnn', '8.  2L-LSTM (per-node, causal)':'rnn',
    '9.  GRU (per-node)':'rnn', '10. TCN (causal dilated, per-node)':'conv',
    '11. SAITS (causal Attn + node-emb)':'attn',
    '12. BRITS (forward GRU only, causal)':'rnn',
    '13. DCRNN (DiffGCN + GRU)':'graph',
    '14. GWN (Adaptive GCN + Gated TCN)':'graph',
    '19. MARST (ours, multi-anchor)':'ours',
    '15. STID (node+ToD identity, causal)':'recent', '16. DLinear (decomp + causal linear)':'recent',
    '17. PatchTST (causal patches)':'recent', '18. iTransformer (cross-sensor, causal)':'recent'}
MODEL_SHORT = {
    '1. Historical Average (HA)':'HA', '2. LOCF (Last-Obs Carried Forward)':'LOCF',
    '3. Global Mean (per-node train mean)':'GlobalMean', '4. Node-wise Ridge Regression':'Ridge',
    '5. KNN Imputer (k=5, masked-dist)':'KNN', '6.  MLP (per-node + node-emb)':'MLP',
    '7.  LSTM (per-node)':'LSTM', '8.  2L-LSTM (per-node, causal)':'2L-LSTM',
    '9.  GRU (per-node)':'GRU', '10. TCN (causal dilated, per-node)':'TCN',
    '11. SAITS (causal Attn + node-emb)':'SAITS',
    '12. BRITS (forward GRU only, causal)':'BRITS',
    '13. DCRNN (DiffGCN + GRU)':'DCRNN',
    '14. GWN (Adaptive GCN + Gated TCN)':'GWN',
    '19. MARST (ours, multi-anchor)':'MARST',
    '15. STID (node+ToD identity, causal)':'STID', '16. DLinear (decomp + causal linear)':'DLinear',
    '17. PatchTST (causal patches)':'PatchTST', '18. iTransformer (cross-sensor, causal)':'iTransformer'}

MARST_KEY = '19. MARST (ours, multi-anchor)'
OURS_KEYS = {MARST_KEY}
def short(k):  return MODEL_SHORT.get(k, k.split('. ', 1)[-1][:12])
def fam(k):    return MODEL_FAMILY.get(k, 'stat')
def mcolor(k):                       # MARST stands out
    if k == MARST_KEY: return '#D62728'
    return FAMILY_COLOR[fam(k)]
def _lead(k):
    m = re.match(r'\s*(\d+)', k); return int(m.group(1)) if m else 999

ha_mae   = RESULTS['1. Historical Average (HA)'].mean()
locf_mae = RESULTS['2. LOCF (Last-Obs Carried Forward)'].mean()
DSET = DATASET_NAME

# ════════════════════════════════════════════════════════════════════════
# Fig 1 — Benchmark: MAE ± std, all models, family-coloured, ours highlighted
# ════════════════════════════════════════════════════════════════════════
items  = sorted(RESULTS.items(), key=lambda kv: kv[1].mean(), reverse=True)
labels = [short(k) for k, _ in items]
maes   = np.array([v.mean() for _, v in items])
stds   = np.array([v.std()  for _, v in items])
colors = [mcolor(k) for k, _ in items]

fig, ax = plt.subplots(figsize=(9, 7.5))
y = np.arange(len(labels))
bars = ax.barh(y, maes, xerr=stds, color=colors, height=0.68,
               error_kw=dict(ecolor='#333', capsize=2.5, elinewidth=1.0))
for k, b in zip([k for k, _ in items], bars):     # outline ours
    if k in OURS_KEYS:
        b.set_edgecolor('black'); b.set_linewidth(1.6)
ax.axvline(ha_mae,   color='#444', ls='--', lw=1.4, alpha=.8, label=f'HA ({ha_mae:.2f})')
ax.axvline(locf_mae, color='#888', ls=':',  lw=1.4, alpha=.8, label=f'LOCF ({locf_mae:.2f})')
ax.set_yticks(y); ax.set_yticklabels(labels)
ax.set_xlabel(f'MAE ({VALUE_UNIT})  —  lower is better')
ax.set_title(f'{DSET} Imputation Benchmark  |  80% Sparsity')
ax.invert_yaxis(); ax.grid(axis='x', alpha=.25); ax.set_xlim(0, maes.max()*1.15)
fams_seen = dict.fromkeys(fam(k) for k, _ in items)
fam_handles = [mpatches.Patch(color=FAMILY_COLOR[f], label=FAMILY_LABEL[f]) for f in fams_seen]
leg1 = ax.legend(handles=fam_handles, loc='lower right', title='Family', framealpha=.95)
ax.add_artist(leg1); ax.legend(loc='upper right', framealpha=.95)
if MARST_KEY in RESULTS:
    bm = RESULTS[MARST_KEY].mean(); bi = labels.index(short(MARST_KEY))
    ax.annotate(f'MARST: +{100*(ha_mae-bm)/ha_mae:.1f}% vs HA', xy=(bm, bi),
                xytext=(bm + .10, bi - 0.9), color='#D62728', fontweight='bold', fontsize=10,
                arrowprops=dict(arrowstyle='->', color='#D62728', lw=1.4))
plt.tight_layout(); plt.savefig('fig1_benchmark.png', bbox_inches='tight'); plt.show()
print('Saved fig1_benchmark.png')

# ════════════════════════════════════════════════════════════════════════
# Fig 2 — Extended-metrics heatmap, all models × 9 metrics (green = best/col)
# ════════════════════════════════════════════════════════════════════════
names  = list(EXT_METRICS.keys())
mkeys  = ["MAE", "JamMAE", "RMSE", "MedAE", "MAPE", "R2", "Pearson", "MBE", "Hit5", "Hit10"]
mlabs  = [f"MAE\n({VALUE_UNIT})", f"JamMAE\n({VALUE_UNIT})", f"RMSE\n({VALUE_UNIT})", f"MedAE\n({VALUE_UNIT})", "MAPE\n(%)",
          "R²", "Pearson\nr", f"MBE\n({VALUE_UNIT})", f"Hit\n@{HIT_TOL[0]:g}", f"Hit\n@{HIT_TOL[1]:g}"]
raw = np.array([[EXT_METRICS[n][k] for k in mkeys] for n in names])
norm = np.zeros_like(raw); higher = {"R2", "Pearson", "Hit5", "Hit10"}
for j, k in enumerate(mkeys):
    col = raw[:, j]; rng = col.max() - col.min() + 1e-9
    norm[:, j] = (col-col.min())/rng if k in higher else 1-(col-col.min())/rng
fig, ax = plt.subplots(figsize=(13, 0.5*len(names) + 2))
im = ax.imshow(norm, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
ax.set_xticks(range(len(mkeys))); ax.set_xticklabels(mlabs, fontsize=10)
ax.set_yticks(range(len(names)));  ax.set_yticklabels([short(n) for n in names], fontsize=10)
for n_i, n in enumerate(names):
    if n in OURS_KEYS: ax.get_yticklabels()[n_i].set_fontweight('bold')
    for j, k in enumerate(mkeys):
        v = raw[n_i, j]
        txt = f"{v:.3f}" if k in ("R2", "Pearson") else (f"{v:.1f}" if k in ("MAPE","Hit5","Hit10") else f"{v:.3f}")
        ax.text(j, n_i, txt, ha='center', va='center', fontsize=7.5,
                color='black' if .25 < norm[n_i, j] < .8 else 'white')
plt.colorbar(im, ax=ax, fraction=0.02, pad=0.01, label='normalised (1 = best in column)')
ax.set_title(f'Extended Metrics — {DSET}  (green = best per column)')
plt.tight_layout(); plt.savefig('fig2_metrics_heatmap.png', bbox_inches='tight'); plt.show()
print('Saved fig2_metrics_heatmap.png')

# ════════════════════════════════════════════════════════════════════════
# Fig 3 — Error CDF + Hit-rate curves, all models (family-coloured, ours bold)
# ════════════════════════════════════════════════════════════════════════
abs_err = {k: np.abs(p - t) for k, (p, t) in EXT_PRED.items()}
order   = sorted(abs_err, key=_lead)
fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))

ax = axes[0]
for k in order:
    e = abs_err[k]; xs = np.sort(e); ys = np.arange(1, len(xs)+1)/len(xs)
    step = max(1, len(xs)//2000)
    lw, a, z = (2.6, 1.0, 5) if k in OURS_KEYS else (1.0, .55, 2)
    ax.plot(xs[::step], ys[::step], color=mcolor(k), lw=lw, alpha=a, zorder=z)
ax.set_xlim(0, 12); ax.set_ylim(0, 1)
ax.set_xlabel(f'Absolute error ({VALUE_UNIT})'); ax.set_ylabel('Fraction of held-out points')
ax.set_title('Error CDF (higher-left = better)'); ax.grid(alpha=.25)

ax = axes[1]
thr = np.linspace(0, HIT_EPS_MAX, 200)
for k in order:
    e = abs_err[k]
    lw, a, z = (2.6, 1.0, 5) if k in OURS_KEYS else (1.0, .55, 2)
    ax.plot(thr, [(e < x).mean()*100 for x in thr], color=mcolor(k), lw=lw, alpha=a, zorder=z)
for vx in HIT_TOL:
    ax.axvline(vx, color='gray', ls=':', lw=1, alpha=.6)
ax.set_xlim(0, HIT_EPS_MAX); ax.set_ylim(0, 100)
ax.set_xlabel(f'Tolerance ε ({VALUE_UNIT})'); ax.set_ylabel('Hit rate (% within ε)')
ax.set_title('Hit-Rate Curves'); ax.grid(alpha=.25)

fam_handles = [mpatches.Patch(color=FAMILY_COLOR[f], label=FAMILY_LABEL[f])
               for f in dict.fromkeys(fam(k) for k in order)]
fam_handles += [Line2D([], [], color=mcolor(MARST_KEY), lw=2.6, label='MARST (ours)')]
axes[1].legend(handles=fam_handles, loc='lower right', framealpha=.95, ncol=1)
plt.suptitle(f'Cumulative Accuracy — {DSET}  (all models)', fontweight='bold')
plt.tight_layout(); plt.savefig('fig3_cdf_hitrate.png', bbox_inches='tight'); plt.show()
print('Saved fig3_cdf_hitrate.png')

# ════════════════════════════════════════════════════════════════════════
# Fig 4 — RMSE vs MAE scatter (outlier sensitivity), all models
# ════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(8, 7))
for k in sorted(EXT_METRICS, key=_lead):
    m = EXT_METRICS[k]; isours = k in OURS_KEYS
    ax.scatter(m['MAE'], m['RMSE'], s=240 if isours else 90, color=mcolor(k),
               marker='*' if isours else 'o', edgecolors='black' if isours else 'none',
               linewidths=1.2 if isours else 0, zorder=5 if isours else 3, alpha=.9)
    ax.annotate(short(k), (m['MAE'], m['RMSE']), fontsize=8,
                xytext=(4, 3), textcoords='offset points',
                fontweight='bold' if isours else 'normal')
lo = min(m['MAE'] for m in EXT_METRICS.values()) - .1
hi = max(m['MAE'] for m in EXT_METRICS.values()) + .1
ax.plot([lo, hi], [lo, hi], 'k--', lw=1, alpha=.4, label='RMSE = MAE')
ax.set_xlabel(f'MAE ({VALUE_UNIT})'); ax.set_ylabel(f'RMSE ({VALUE_UNIT})')
ax.set_title(f'RMSE vs MAE — {DSET}  (lower-left = better)')
ax.legend(loc='upper left'); ax.grid(alpha=.25)
plt.tight_layout(); plt.savefig('fig4_rmse_vs_mae.png', bbox_inches='tight'); plt.show()
print('Saved fig4_rmse_vs_mae.png')

# ════════════════════════════════════════════════════════════════════════
# Fig 5 — Per-seed stability, all models (strip + mean)
# ════════════════════════════════════════════════════════════════════════
st_items = sorted(RESULTS.items(), key=lambda kv: kv[1].mean())
fig, ax = plt.subplots(figsize=(13, 5))
rng = np.random.default_rng(0)
for i, (k, arr) in enumerate(st_items):
      arr = np.asarray(arr).ravel()                       # MARST is 2D [seeds x masks]; baselines 1D — flatten both
      jit = rng.uniform(-.16, .16, arr.size)
      ax.scatter(np.full(arr.size, i)+jit, arr, s=42, color=mcolor(k),
                 alpha=.85, zorder=4, edgecolors='black' if k in OURS_KEYS else 'none', linewidths=.8)
      ax.hlines(arr.mean(), i-.32, i+.32, color=mcolor(k), lw=2.6, zorder=5)
ax.set_xticks(range(len(st_items)))
ax.set_xticklabels([short(k) for k, _ in st_items], rotation=40, ha='right')
for i, (k, _) in enumerate(st_items):
    if k in OURS_KEYS: ax.get_xticklabels()[i].set_fontweight('bold')
ax.set_ylabel(f'MAE ({VALUE_UNIT})'); ax.set_title(f'Per-Seed Stability — {DSET}')
ax.grid(axis='y', alpha=.25)
ax.text(0.005, -0.32, 'Bars = mean. Ours (MARST): 3 training seeds; baselines: 5 eval-mask seeds.',
        transform=ax.transAxes, fontsize=8.5, style='italic', color='#555')
plt.tight_layout(); plt.savefig('fig5_per_seed_stability.png', bbox_inches='tight'); plt.show()
print('Saved fig5_per_seed_stability.png')

# ════════════════════════════════════════════════════════════════════════
# Fig 6 — MARST learned anchor mixture π (interpretability)
# ════════════════════════════════════════════════════════════════════════
net_marst.eval()
_ES, _EL = EVAL_START, EVAL_LEN
_xb = speed_gpu[_ES:_ES+_EL].T.unsqueeze(0); _hab = ha_prior[_ES:_ES+_EL].T.unsqueeze(0)
_vb = valid_gpu[_ES:_ES+_EL].T.unsqueeze(0)
_ti = torch.arange(_ES, _ES+_EL, device=device)
_ts = torch.sin(2*np.pi*(_ti%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,1,-1).expand(1,NUM_NODES,-1)
_tc = torch.cos(2*np.pi*(_ti%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,1,-1).expand(1,NUM_NODES,-1)
_m = torch.tensor(make_eval_mask_np(EVAL_SEEDS[0], _EL, NUM_NODES), device=device).T.unsqueeze(0)
_me = _m * _vb; _pis = []; pi_mean = None
if getattr(net_marst, 'last_pi', None) is not None:
    with torch.no_grad():
        for c0 in range(0, _EL, BATCH_TIME):
            c1 = min(c0+BATCH_TIME, _EL)
            _ = net_marst(_xb[:,:,c0:c1]*_me[:,:,c0:c1], _me[:,:,c0:c1],
                          _ts[:,:,c0:c1], _tc[:,:,c0:c1], _hab[:,:,c0:c1])
            _pis.append(net_marst.last_pi.mean(0).cpu().numpy())
    pi_mean = np.mean(_pis, axis=0)
fig, ax = plt.subplots(figsize=(6.5, 5))
if pi_mean is not None:
    anames = ['LOCF\n(recency)', 'HA\n(seasonality)', 'KNN\n(spatial)']
    acols  = ['#1f77b4', '#ff7f0e', '#9467bd']
    bars = ax.bar(anames, pi_mean, color=acols, alpha=.9, edgecolor='black', lw=.8)
    for b, v in zip(bars, pi_mean):
        ax.text(b.get_x()+b.get_width()/2, v+.01, f'{v:.2f}', ha='center', fontsize=11, fontweight='bold')
    ax.set_ylim(0, max(pi_mean)*1.25); ax.set_ylabel('mean anchor weight  π')
else:
    ax.text(.5, .5, 'anchor mixture unavailable', ha='center', va='center', transform=ax.transAxes); ax.set_axis_off()
ax.set_title(f'MARST learned anchor mixture — {DSET}', fontweight='bold')
plt.tight_layout(); plt.savefig('fig6_marst_anchor_mix.png', bbox_inches='tight'); plt.show()
print('Saved fig6_marst_anchor_mix.png')


print('\nAll publication figures saved: fig1..fig6 (PNG @ 300 dpi)')






## Ours-vs-Others Comparison Figures
Three figures focused on how MARST compares to the baselines (incl. the recent causal SOTA): per-model improvement, overall-vs-congestion error, and family means.

In [ ]:
# ============================================================================
#  Ours-vs-others comparison figures (complement fig1-6).  PNG @ 300 dpi.
#  Reuses helpers from the figures cell above (short/mcolor/fam/EXT_METRICS/...).
#  Recent causal baselines (STID/DLinear/PatchTST/iTransformer) are family 'recent'.
# ============================================================================

# ── Fig 7 — How far MARST beats every other model (% lower MAE) ─────────────
ref_mae = RESULTS[MARST_KEY].mean()
items7  = sorted([(k, v.mean()) for k, v in RESULTS.items() if k != MARST_KEY],
                 key=lambda kv: kv[1])
labels7 = [short(k) for k, _ in items7]
impr    = np.array([100.0 * (m - ref_mae) / m for _, m in items7])   # +% = MARST better
cols7   = [mcolor(k) for k, _ in items7]

fig, ax = plt.subplots(figsize=(9, 7.2))
y = np.arange(len(labels7))
bars = ax.barh(y, impr, color=cols7, height=0.7)
for (k, _), b in zip(items7, bars):
    if fam(k) == 'recent':
        b.set_edgecolor('black'); b.set_linewidth(1.5)
ax.axvline(0, color='#333', lw=1)
ax.set_yticks(y); ax.set_yticklabels(labels7)
ax.invert_yaxis(); ax.grid(axis='x', alpha=.25)
ax.set_xlabel('MARST improvement  (% lower MAE than this model)')
ax.set_title(f'MARST vs every baseline — {DSET}  (right = MARST wins by more)')
for yi, v in zip(y, impr):
    ax.text(v + (0.4 if v >= 0 else -0.4), yi, f'{v:+.1f}%',
            va='center', ha='left' if v >= 0 else 'right', fontsize=8)
rec_handle = [mpatches.Patch(facecolor='#8C564B', edgecolor='black', label='Recent SOTA (causal)')]
ax.legend(handles=rec_handle, loc='lower right', framealpha=.95)
plt.tight_layout(); plt.savefig('fig7_marst_vs_all.png', bbox_inches='tight'); plt.show()
print('Saved fig7_marst_vs_all.png')

# ── Fig 8 — MAE vs JamMAE (congestion-regime error), ours vs recent vs rest ──
fig, ax = plt.subplots(figsize=(8.5, 7))
for k in sorted(EXT_METRICS, key=_lead):
    m = EXT_METRICS[k]
    if not np.isfinite(m.get('JamMAE', float('nan'))):
        continue
    isours = k in OURS_KEYS; isrec = fam(k) == 'recent'
    mk = '*' if isours else ('s' if isrec else 'o')
    sz = 280 if isours else (130 if isrec else 80)
    ax.scatter(m['MAE'], m['JamMAE'], s=sz, color=mcolor(k), marker=mk,
               edgecolors='black' if (isours or isrec) else 'none',
               linewidths=1.2 if (isours or isrec) else 0,
               zorder=5 if isours else (4 if isrec else 3), alpha=.9)
    ax.annotate(short(k), (m['MAE'], m['JamMAE']), fontsize=8, xytext=(4, 3),
                textcoords='offset points',
                fontweight='bold' if (isours or isrec) else 'normal')
ax.set_xlabel(f'MAE ({VALUE_UNIT})  —  overall')
ax.set_ylabel(f'JamMAE ({VALUE_UNIT})  —  congestion regime')
ax.set_title(f'Overall vs Congestion-Regime Error — {DSET}  (lower-left = better)')
ax.grid(alpha=.25)
legend8 = [Line2D([], [], marker='*', color='w', markerfacecolor='#D62728',
                  markeredgecolor='black', markersize=15, label='Ours (MARST)'),
           Line2D([], [], marker='s', color='w', markerfacecolor='#8C564B',
                  markeredgecolor='black', markersize=10, label='Recent SOTA (causal)'),
           Line2D([], [], marker='o', color='w', markerfacecolor='#888',
                  markersize=9, label='Classical / earlier')]
ax.legend(handles=legend8, loc='upper left', framealpha=.95)
plt.tight_layout(); plt.savefig('fig8_mae_vs_jammae.png', bbox_inches='tight'); plt.show()
print('Saved fig8_mae_vs_jammae.png')

# ── Fig 9 — Mean MAE per model family (landscape summary) ────────────────────
fam_maes = {}
for k, v in RESULTS.items():
    fam_maes.setdefault(fam(k), []).append(v.mean())
order9 = sorted(fam_maes, key=lambda f: np.mean(fam_maes[f]))
fvals  = [np.mean(fam_maes[f]) for f in order9]
fig, ax = plt.subplots(figsize=(9, 4.8))
bars = ax.bar([FAMILY_LABEL.get(f, f) for f in order9], fvals,
              color=[FAMILY_COLOR.get(f, '#777') for f in order9],
              edgecolor='black', linewidth=.8)
for b, v, f in zip(bars, fvals, order9):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.02,
            f'{v:.2f}\n(n={len(fam_maes[f])})', ha='center', fontsize=9)
ax.set_ylabel(f'mean MAE ({VALUE_UNIT})'); ax.set_ylim(0, max(fvals) * 1.18)
ax.set_title(f'Mean MAE by Model Family — {DSET}  (lower = better)')
ax.grid(axis='y', alpha=.25)
plt.xticks(rotation=20, ha='right')
plt.tight_layout(); plt.savefig('fig9_family_means.png', bbox_inches='tight'); plt.show()
print('Saved fig9_family_means.png')

print('\nComparison figures saved: fig7..fig9 (PNG @ 300 dpi)')


## Thesis Figures — Qualitative, Spatial & Cross-Dataset
Explanatory figures for the write-up: qualitative imputation traces (GT vs MARST with held-out gaps shaded), the sensor×time missingness map, a spatial graph view (per-sensor error + dominant anchor), and the cross-dataset bar. Saved as PNG + vector PDF. NOTE: the model architecture schematic must be drawn separately (TikZ / draw.io) — it cannot be auto-generated from run data.

In [ ]:
# ============================================================================
#  Thesis figures: (10) qualitative imputation, (11) data/missingness map,
#  (12) spatial interpretability, (13) cross-dataset bar.  Saved PNG @300dpi + PDF.
#  Uses the representative MARST run (net_marst) and the shared eval window.
# ============================================================================
def _savef(stem):
    plt.savefig(stem + '.png', bbox_inches='tight')
    plt.savefig(stem + '.pdf', bbox_inches='tight')   # vector for the thesis
    plt.show(); print('Saved ' + stem + '.png/.pdf')

def _marst_eval_pass(net, seed=EVAL_SEEDS[0]):
    """One eval-window pass (B=1): de-normed pred/true [N,EL], observed & held
    masks, per-sensor pi [N,3] (reshaped from last_pi) and per-sensor MAE [N]."""
    net.eval()
    ES, EL = EVAL_START, EVAL_LEN
    x_ev  = speed_gpu[ES:ES+EL].T.unsqueeze(0)
    ha_ev = ha_prior [ES:ES+EL].T.unsqueeze(0)
    v_ev  = valid_gpu[ES:ES+EL].T.unsqueeze(0)
    ti = torch.arange(ES, ES+EL, device=device)
    ts = torch.sin(2*np.pi*(ti%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,1,-1).expand(1,NUM_NODES,-1)
    tc = torch.cos(2*np.pi*(ti%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,1,-1).expand(1,NUM_NODES,-1)
    st = torch.tensor(node_stds,  device=device).view(1,-1,1)
    mn = torch.tensor(node_means, device=device).view(1,-1,1)
    m_ev = torch.tensor(make_eval_mask_np(seed, EL, NUM_NODES), device=device).T.unsqueeze(0)
    me = m_ev * v_ev
    preds, pis = [], []
    with torch.no_grad():
        for c0 in range(0, EL, BATCH_TIME):
            c1 = min(c0 + BATCH_TIME, EL)
            p = net(x_ev[:,:,c0:c1]*me[:,:,c0:c1], me[:,:,c0:c1],
                    ts[:,:,c0:c1], tc[:,:,c0:c1], ha_ev[:,:,c0:c1])
            preds.append(p)
            if getattr(net, 'last_pi', None) is not None:
                pis.append(net.last_pi.reshape(NUM_NODES, c1 - c0, 3).cpu().numpy())
    pred = torch.cat(preds, 2)
    pk  = (pred  * st + mn).clamp(CLAMP_LO, CLAMP_HI)[0].cpu().numpy()   # [N,EL]
    tk  = (x_ev  * st + mn).clamp(CLAMP_LO, CLAMP_HI)[0].cpu().numpy()   # [N,EL]
    obs  = me[0].cpu().numpy().astype(bool)
    held = (m_ev[0].cpu().numpy() == 0) & (v_ev[0].cpu().numpy() > 0)
    pi = np.concatenate(pis, axis=1).mean(axis=1) if pis else None       # [N,3]
    err = np.abs(pk - tk)
    mae = np.where(held.any(1), (err * held).sum(1) / np.maximum(held.sum(1), 1), np.nan)
    return dict(pred=pk, true=tk, obs=obs, held=held, pi=pi, mae=mae, EL=EL)

M = _marst_eval_pass(net_marst)
_anames = ['LOCF (recency)', 'HA (seasonality)', 'KNN (spatial)']
_acols  = ['#1f77b4', '#ff7f0e', '#9467bd']

# ── Fig 10 — Qualitative imputation: GT vs MARST, held-out spans shaded ───────
held = M['held']; rng_t = M['true'].max(1) - M['true'].min(1)
cand = np.where(held.sum(1) >= max(3, int(0.25 * M['EL'])))[0]
cand = cand[np.argsort(-rng_t[cand])] if cand.size else np.argsort(-rng_t)
sel = cand[:3] if cand.size >= 3 else np.argsort(-rng_t)[:3]
fig, axes = plt.subplots(len(sel), 1, figsize=(12, 2.5 * len(sel)), sharex=True)
axes = np.atleast_1d(axes)
tx = np.arange(M['EL'])
for ax, n in zip(axes, sel):
    ax.plot(tx, M['true'][n], color='#222', lw=1.6, label='Ground truth', zorder=3)
    ax.plot(tx, M['pred'][n], color='#D62728', lw=1.6, ls='--', label='MARST imputed', zorder=4)
    obs_t = np.where(M['obs'][n])[0]
    ax.scatter(obs_t, M['true'][n][obs_t], s=14, color='#2CA02C', zorder=5, label='Observed (input)')
    in_gap = False
    for t in range(M['EL'] + 1):
        h = t < M['EL'] and held[n, t]
        if h and not in_gap: g0 = t; in_gap = True
        if (not h) and in_gap:
            ax.axvspan(g0 - .5, t - .5, color='#cfe2ff', alpha=.55, zorder=0); in_gap = False
    ax.set_ylabel(f'{VALUE_NAME}\n({VALUE_UNIT})')
    ax.set_title(f'Sensor {n}  |  held-out MAE = {M["mae"][n]:.2f} {VALUE_UNIT}', fontsize=11)
    ax.grid(alpha=.25)
axes[0].legend(loc='upper right', ncol=3, fontsize=9, framealpha=.95)
axes[-1].set_xlabel('Time step (5-min)')
plt.suptitle(f'Qualitative Imputation — {DATASET_NAME}  (blue = held-out gap, 80% missing)', fontweight='bold')
plt.tight_layout(); _savef('fig10_qualitative_imputation')

# ── Fig 11 — Data & missingness map (sensor × time) ──────────────────────────
ns = min(60, NUM_NODES)
true_sub = M['true'][:ns]; obs_sub = M['obs'][:ns]
masked = true_sub.copy(); masked[~obs_sub] = np.nan
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
vmin, vmax = np.nanpercentile(true_sub, 2), np.nanpercentile(true_sub, 98)
im0 = axes[0].imshow(true_sub, aspect='auto', cmap='viridis', vmin=vmin, vmax=vmax)
axes[0].set_title('Ground truth (complete)')
cmap = plt.cm.viridis.copy(); cmap.set_bad('#dddddd')
im1 = axes[1].imshow(masked, aspect='auto', cmap=cmap, vmin=vmin, vmax=vmax)
axes[1].set_title(f'Observed only ({100*(1-obs_sub.mean()):.0f}% missing = grey)')
for ax in axes:
    ax.set_xlabel('Time step'); ax.set_ylabel('Sensor')
plt.colorbar(im1, ax=axes, fraction=0.025, pad=0.02, label=f'{VALUE_NAME} ({VALUE_UNIT})')
plt.suptitle(f'Imputation Task Setup — {DATASET_NAME}  (first {ns} sensors)', fontweight='bold')
_savef('fig11_data_missingness')

# ── Fig 12 — Spatial interpretability: error map + dominant anchor ───────────
def _spectral_layout(Adj):
    d = Adj.sum(1); L = np.diag(d) - Adj
    w, v = np.linalg.eigh(L)
    idx = np.argsort(w)
    xy = v[:, idx[1:3]] if Adj.shape[0] > 2 else np.random.randn(Adj.shape[0], 2)
    xy = (xy - xy.mean(0)) / (xy.std(0) + 1e-9)
    return xy
XY = _spectral_layout(adj)
ex, ey = [], []
ii, jj = np.where(np.triu(adj, 1) > 0)
step = max(1, len(ii) // 1500)
for a, b in zip(ii[::step], jj[::step]):
    ex += [XY[a, 0], XY[b, 0], None]; ey += [XY[a, 1], XY[b, 1], None]
have_pi = M['pi'] is not None
fig, axes = plt.subplots(1, 2 if have_pi else 1, figsize=(14 if have_pi else 7.5, 6.2))
axes = np.atleast_1d(axes)
axes[0].plot(ex, ey, color='#cccccc', lw=.5, zorder=1)
sc = axes[0].scatter(XY[:, 0], XY[:, 1], c=M['mae'], cmap='YlOrRd', s=45,
                     edgecolors='black', linewidths=.4, zorder=2)
plt.colorbar(sc, ax=axes[0], fraction=0.046, pad=0.04, label=f'per-sensor MAE ({VALUE_UNIT})')
axes[0].set_title('Per-sensor error'); axes[0].set_xticks([]); axes[0].set_yticks([])
if have_pi:
    dom = M['pi'].argmax(1)
    axes[1].plot(ex, ey, color='#cccccc', lw=.5, zorder=1)
    for a in range(3):
        sl = dom == a
        axes[1].scatter(XY[sl, 0], XY[sl, 1], c=_acols[a], s=45, edgecolors='black',
                        linewidths=.4, zorder=2, label=_anames[a])
    axes[1].set_title('Dominant anchor (argmax π)'); axes[1].legend(loc='best', fontsize=9)
    axes[1].set_xticks([]); axes[1].set_yticks([])
plt.suptitle(f'Spatial View — {DATASET_NAME}  (graph layout from adjacency)', fontweight='bold')
plt.tight_layout(); _savef('fig12_spatial_interpretability')

# ── Fig 13 — Cross-dataset bar: MARST vs best baseline ──────────────────────
import glob as _glob
runs = {}
for fp in sorted(_glob.glob('results_*.json')):
    try:
        o = json.load(open(fp)); runs[o['dataset']] = o
    except Exception:
        pass
if runs:
    dsl = list(runs.keys())
    def _m(o, key):
        v = o['mae'].get(key); return float(np.mean(v)) if v else np.nan
    mk = '19. MARST (ours, multi-anchor)'
    marst_v, base_v, base_n = [], [], []
    for d in dsl:
        o = runs[d]
        marst_v.append(_m(o, mk))
        cand = {k: _m(o, k) for k in o['mae'] if k != mk and np.isfinite(_m(o, k))}
        bk = min(cand, key=cand.get) if cand else None
        base_v.append(cand[bk] if bk else np.nan)
        base_n.append(bk.split('. ', 1)[-1][:14] if bk else '-')
    x = np.arange(len(dsl)); w = 0.38
    fig, ax = plt.subplots(figsize=(max(7, 2.2 * len(dsl)), 5))
    ax.bar(x - w/2, marst_v, w, label='MARST (ours)', color='#D62728', edgecolor='black', lw=.7)
    ax.bar(x + w/2, base_v,  w, label='Best baseline', color='#7F7F7F', edgecolor='black', lw=.7)
    for xi, bn in zip(x, base_n):
        ax.text(xi + w/2, 0.02, bn, rotation=90, ha='center', va='bottom', fontsize=7, color='white')
    ax.set_xticks(x); ax.set_xticklabels(dsl)
    ax.set_ylabel(f'MAE ({VALUE_UNIT})'); ax.set_title('Cross-Dataset Comparison (lower = better)')
    ax.legend(); ax.grid(axis='y', alpha=.25)
    plt.tight_layout(); _savef('fig13_cross_dataset')
else:
    print('Fig 13 skipped: no results_*.json yet (run more datasets, then re-run this cell).')

print('\nThesis figures done: fig10..fig13')




## Sparsity Plot
Saves `fig_sparsity_<DATASET>.png`. Skips gracefully if the sparsity cell above was not run.

In [ ]:
# Sparsity curve (saved as PNG)
import matplotlib.pyplot as plt

if 'SPARSITY_SWEEP' in globals() and SPARSITY_SWEEP:
    fmts = ['o-', 's--', '^:']
    fig, ax = plt.subplots(figsize=(7, 4.2))
    for (k, d), fmt in zip(SPARSITY_SWEEP.items(), fmts):
        xs = sorted(d)
        ys = [d[sp].mean() for sp in xs]
        es = [d[sp].std()  for sp in xs]
        ax.errorbar([s*100 for s in xs], ys, yerr=es, fmt=fmt, capsize=3, label=k)
    ax.set_xlabel('Test sparsity  (% sensors blind)')
    ax.set_ylabel(f'MAE  ({DATASET_NAME})')
    ax.set_title('Sparsity sensitivity', fontweight='bold')
    ax.legend()
    plt.tight_layout()
    plt.savefig(f'fig_sparsity_{DATASET_NAME}.png', dpi=140, bbox_inches='tight')
    plt.show()
else:
    print("Run the sparsity cell first.")


## Recompute Metrics from Saved Predictions

Cheap path for trying new metric definitions **without retraining**.
Loads `predictions_<DATASET>.npz` + `results_<DATASET>.json` for every dataset on disk,
and re-applies `recompute_metrics(...)` (edit the function below) to each model's raw predictions.
Side-by-side prints the **recomputed** vs **saved** value for each model.

Typical use:
1. Change `OVERRIDE_REGIME_EDGES` / `OVERRIDE_HIT_TOL` below, **or** edit `recompute_metrics`.
2. Run this cell. ~seconds, CPU only, no GPU needed.
3. If you like the new numbers, fold the change into `extended_metrics` (cell 12) and re-run the driver.

In [ ]:
# Recompute metrics from saved per-model predictions. No retraining.
import glob, json
import numpy as np

# --- Optional global overrides (None => use whatever was saved with the run)
OVERRIDE_REGIME_EDGES = None    # e.g. (25.0, 55.0)
OVERRIDE_HIT_TOL      = None    # e.g. (3.0, 6.0)

def recompute_metrics(pred, true, *, kind, regime_edges, hit_tol):
    """Mirror of `extended_metrics` from cell 12 but takes thresholds explicitly.
    Edit this function to add new metrics or change definitions."""
    err = pred - true
    mae   = float(np.abs(err).mean())
    rmse  = float(np.sqrt((err**2).mean()))
    mape  = float((np.abs(err) / np.maximum(true, 1.0)).mean() * 100)
    ss_r  = (err**2).sum(); ss_t = ((true - true.mean())**2).sum()
    r2    = float(1 - ss_r / ss_t) if ss_t > 0 else float('nan')
    mbe   = float(err.mean())
    medae = float(np.median(np.abs(err)))
    pear  = float(np.corrcoef(pred, true)[0, 1])
    hit5  = float((np.abs(err) < hit_tol[0]).mean() * 100)
    hit10 = float((np.abs(err) < hit_tol[1]).mean() * 100)
    if kind == 'speed':
        jam = true < regime_edges[0]
    else:
        jam = true > regime_edges[1]
    jam_mae = float(np.abs(err[jam]).mean()) if jam.any() else float('nan')
    return dict(MAE=mae, JamMAE=jam_mae, RMSE=rmse, MedAE=medae, MAPE=mape,
                R2=r2, Pearson=pear, MBE=mbe, Hit5=hit5, Hit10=hit10)


files = sorted(glob.glob('predictions_*.npz'))
if not files:
    print('No predictions_*.npz on disk. Run the driver cell first (it writes them now).')
else:
    for npz_path in files:
        dataset = npz_path[len('predictions_'):-len('.npz')]
        json_path = f'results_{dataset}.json'
        try:
            with open(json_path) as f:
                meta = json.load(f)
        except FileNotFoundError:
            print(f'[skip] {dataset}: missing {json_path}')
            continue

        kind         = meta['kind']
        regime_edges = tuple(OVERRIDE_REGIME_EDGES or meta['regime_edges'])
        hit_tol      = tuple(OVERRIDE_HIT_TOL      or meta['hit_tol'])

        data  = np.load(npz_path, allow_pickle=True)
        names = [str(x) for x in data['__names__']]
        true  = data['__true__']

        print('')
        print(f'=== {dataset}  ({kind})  regime={regime_edges}  hit_tol={hit_tol} ===')
        hdr = '{:<48}{:>12}{:>14}{:>14}{:>16}'.format(
            'Model', 'MAE (new)', 'MAE (saved)', 'JamMAE (new)', 'JamMAE (saved)')
        print(hdr)
        print('-' * len(hdr))
        for i, name in enumerate(names):
            pred  = data[f'pred_{i:03d}']
            m_new = recompute_metrics(pred, true, kind=kind,
                                       regime_edges=regime_edges, hit_tol=hit_tol)
            saved = meta.get('ext_metrics', {}).get(name, {})
            mae_s = saved.get('MAE', float('nan'))
            jam_s = saved.get('JamMAE', float('nan'))
            print('{:<48}{:>12.4f}{:>14.4f}{:>14.4f}{:>16.4f}'.format(
                name, m_new['MAE'], mae_s, m_new['JamMAE'], jam_s))


## Dataset Summary (Table 1)

Paper-style summary of the four benchmark datasets, built from saved `results_<DATASET>.json`.


In [ ]:
# Paper-style dataset summary table.
import glob, json

files = sorted(glob.glob('results_*.json'))
if not files:
    print('No results_*.json on disk. Run the driver cell first.')
else:
    print('| Dataset  | Kind  | #Nodes | T     | Sparsity | Unit       | Valid % | Source |')
    print('|----------|-------|--------|-------|----------|------------|---------|--------|')
    for fp in files:
        with open(fp) as f: m = json.load(f)
        url = m.get('data_url', '')
        url_short = url.replace('https://', '').split('/')[0] if url else ''
        print('| {:<8} | {:<5} | {:>6} | {:>5} | {:>7.0%} | {:<10} | {:>6.1%} | {} |'.format(
            m['dataset'], m['kind'], m['num_nodes'], m.get('T', '-'), m['sparsity'],
            m.get('unit', '?'), m.get('valid_fraction', float('nan')), url_short))


## Multi-Dataset Anchor Ablation (Matrix)

Compact ablation table: rows = MARST variants, columns = datasets, cells = mean MAE over the 5 eval seeds.
ΔMAE shown relative to Full MARST per column.


In [ ]:
# Multi-dataset MARST anchor ablation matrix.
import glob, json
import numpy as np

files = sorted(glob.glob('results_*.json'))
if not files:
    print('No results_*.json on disk.')
else:
    datasets, ablations = [], {}
    for fp in files:
        with open(fp) as f: m = json.load(f)
        ds = m['dataset']
        ab = m.get('marst_anchor_ablation', {})
        if not ab: continue
        datasets.append(ds)
        for variant, seeds in ab.items():
            ablations.setdefault(variant, {})[ds] = float(np.mean(seeds))

    variants = ['Full MARST (LOCF+HA+KNN)', '- LOCF anchor', '- HA anchor', '- KNN anchor',
                'LOCF only', 'HA only', 'KNN only']
    variants = [v for v in variants if v in ablations]

    print(f"{'Variant':<28}" + ''.join(f"{d:>14}" for d in datasets))
    print('-' * (28 + 14 * len(datasets)))
    full = ablations.get('Full MARST (LOCF+HA+KNN)', {})
    for v in variants:
        row = f"{v:<28}"
        for d in datasets:
            mae  = ablations[v].get(d, float('nan'))
            base = full.get(d, float('nan'))
            row += f'{mae:>8.3f}({mae-base:+.2f})'
        print(row)


## Statistical Significance — MARST vs Best Baseline

**Test**: Wilcoxon signed-rank (two-sided) on per-eval-seed MAE pairs. Reviewer-grade for n=5;
robust to non-normality, no Gaussianity assumption.

**Multiplicity**: Holm-Bonferroni correction across the 4 dataset comparisons. Significance
stars (`*`/`**`/`***`) are read off the **Holm-adjusted** p-value, not the raw one.

Reports MARST mean, best-non-MARST baseline mean, ΔMAE, W statistic, raw p, adjusted p, and
the significance star.


In [ ]:
# Statistical significance: Wilcoxon signed-rank test (paired) + Holm-Bonferroni correction
# across the 4 datasets. Compares MARST vs the best non-MARST baseline per dataset.
import glob, json
import numpy as np
from scipy import stats

MARST_KEY = '19. MARST (ours, multi-anchor)'
files = sorted(glob.glob('results_*.json'))
if not files:
    print('No results_*.json on disk.')
else:
    rows = []
    for fp in files:
        with open(fp) as f: m = json.load(f)
        mae = m.get('mae', {})
        if MARST_KEY not in mae:
            print(f"{m['dataset']}: no MARST entry in {fp}")
            continue
        # 2D arrays from multi-seed training: flatten [n_seeds, n_eval_masks] -> [n_seeds * n_eval_masks]
        marst = np.array(mae[MARST_KEY], dtype=float).ravel()
        others = {k: np.array(v, dtype=float).ravel() for k, v in mae.items() if k != MARST_KEY}
        best_name = min(others, key=lambda k: others[k].mean())
        b = others[best_name]
        n = min(len(marst), len(b))
        a, c = marst[:n], b[:n]
        d = a - c
        # Wilcoxon signed-rank, two-sided. zero_method='wilcox' (default) drops zero diffs.
        try:
            res = stats.wilcoxon(a, c, alternative='two-sided', zero_method='wilcox')
            W, p_raw = float(res.statistic), float(res.pvalue)
        except ValueError:
            W, p_raw = float('nan'), float('nan')
        rows.append({
            'dataset': m['dataset'], 'best_name': best_name,
            'marst_mean': float(a.mean()), 'baseline_mean': float(c.mean()),
            'dmae': float(a.mean() - c.mean()), 'W': W, 'p_raw': p_raw, 'n': n,
        })

    # Holm-Bonferroni correction across the family of `len(rows)` tests
    k = len(rows)
    order = sorted(range(k), key=lambda i: (rows[i]['p_raw'] if rows[i]['p_raw'] == rows[i]['p_raw'] else 1.0))
    adj = [1.0] * k
    running_max = 0.0
    for rank, i in enumerate(order):
        p = rows[i]['p_raw']
        if p != p:
            adj[i] = float('nan')
            continue
        h = min(1.0, (k - rank) * p)
        running_max = max(running_max, h)
        adj[i] = running_max
    for i, r in enumerate(rows):
        r['p_holm'] = adj[i]
        r['sig']    = '***' if r['p_holm'] < 1e-3 else ('**' if r['p_holm'] < 1e-2 else ('*' if r['p_holm'] < 0.05 else 'ns'))

    hdr = f"{'Dataset':<10}{'Best baseline':<46}{'MARST':>9}{'Base':>9}{'dMAE':>9}{'n':>4}{'W':>9}{'p_raw':>11}{'p_holm':>11}  sig"
    print(hdr); print('-' * len(hdr))
    for r in rows:
        print(f"{r['dataset']:<10}{r['best_name'][:44]:<46}{r['marst_mean']:>9.4f}{r['baseline_mean']:>9.4f}"
              f"{r['dmae']:>9.4f}{r['n']:>4d}{r['W']:>9.1f}{r['p_raw']:>11.2e}{r['p_holm']:>11.2e}  {r['sig']}")
    print()
    print('Test: Wilcoxon signed-rank (two-sided) on per-eval-seed MAE pairs.')
    print('p_holm: Holm-Bonferroni adjusted across the {} dataset comparisons.'.format(k))
    print('sig:    *** p<0.001, ** p<0.01, * p<0.05, ns >=0.05.')




## Model Efficiency (Parameter Counts)

Trainable parameters per model. Built from `model_stats` saved during each run. MARST's count includes
the multi-anchor mixture head and meta-gate; classical baselines (HA/LOCF/Global Mean) have no learned
parameters and are omitted.


In [ ]:
# Efficiency / parameter counts. Requires the run to have written `model_stats` into the JSON.
import glob, json

files = sorted(glob.glob('results_*.json'))
if not files:
    print('No results_*.json on disk.')
else:
    # Union of labels seen across datasets
    by_label = {}
    for fp in files:
        with open(fp) as f: m = json.load(f)
        for label, info in (m.get('model_stats') or {}).items():
            if isinstance(info, dict) and info.get('params'):
                by_label[label] = info['params']
    if not by_label:
        print('No model_stats found. Re-run the notebook to populate them.')
    else:
        import re
        def lead(label):
            mm = re.match(r'\s*(\d+)', label)
            return int(mm.group(1)) if mm else 999
        print(f"{'Model':<48}{'Params':>14}{'Params (k)':>14}")
        print('-' * 76)
        for label in sorted(by_label, key=lead):
            n = by_label[label]
            print(f'{label:<48}{n:>14,}{n/1000:>14.1f}')


## Multi-Rate Main Results (paper main table)

All models evaluated at multiple sparsity levels using their existing trained weights
(no retraining). Produced by the multi-rate sweep added inside `run_dataset` —
saved as `multi_rate` in each `results_<DATASET>.json`.

Standard paper format: methods (rows) × sparsity (columns) × MAE per dataset.
Set `METRIC = 'rmse'` below to render the RMSE variant.


In [ ]:
# Multi-Rate Main Results table. One table per dataset.
import glob, json, re
import numpy as np

METRIC = 'mae'      # 'mae' or 'rmse'
DECIMALS = 3

def _lead(label):
    mm = re.match(r'\s*(\d+)', label)
    return int(mm.group(1)) if mm else 999

files = sorted(glob.glob('results_*.json'))
if not files:
    print('No results_*.json on disk. Run the driver cell first.')
else:
    for fp in files:
        with open(fp) as f: m = json.load(f)
        mr = m.get('multi_rate') or {}
        if not mr:
            print(f'[{m["dataset"]}] no multi_rate data — re-run the notebook to populate it.')
            continue
        # Discover columns (sparsity levels) from any model entry
        rates = sorted({float(r) for d in mr.values() for r in d.keys()})
        labels = sorted(mr.keys(), key=_lead)
        title = f"=== {m['dataset']} — {METRIC.upper()} by sparsity (lower is better) ==="
        print('\n' + title)
        hdr = f"{'Model':<48}" + ''.join(f"{int(r*100):>10}%" for r in rates)
        print(hdr)
        print('-' * len(hdr))
        # Find the best (min) per column for bolding
        col_best = {}
        for r in rates:
            col_vals = []
            for label in labels:
                vals = mr.get(label, {}).get(f'{r:.2f}', {}).get(METRIC, [])
                if vals:
                    col_vals.append((label, float(np.mean(vals))))
            if col_vals:
                col_best[r] = min(col_vals, key=lambda kv: kv[1])[0]
        for label in labels:
            row = f'{label:<48}'
            for r in rates:
                vals = mr.get(label, {}).get(f'{r:.2f}', {}).get(METRIC, [])
                if not vals:
                    row += f'{"-":>11}'
                else:
                    v = float(np.mean(vals))
                    mark = '*' if col_best.get(r) == label else ' '
                    row += f'{v:>10.{DECIMALS}f}{mark}'
            print(row)
        print('  * = best per column')


## Hit@N Tolerance Sensitivity (Q16)

Recomputes Hit@N (fraction of held-out predictions within ε of truth) at a *grid* of
tolerances, not just the two HIT_TOL values saved during the run. Reads `predictions_<DATASET>.npz`
directly — no retraining. Edit `TOL_FRACTIONS_OF_MEDIAN` below to change the grid.


In [ ]:
# Q16: Hit@N tolerance sensitivity from saved predictions (no retraining).
import glob, json
import numpy as np

# Grid of tolerances expressed as fractions of the per-dataset median true value.
# Default covers a wide range so reviewers can see the curve.
TOL_FRACTIONS_OF_MEDIAN = [0.03, 0.05, 0.07, 0.10, 0.15, 0.20]

files = sorted(glob.glob('predictions_*.npz'))
if not files:
    print('No predictions_*.npz on disk. Run the driver cell first.')
else:
    for npz_path in files:
        dataset = npz_path[len('predictions_'):-len('.npz')]
        json_path = f'results_{dataset}.json'
        try:
            with open(json_path) as f: meta = json.load(f)
        except FileNotFoundError:
            print(f'[skip] {dataset}: missing {json_path}')
            continue
        data = np.load(npz_path, allow_pickle=True)
        names = [str(x) for x in data['__names__']]
        true  = data['__true__']
        med   = float(np.median(true))
        tols  = [max(0.5, float(np.round(f * med))) for f in TOL_FRACTIONS_OF_MEDIAN]
        unit  = meta.get('unit', '')
        print(f"\n=== {dataset}  (median={med:.2f} {unit}) ===")
        hdr = f"{'Model':<48}" + ''.join(f'{int(t):>7}' for t in tols)
        print(hdr); print('-' * len(hdr))
        # Sort by lead num
        import re
        def lead(label):
            mm = re.match(r'\s*(\d+)', label)
            return int(mm.group(1)) if mm else 999
        order = sorted(range(len(names)), key=lambda i: lead(names[i]))
        for i in order:
            pred = data[f'pred_{i:03d}']
            err  = np.abs(pred - true)
            row  = f"{names[i]:<48}" + ''.join(f'{(err < t).mean()*100:>6.1f}%' for t in tols)
            print(row)
        print(f"  tolerances (in {unit}, ~5%-20% of median): {', '.join(f'{int(t)}' for t in tols)}")


## Q12 — Ridge α + KNN k Hyperparameter Sensitivity

Reads `ridge_alpha_sweep` and `knn_k_sweep` from each `results_<DATASET>.json`.
These are produced inside `run_dataset` without retraining (Ridge is refit; KNN re-evaluated).


In [ ]:
# Q12 sensitivity tables (Ridge α, KNN k) — one block per dataset.
import glob, json
import numpy as np

files = sorted(glob.glob('results_*.json'))
if not files:
    print('No results_*.json on disk.')
else:
    for fp in files:
        with open(fp) as f: m = json.load(f)
        rs = m.get('ridge_alpha_sweep') or {}
        ks = m.get('knn_k_sweep') or {}
        print(f"\n=== {m['dataset']} — Q12 sensitivity (MAE) ===")
        if rs:
            print('  Ridge α   ' + '  '.join(f'{float(a):>7.2f}' for a in sorted(rs, key=float)))
            print('  Ridge MAE ' + '  '.join(f'{np.mean(rs[a]):>7.4f}' for a in sorted(rs, key=float)))
        if ks:
            print('  KNN k     ' + '  '.join(f'{int(k):>7d}' for k in sorted(ks, key=int)))
            print('  KNN MAE   ' + '  '.join(f'{np.mean(ks[k]):>7.4f}' for k in sorted(ks, key=int)))


## Q2 — Soft-LOCF EMA Decay Sensitivity

MARST trained at decay ∈ {0.80, 0.90, 0.95, 0.99} per dataset (single seed). Reports MAE
for each setting; the production model uses 0.95.


In [ ]:
# Q2 — Soft-LOCF decay sensitivity table.
import glob, json
import numpy as np

files = sorted(glob.glob('results_*.json'))
if not files:
    print('No results_*.json on disk.')
else:
    decays = sorted({float(d) for fp in files
                     for d in (json.load(open(fp)).get('soft_locf_decay_sweep') or {})})
    print(f"{'Dataset':<12}" + ''.join(f'  d={d:.2f}' for d in decays))
    print('-' * (12 + 9 * len(decays)))
    for fp in files:
        m = json.load(open(fp)); sweep = m.get('soft_locf_decay_sweep') or {}
        row = f"{m['dataset']:<12}"
        for d in decays:
            v = sweep.get(str(d)) or sweep.get(d)
            row += f"  {np.mean(v):>6.3f}" if v else f"  {'-':>6}"
        print(row)


## Q6 — Sparsity Curriculum Ablation

Compares MARST trained with the 60%→80% sparsity curriculum (first 600 epochs)
against MARST trained with sparsity fixed at 80% from epoch 0. Single seed per setting.


In [ ]:
# Q6 — Curriculum vs fixed sparsity table.
import glob, json
import numpy as np

files = sorted(glob.glob('results_*.json'))
if not files:
    print('No results_*.json on disk.')
else:
    variants = set()
    for fp in files:
        variants |= set((json.load(open(fp)).get('curriculum_ablation') or {}).keys())
    variants = sorted(variants)
    hdr = f"{'Dataset':<12}" + ''.join(f'  {v[:22]:<22}' for v in variants) + '  Δ (fixed-curriculum)'
    print(hdr); print('-' * len(hdr))
    for fp in files:
        m = json.load(open(fp))
        ca = m.get('curriculum_ablation') or {}
        means = {v: np.mean(ca.get(v, [])) if ca.get(v) else float('nan') for v in variants}
        delta = float('nan')
        cu_k = next((k for k in variants if 'curriculum' in k.lower()), None)
        fx_k = next((k for k in variants if 'fixed'      in k.lower()), None)
        if cu_k and fx_k and means[cu_k] == means[cu_k] and means[fx_k] == means[fx_k]:
            delta = means[fx_k] - means[cu_k]
        row = f"{m['dataset']:<12}" + ''.join(f'  {means[v]:>22.4f}' for v in variants)
        row += f'  {delta:+.4f}' if delta == delta else '  -'
        print(row)
